# Powderday flux catalogs — quenched galaxies in the high-res 25 Mpc box

**Goal.** Multi-aperture, dusty vs dust-free photometric catalogs (fluxes **with errors**) for
**quenched** galaxies in SIMBA high-res **m25n512** (`cis25`) at **z ≈ 0.3, 0.6, 0.7, 1.0**.

**Sample (per anchor snapshot).** `log10 M* > 10`, **passive** by the 0.2/τ criterion
(sSFR < 0.2/t_H at the anchor), and **> 20 gas particles** (plus the usual ≥ 20 star-particle floor).
The sample is split by **weak vs strong AGN feedback over the quench window**: the AGN–ISM coupling
strength `xcoup_hist` (jet-mode strength gated by gas-poorness, §8j physics) averaged between each
galaxy's **SFT and QT** (1/t and 0.2/t crossings from `find_quenching_times`); classes: **strong** = fully coupled (`xstr_quench` = 1) through the window, **weak** = bottom tercile of the remainder, **intermediate** = the rest.

**Pipeline** (same skeleton as `test_powderday.ipynb`, selection machinery from
`quench_mode_vs_sigma_gas.ipynb`):

| Part | What | Where |
|---|---|---|
| 1 | anchors + gated `BUILD_MULTI_Z` / `BUILD_BH` history builds | cluster |
| 2–3 | selection, SFT/QT, AGN split, **sample statistics** | anywhere (needs the HDF5s) |
| 4 | Stage 0 — per-galaxy particle files | cluster |
| 4b | annulus sampling QC — star/gas/dust counts per projected annulus × sightline | anywhere (needs Stage 0) |
| 5 | Stage 1 — selection HDF5 + Slurm masters (dust_on / dust_off) → run RT | cluster |
| 6 | aperture QC on the first `.rtout.sed` | cluster |
| 7 | Stage 2 — per-aperture flux extraction → **one catalog per aperture per dust mode** (7c: annular CIGALE inputs) | cluster |

**Apertures & sightlines (mock observation).** Stage 0 cuts a **100 pkpc spherical region**
around each galaxy (everything: CGM, satellites, projected neighbours — a true mock aperture,
not just member particles); the RT grid spans ±100 kpc (`zoom_box_len`). Hyperion log-spaces
`N_AP = 5` projected apertures 1→100 kpc — the 10^(k/2) ladder **1, 3.16, 10, 31.6, 100 kpc**
(central → outskirts), all extracted. Each SED is peeled along **4 sightlines**
(θ,φ) = (0,0), (45,90), (90,180), (135,270) deg — one catalog per (dust mode, aperture,
inclination). `N_AP/AP_MIN_KPC/AP_MAX_KPC` and `THETA_DEG/PHI_DEG` below must match the
parameter masters that the RT jobs copy. **Requires the one-time powderday patch documented
before Stage 1 (already applied on this cluster's install).**

**Flux errors.** Hyperion's Monte-Carlo SED uncertainty, read with
`get_sed(..., uncertainties=True)` and propagated through the filter convolution
(`<filter>_err` columns; NaN if a run stored no uncertainties).

# Part 0 — Setup & configuration

In [ ]:
import os
import gc
import glob
import json
import re
import subprocess
import warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table, vstack, join
from astropy import units as u
from astropy.cosmology import Planck15 as COSMO   # matches the quenching machinery

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen
from simbanator.analysis.quenching import find_quenching_times
from simbanator.utils.geometry import sightline_unit_vectors, projected_radius

# ── simulation ────────────────────────────────────────────────────────────────
SIM_NAME = "cis25"        # SIMBA high-res 25 Mpc/h box (m25n512); must exist in ~/.simbanator/config.json
try:
    sim = Simulation(SIM_NAME)
except KeyError as e:
    raise KeyError(
        f"'{SIM_NAME}' is not registered in ~/.simbanator/config.json on this machine.\n"
        "Register it once (adjust paths to where the 25 Mpc snapshots+catalogs live):\n"
        "  from simbanator.io.config import add_simulation\n"
        "  add_simulation('cis25', data_dir='<...>/SIMBA_25/s25',\n"
        "                 catalog_dir='<...>/SIMBA_25/s25/Groups',\n"
        "                 file_format='m25n512_{snap:03d}.hdf5')\n"
        "then add \"snap_z_map\": \"zsnap_map_caesar_box100.txt\" to that entry "
        "(SIMBA boxes share the snapshot schedule)."
    ) from e
if sim.scale_factors is None:
    raise ValueError(f"'{SIM_NAME}' config has no snap_z_map — add "
                     '"snap_z_map": "zsnap_map_caesar_box100.txt" to its entry in ~/.simbanator/config.json')

# filtered-particle filename prefix (Stage 0 == Stage 1, never let them drift)
PARTICLE_PREFIX = sim.file_format.split("_{")[0]        # 'm25n512'

# ── selection: quenched + massive + realistically gas-populated ───────────────
TARGET_REDSHIFTS = [0.3, 0.7, 1.0, 1.5, 2]   # z=2 (snap 78) dropped 2026-08-03: 8-gal sample too small for the class split
MASS_FLOOR       = 10.0        # log10(M*/Msun) > 10
PASSIVE_FACTOR   = 0.2         # passive if sSFR < 0.2 / t_H  (== the QT threshold of find_quenching_times)
NGAS_MIN         = 21          # STRICTLY > 20 gas particles at the anchor
NSTAR_MIN        = 20          # star-particle floor (same as quench_mode_vs_sigma_gas)
DUST_TO_H2_MIN   = 1e-4        # keep only M_dust/M_H2 >= 1e-4 at the anchor (drops the dust-poor half; 2026-08-05)

# ── AGN / coupling constants (identical to quench_mode_vs_sigma_gas §0) ──────
JET_LOGMBH    = 7.5            # jet mode: log10(M_BH) > 7.5 ...
JET_FEDD      = 0.2            #           ... AND f_Edd < 0.2
XRAY_FEDD_MAX = 0.02           # (kept for reference; xcoup uses the f_gas gate)
XRAY_FGAS_MAX = 0.2            # coupling gate: f_gas = Mgas/M* < 0.2
GYR = 1e9

# ── history tracking ──────────────────────────────────────────────────────────
TRACK_AGE_FRAC      = 0.09     # track back to ~this fraction of the cosmic age at selection
ANCHOR_END_OVERRIDE = {}
CORRUPT_SNAPS       = set()

# ── heavy-build gates (set True on the cluster, then reuse the cached HDF5s) ──
BUILD_MULTI_Z = False          # per-anchor progenitor FITS + property history HDF5
BUILD_BH      = False          # per-anchor BH (mass / mdot / f_Edd) history HDF5

# ── apertures (MUST match SED_APERTURE_* in simbanator/sed/parameters_master*.py) ──
N_AP       = 5           # SED_APERTURE_NAP: 10^(k/2) ladder -> 1, 3.16, 10, 31.6, 100 kpc
AP_MIN_KPC = 1.0
AP_MAX_KPC = 100.0
APERTURE_RADII_KPC = np.geomspace(AP_MIN_KPC, AP_MAX_KPC, N_AP)
# central -> outskirts; ALL rungs are extracted (nominal labels, true radii above)
TARGET_AP_KPC   = [1, 3, 10, 32, 100]
WANTED_AP_IDX   = list(range(N_AP))
APERTURE_LABELS = [f"ap{t:g}kpc" for t in TARGET_AP_KPC]
# annulus edges between consecutive rungs; the OUTER rung names the annulus
# (ann1kpc = the 0->1 kpc disc == ap1kpc) — shared by Parts 4b/4c/7a/7c/7f
R_EDGES        = np.concatenate([[0.0], APERTURE_RADII_KPC])   # [pkpc]
ANNULUS_LABELS = [l.replace("ap", "ann") for l in APERTURE_LABELS]

# ── viewing angles (MUST match THETA/PHI in the parameter masters) ──
THETA_DEG   = [0, 45, 90, 135]
PHI_DEG     = [0, 90, 180, 270]
N_INCL      = len(THETA_DEG)
INCL_LABELS = [f"i{t:g}p{p:g}" for t, p in zip(THETA_DEG, PHI_DEG)]   # i0p0, i45p90, ...
NHAT        = sightline_unit_vectors(THETA_DEG, PHI_DEG)      # LOS unit vectors

# ── Stage-0 region cutout: EVERYTHING (CGM, satellites) within this proper radius ──
# sphere radius = zoom_box_len = largest aperture (100 kpc): the grid's inscribed sphere
# is fully populated; only the outermost aperture is slightly depth-truncated at its edge
R_CUTOUT_KPC = 100.0

# ── powderday run layout (same conventions as test_powderday.ipynb) ──────────
GVFS_BASE   = ''
# '+' not os.path.join: with GVFS_BASE='' this must stay ABSOLUTE (see test_powderday)
REMOTE_HOME = GVFS_BASE + "/mnt/home/glorenzon/analize_simba_cgm"

hydro_dir_base = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')
selection_file = 'selection_m25_quenched'                  # MakeSED appends '.h5'
sed_output_dir = os.path.join(REMOTE_HOME, 'output', sim.name, 'sed_quenched_regions')

RUNS = {
    'dust_on':  dict(run_tag='dusty_simdust', paramf='parameters_master.py'),
    'dust_off': dict(run_tag='nodust_1e-12',  paramf='parameters_master-nodust.py'),
    # dust_on + AGN point sources (BH_SED=True, Hopkins+2007 template, BH_var=False:
    # L_bol = 0.1*BH_Mdot*c^2 from the SIMBA accretion rates). Needs PartType5 in the
    # Stage-0 cutouts -> re-run Part 4 (EXTRACT_OVERWRITE=True) before Stage 1.
    'agn_on':   dict(run_tag='dusty_simdust_agn', paramf='parameters_master-agn.py'),
}

# ── local output tree ─────────────────────────────────────────────────────────
OUT      = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR   = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
PLOTDIR  = os.path.join(OUT, "plots", "powderday_quenched")
CATDIR   = os.path.join(OUT, "sed_aperture_catalogs")
for _d in (SFHDIR, TABLEDIR, PLOTDIR, CATDIR):
    os.makedirs(_d, exist_ok=True)
SELECTION_FITS = os.path.join(TABLEDIR, "powderday_quenched_selection.fits")
AV_DUSTY = 0.1     # global A_V above which a galaxy counts as 'dusty' (Parts 4b/7a/7g)

# ── CIGALE tree (Parts 7b-7k) ──
CIGALE_DIR = os.path.join(CATDIR, "cigale")     # CIGALE input files (Parts 7b/7c)
RUN_BASE   = os.path.join(OUT, "cigale_runs")   # one run dir per (tag, snap, Z group)

def _ztag(z):
    return ("z%g" % z).replace(".", "p")

print(f"sim={sim.name}  data_dir={sim.data_dir}")
print(f"prefix={PARTICLE_PREFIX}  anchors z={TARGET_REDSHIFTS}")
print("aperture ladder [kpc]:", np.round(APERTURE_RADII_KPC, 2))
print("extracted rungs:", {l: f"{APERTURE_RADII_KPC[i]:.3g} kpc (idx {i})"
                           for l, i in zip(APERTURE_LABELS, WANTED_AP_IDX)})
print("sightlines:", INCL_LABELS, "  region cutout:", R_CUTOUT_KPC, "pkpc")
print("SED output:", sed_output_dir)

# Part 0b — shared helpers

Small loaders used by several parts, so each part stays runnable in a fresh session after
Parts 0/0b: the selection catalog, `(snap, gal_id)`-keyed alignment, the RT-grid centres
(Stage-1 selection HDF5, caesar fallback — identical values), the Stage-0 cutout reader
(code units → proper kpc about a given centre), anchor-epoch (row 0) history values, and
the Part 7a dusty flag.


In [ ]:
# ── shared helpers: selection, alignment, centres, cutouts, row-0 histories ──
def load_selection():
    """SELECTION_FITS (written by Part 3) -> (table, snap array, gal_id array)."""
    sel = Table.read(SELECTION_FITS)
    return sel, np.asarray(sel["snap"], int), np.asarray(sel["gal_id"], int)

def by_snap_gal(db, snaps, gids, default=np.nan):
    """Align a {(snap, gal_id): value} dict to (snaps, gids) rows -> array."""
    return np.array([db.get((int(s), int(g)), default)
                     for s, g in zip(snaps, gids)])

def rt_centers(snaps, gids):
    """RT-grid centres (code units): Stage-1 selection h5, else caesar (identical values)."""
    from simbanator.sed.makesed import read_selection_centers
    selh5 = os.path.join(sed_output_dir, RUNS["dust_on"]["run_tag"],
                         "target_selection", selection_file + ".h5")
    cen = read_selection_centers(selh5)
    if cen:
        print(f"grid centres from the Stage-1 selection h5 ({len(cen)} galaxies)")
        return cen
    print(f"[fallback] {selh5} missing -> reading centres from the caesar catalogs")
    for s in np.unique(snaps):
        cs = sim.load_catalog(snap=int(s))
        for g in np.unique(np.asarray(gids)[np.asarray(snaps) == s]):
            cen[(int(s), int(g))] = cs.galaxies[int(g)].pos.in_units("code_length").value
        del cs
        gc.collect()
    return cen

def cutout_file(snap, gid):
    """Path of one Stage-0 per-galaxy particle cutout."""
    return os.path.join(hydro_dir_base, f"snap_{int(snap):03d}",
                        f"{PARTICLE_PREFIX}_snap{int(snap):03d}_gal{int(gid):06d}.h5")

def read_cutout(snap, gid, center_code, ptype, fields=()):
    """One Stage-0 cutout particle type -> dict(pos [proper kpc], a, h, <fields> raw).

    `center_code` (code units, ckpc/h) is subtracted before the a/h conversion,
    so `pos` is proper kpc about that centre. Requested `fields` are returned
    raw (code units); a field absent from the file comes back as None.
    Returns None if the centre, the cutout file or the particle type is missing.
    """
    pf = cutout_file(snap, gid)
    if center_code is None or not os.path.exists(pf):
        return None
    with h5py.File(pf, "r") as f:
        if ptype not in f:
            return None
        a  = float(f["Header"].attrs["Time"])
        hh = float(f["Header"].attrs["HubbleParam"])
        out = dict(a=a, h=hh,
                   pos=(np.asarray(f[f"{ptype}/Coordinates"][:], float)
                        - center_code) * a / hh)
        for fld in fields:
            out[fld] = np.asarray(f[f"{ptype}/{fld}"][:]) if fld in f[ptype] else None
    return out

def anchor_row0(keys):
    """Anchor-epoch (row 0) values from every history under SFHDIR.

    -> {(anchor_snap, gal_id): {key: value, 'z0': anchor redshift}};
    keys absent from a history are simply missing from its dicts.
    """
    db = {}
    for hf in sorted(glob.glob(os.path.join(SFHDIR, "history_anchor_*.hdf5"))):
        with h5py.File(hf, "r") as f:
            snap0 = int(f["metadata/snapshots"][0])
            gid   = np.asarray(f["metadata/galaxy_ids"][:], int)
            z0    = float(f["redshift/Redshift"][0])
            row0  = {k: f[f"properties/{k}"][0] for k in keys
                     if f"properties/{k}" in f}
        for j, g in enumerate(gid):
            db[(snap0, int(g))] = {k: float(v[j]) for k, v in row0.items()}
            db[(snap0, int(g))]["z0"] = z0
    return db

def row0_arr(db, snaps, gids, key):
    """anchor_row0 value `key` aligned to (snaps, gids) rows (NaN where absent)."""
    return np.array([db.get((int(s), int(g)), {}).get(key, np.nan)
                     for s, g in zip(snaps, gids)])

def dusty_flags(snaps, gids):
    """Part 7a global A_V aligned to (snaps, gids) -> (A_V array, dusty flag array).

    dusty: 1 (A_V > AV_DUSTY), 0 (transparent), -1 (not measured yet — run
    Part 7a, then re-run the caller to get the dusty/non-dusty split).
    """
    avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
    db = {}
    if os.path.exists(avf):
        at = Table.read(avf)
        db = {(int(s), int(g)): float(a)
              for s, g, a in zip(at["snap"], at["gal_id"], at["A_V"])}
    else:
        print(f"[dusty split] {os.path.basename(avf)} not found — run Part 7a first")
    av = by_snap_gal(db, snaps, gids)
    return av, np.where(np.isnan(av), -1, (av > AV_DUSTY).astype(int))


# Part 1 — Anchors & gated cluster builds

Each anchor (z ≈ 0.3, 0.6, 0.7, 1.0, 2.0 → nearest snapshot) gets its **own** progenitor table +
property history with that snapshot as row 0, and a BH history aligned to the same rows — exactly
the `quench_mode_vs_sigma_gas.ipynb` machinery, pointed at `cis25`. Histories are pre-selected to
**massive + passive** at the anchor (the gas/star floors are applied later so the statistics can
count them).

In [ ]:
# ── anchor table: snapshot, track end, per-anchor product paths ──
_sall, _zall = [], []
for _s in range(0, 152):
    try:
        _zv = float(sim.get_z_from_snap(_s))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)
_aall = COSMO.age(_zall).value

ANCHORS = {}
for _zt in TARGET_REDSHIFTS:
    _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
    _age_end = TRACK_AGE_FRAC * float(_aall[_sall == _snap][0])
    _end = int(ANCHOR_END_OVERRIDE.get(_zt, int(_sall[np.searchsorted(_aall, _age_end)])))
    _tag = _ztag(_zt)
    ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap,
                        z=float(sim.get_z_from_snap(_snap)), end_snap=_end,
                        prog_file=f"progenitors_anchor_{_tag}.fits",
                        hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"),
                        bh_path=os.path.join(SFHDIR, f"bh_history_anchor_{_tag}.hdf5"))

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'end':>5s} {'hist':>6s} {'BH':>4s}")
for _zt, A in ANCHORS.items():
    print(f"{_zt:6.1f} {A['snap']:5d} {A['z']:7.3f} {A['end_snap']:5d} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(A['bh_path']) else '--':>4s}")

In [ ]:
# ── property list tracked per anchor (superset of what selection + coupling need) ──
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust", "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass",
        "pos", "ngas", "nstar", "ages.mass_weighted",
    ],
    "halo_data": ["masses.total"],
}

# ── GATED (cluster): per-anchor progenitor table + property history ──
# Verbatim port of quench_mode_vs_sigma_gas 1z·build, with the (stricter) M*>10 pre-selection.
if BUILD_MULTI_Z:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['hist_path'])}"); continue
        end = int(A["end_snap"])
        while end < A["snap"] and (end in CORRUPT_SNAPS or not os.path.exists(sim.get_caesar_file(end))):
            end += 1
        A["end_snap"] = end
        print(f"[{A['tag']}] anchor snap {A['snap']} (z={A['z']:.2f}) <- {end}: progenitor table ...")
        cs_a = sim.load_catalog(snap=A["snap"])
        caesar_read_progen([g.GroupID for g in cs_a.galaxies], A["prog_file"],
                           range(end, A["snap"] + 1), sim, output_dir=None)
        hist = HDF5BuildHistory(sim, cs_a, progfilename=A["prog_file"])
        with fits.open(hist.progen_file) as hdul:
            valid_ids = np.asarray(hdul[1].data["GroupID"])
            _tHa = COSMO.age(float(A["z"])).value * 1e9
            _gid = np.array([g.GroupID for g in cs_a.galaxies])
            _ms  = np.array([float(g.masses["stellar"]) for g in cs_a.galaxies])
            _sf  = np.array([float(g.sfr) for g in cs_a.galaxies])
            with np.errstate(all="ignore"):
                _ss = np.where(_ms > 0, _sf / _ms, np.nan)
                _ok = (np.log10(np.where(_ms > 0, _ms, np.nan)) > MASS_FLOOR) & (_ss < PASSIVE_FACTOR / _tHa)
            _keep = {int(g) for g in _gid[_ok]}
            valid_ids = np.asarray([i for i in valid_ids if int(i) in _keep], dtype=valid_ids.dtype)
            print(f"  [pre-select] {len(valid_ids)}/{len(_gid)} massive+passive at z={A['z']:.2f}")
        hist.get_history_indx(valid_ids, A["snap"], end)
        props_try = {k: list(v) for k, v in PROPS.items()}
        while True:   # drop-and-retry: some catalog versions miss some fields
            try:
                hist.get_property_history(props_try, verbose=0); break
            except KeyError as e:
                msg = str(e); dropped = False
                for fam, plist in props_try.items():
                    for pr in list(plist):
                        if pr in msg or pr.split("/")[-1] in msg:
                            plist.remove(pr); print("  [drop]", pr); dropped = True
                if not dropped:
                    raise
        hist.save_history_to_hdf5(os.path.basename(A["hist_path"]))
        del cs_a, hist; gc.collect()
        print(f"[{A['tag']}] history -> {A['hist_path']}")
else:
    print("BUILD_MULTI_Z=False -> expecting per-anchor histories under", SFHDIR)

In [ ]:
# ── loaders (verbatim from quench_mode_vs_sigma_gas): row 0 = the anchor epoch ──
def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H

def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP; gc.collect()
    return M

In [ ]:
# ── BH history: per-anchor build (GATED) + loader (verbatim quench_mode §4b) ──
BH_CANDIDATES = {"bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
                 "bh_mdot": ["bhmdot", "bh_mdot"],
                 "bh_fedd": ["bh_fedd", "bhfedd", "fedd"]}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

def build_bh_for_anchor(A, galaxy_ids, snaps_arr, n_gal):
    pidx = build_prog_index(A, galaxy_ids, snaps_arr)
    n_snap = len(snaps_arr)
    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri]); cv = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    if p is not None:
                        BH[k][ri, cv] = f[p][:][vi]
        except (OSError, KeyError) as e:
            print(f"  [skip] snap {snap}: {type(e).__name__}"); CORRUPT_SNAPS.add(snap)
    with h5py.File(A["bh_path"], "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print(f"[{A['tag']}] BH history -> {A['bh_path']}")
    return BH

def load_bh(bh_hist_path):
    with h5py.File(bh_hist_path, "r") as f:
        return {k: f[k][:] for k in f.keys()}

if BUILD_BH:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["bh_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['bh_path'])}"); continue
        if not os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] no history yet -> run BUILD_MULTI_Z first"); continue
        _H = load_anchor_history(A)
        build_bh_for_anchor(A, _H["galaxy_ids"], _H["snaps_arr"], len(_H["galaxy_ids"]))
        del _H; gc.collect()
else:
    print("BUILD_BH=False -> expecting per-anchor BH histories under", SFHDIR)

# Part 2 — Selection, quench events (SFT/QT) & the weak/strong AGN split

- **Selection** (at row 0 = the anchor): `log10 M* > 10`, passive (`sSFR < 0.2/t_H`), `ngas > 20`,
  `nstar ≥ 20`.
- **SFT/QT** per galaxy from `find_quenching_times` on the tracked sSFR history (SFT = crossing
  below 1/t, QT = subsequent crossing below 0.2/t with persistence); the **last** event is kept.
- **AGN split**: `xstr_quench` = mean of `xcoup_hist` (jet strength `clip(log10(0.2/f_Edd),0,1)`
  for `log M_BH > 7.5`, gated by `f_gas < 0.2`) over snapshots with `t_SFT ≤ t ≤ t_QT`; if the
  window is narrower than the snapshot spacing, the finite snapshot nearest SFT is used.
  **strong = fully coupled** (`xstr_quench` = 1 through the window; plain terciles degenerate
  into this tie-clump — 15–20 gals per anchor sit exactly at 1); **weak** = bottom tercile of the
  non-saturated remainder (per anchor); `intermediate` = the rest; no finite coupling = `no_AGN`;
  no detected quench event = `no_event`.

In [ ]:
# ── selection mask at the anchor epoch (row 0) ──
def selection_mask(P, t_cosmic_yr):
    mstar0 = P["masses.stellar"][0]
    sfr0   = P["sfr"][0]
    ngas0  = P["ngas"][0]
    nstar0 = P["nstar"][0] if "nstar" in P else np.full_like(mstar0, np.inf)
    _has_dh2 = ("masses.dust" in P) and ("masses.H2" in P)
    if not _has_dh2:
        print("[selection_mask] WARNING: masses.dust/masses.H2 missing from history -> dust/H2 cut skipped")
    mdust0 = P["masses.dust"][0] if _has_dh2 else None
    mh2_0  = P["masses.H2"][0]   if _has_dh2 else None
    with np.errstate(all="ignore"):
        ssfr0 = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)
        cuts = {
            "massive":  np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR,
            "passive":  ssfr0 < (PASSIVE_FACTOR / t_cosmic_yr[0]),
            "gas>20":   ngas0 >= NGAS_MIN,
            "star>=20": nstar0 >= NSTAR_MIN,
            # multiplicative form so M_H2=0 rows pass instead of dividing by zero
            "dust/H2":  (mdust0 >= DUST_TO_H2_MIN * mh2_0) if _has_dh2
                        else np.ones_like(mstar0, dtype=bool),
        }
    m = (cuts["massive"] & cuts["passive"] & cuts["gas>20"] & cuts["star>=20"]
         & cuts["dust/H2"])
    return m, cuts

# ── SFT/QT per selected galaxy (trimmed from quench_mode build_records) ──
def quench_records(P, t_cosmic_yr, redshift, galaxy_ids, cols):
    """One record per selected column; galaxies without a detected quench event keep NaN times."""
    records = []
    for col in np.asarray(cols, int):
        gid = galaxy_ids[col]
        mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
        with np.errstate(all="ignore"):
            ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
        valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
        rec = dict(gid=int(gid), col=int(col), t_sft=np.nan, t_qt=np.nan,
                   tau_q=np.nan, tau_q_over_tH=np.nan, z_qt=np.nan)
        if valid.sum() >= 5:
            t = t_cosmic_yr[valid]; s = ssfr[valid]
            o = np.argsort(t); t, s = t[o], s[o]
            tu, ui = np.unique(t, return_index=True); su = s[ui]
            if len(tu) >= 5:
                qts, sfts, _, dbg = find_quenching_times(
                    tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True)
                if len(qts):
                    k = int(np.argmax(qts))                     # last (surviving) quench event
                    rec["t_qt"], rec["t_sft"] = float(qts[k]), float(sfts[k])
                    rec["tau_q"] = rec["t_qt"] - rec["t_sft"]
                    z_qt = float(np.interp(rec["t_qt"], t_cosmic_yr[::-1], redshift[::-1]))
                    rec["z_qt"] = z_qt
                    rec["tau_q_over_tH"] = rec["tau_q"] / (COSMO.age(z_qt).value * 1e9)
        records.append(rec)
    return records

In [ ]:
# ── AGN–ISM coupling over the quench window [SFT, QT] (physics verbatim from §8j build_coupling) ──
def coupling_quench_window(BH, P, records, t_cosmic_yr):
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    with np.errstate(all="ignore"):
        fgas_hist = np.where(P["masses.stellar"] > 0, P["masses.gas"] / P["masses.stellar"], np.nan)
        _bh_ok  = np.isfinite(BH["bh_mass"]) & np.isfinite(BH["bh_fedd"])
        _mbh_ok = BH["bh_mass"] > 10 ** JET_LOGMBH
        wjet_hist = np.where(_bh_ok, np.where(_mbh_ok,
                             np.clip(np.log10(JET_FEDD / np.clip(BH["bh_fedd"], 1e-12, None)), 0.0, 1.0),
                             0.0), np.nan)
        xcoup_hist = np.where(np.isfinite(wjet_hist) & np.isfinite(fgas_hist),
                              wjet_hist * (fgas_hist < XRAY_FGAS_MAX).astype(float), np.nan)
    n = len(records)
    xstr_q = np.full(n, np.nan)
    for i, r in enumerate(records):
        if not (np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"])):
            continue                                   # no quench event -> stays NaN ('no_event')
        cs = xcoup_hist[_ord, r["col"]].astype(float)
        fin = np.isfinite(cs)
        win = (t_inc >= r["t_sft"]) & (t_inc <= r["t_qt"]) & fin
        if not win.any() and fin.any():
            # quench window narrower than the snapshot spacing -> nearest finite snapshot to SFT
            j = np.where(fin)[0]
            win = np.zeros_like(fin); win[j[np.argmin(np.abs(t_inc[j] - r["t_sft"]))]] = True
        if win.any():
            xstr_q[i] = np.nanmean(cs[win])
    bx = np.isfinite(xstr_q)
    # Physical classes (2026-08-03): xstr_quench piles up at exactly 1.0 (fully
    # coupled through the whole quench window; 15-20 gals per anchor), so plain
    # terciles degenerate into the tie-clump at 1. Instead: strong = the fully
    # coupled clump, weak = bottom tercile of the non-saturated remainder,
    # intermediate = the rest. xstr_quench stays continuous in the catalogs.
    XSTR_FULL_EPS = 1e-6
    strong = bx & (xstr_q >= 1.0 - XSTR_FULL_EPS)
    weak = np.zeros(n, bool); lo_q = np.nan; hi_q = 1.0
    _rest = bx & ~strong
    if _rest.sum() >= 3:
        lo_q = float(np.nanquantile(xstr_q[_rest], 1.0 / 3.0))
        weak = _rest & (xstr_q <= lo_q)
    inter = bx & ~strong & ~weak
    no_fb = ~bx
    return dict(xstr_quench=xstr_q, strong=strong, weak=weak, inter=inter, no_fb=no_fb,
                tercile=(lo_q, hi_q))

def agn_class_labels(CO, records):
    """Per-record string label; galaxies without a quench event are 'no_event'."""
    n = len(records)
    has_event = np.array([np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"]) for r in records])
    lab = np.array(["unclassified"] * n, dtype=object)
    if CO is not None:
        lab[CO["no_fb"]] = "no_AGN"
        lab[CO["inter"]] = "intermediate"
        lab[CO["weak"]]  = "weak"
        lab[CO["strong"]] = "strong"
    lab[~has_event] = "no_event"
    return lab

In [ ]:
# ── driver: per anchor -> selection, records, coupling, labels ──
RESULTS = {}
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        print(f"[{A['tag']}] MISSING history -> run BUILD_MULTI_Z on the cluster; skipped")
        continue
    H = load_anchor_history(A)
    m, cuts = selection_mask(H["P"], H["t_cosmic_yr"])
    cols = np.where(m)[0]
    recs = quench_records(H["P"], H["t_cosmic_yr"], H["redshift"], H["galaxy_ids"], cols)
    BH = load_bh(A["bh_path"]) if os.path.exists(A["bh_path"]) else None
    CO = coupling_quench_window(BH, H["P"], recs, H["t_cosmic_yr"]) if BH is not None else None
    labels = agn_class_labels(CO, recs)
    if BH is None:
        print(f"[{A['tag']}] WARNING: no BH history -> AGN split = 'unclassified' (run BUILD_BH)")
    RESULTS[_zt] = dict(A=A, H=H, mask=m, cuts=cuts, cols=cols, records=recs, CO=CO, labels=labels)
    n_ev = int(np.isfinite([r["t_qt"] for r in recs]).sum())
    print(f"[{A['tag']}] snap {A['snap']} (z={A['z']:.3f}): pool={m.size} "
          f"selected={len(cols)} with_event={n_ev} "
          f"classes={dict(zip(*np.unique(labels, return_counts=True))) if len(labels) else {}}")

# Part 3 — Sample statistics & the selection catalog

How many galaxies survive each cut per snapshot, how many have gas at all, and how the AGN classes
populate. **Note:** the pool is the history's build-time pre-selection (massive + passive at the
anchor), not the full galaxy catalog — the funnel starts there. Also writes the per-galaxy
selection table (`powderday_quenched_selection.fits`) that Stages 0–2 read, so the RT stages never
depend on this session's memory.

In [ ]:
# ── funnel table + per-galaxy selection FITS ──
_rows, _sel_rows = [], []
for _zt, R in RESULTS.items():
    A, H, cuts = R["A"], R["H"], R["cuts"]
    ngas0 = H["P"]["ngas"][0]
    n_pool = int(np.isfinite(H["P"]["masses.stellar"][0]).sum())
    lab = R["labels"]
    _rows.append(dict(
        z_target=_zt, snap=A["snap"], z_snap=round(A["z"], 4),
        pool_massive_passive=n_pool,
        with_any_gas=int((ngas0 > 0).sum()),
        gas_gt20=int(cuts["gas>20"].sum()),
        massive=int(cuts["massive"].sum()),
        passive=int(cuts["passive"].sum()),
        star_ge20=int(cuts["star>=20"].sum()),
        dust_h2_ok=int(cuts["dust/H2"].sum()),
        selected=len(R["cols"]),
        with_event=int(np.isfinite([r["t_qt"] for r in R["records"]]).sum()),
        strong=int((lab == "strong").sum()), weak=int((lab == "weak").sum()),
        intermediate=int((lab == "intermediate").sum()), no_AGN=int((lab == "no_AGN").sum()),
        no_event=int((lab == "no_event").sum()), unclassified=int((lab == "unclassified").sum()),
    ))
    # per-galaxy rows
    P0 = H["P"]
    for i, (r, l) in enumerate(zip(R["records"], lab)):
        c = r["col"]
        with np.errstate(all="ignore"):
            _ms = float(P0["masses.stellar"][0, c])
            _sf = float(P0["sfr"][0, c])
            _md  = float(P0["masses.dust"][0, c]) if "masses.dust" in P0 else np.nan
            _mh2 = float(P0["masses.H2"][0, c])   if "masses.H2"   in P0 else np.nan
            xs = R["CO"]["xstr_quench"][i] if R["CO"] is not None else np.nan
        _sel_rows.append(dict(
            snap=int(A["snap"]), z_snap=float(A["z"]), z_target=float(_zt),
            gal_id=int(r["gid"]),
            log_mstar=float(np.log10(_ms)) if _ms > 0 else np.nan,
            ssfr=float(_sf / _ms) if _ms > 0 else np.nan,
            ngas=int(P0["ngas"][0, c]), nstar=int(P0["nstar"][0, c]) if "nstar" in P0 else -1,
            t_sft=r["t_sft"], t_qt=r["t_qt"], tau_q=r["tau_q"],
            tau_q_over_tH=r["tau_q_over_tH"], z_qt=r["z_qt"],
            xstr_quench=float(xs), agn_class=str(l),
            mdust=_md, mh2=_mh2,
            dust_to_h2=(_md / _mh2) if (np.isfinite(_md) and np.isfinite(_mh2) and _mh2 > 0) else np.nan,
        ))

STATS = Table(_rows)
STATS.write(os.path.join(TABLEDIR, "powderday_quenched_stats.fits"), overwrite=True)
STATS.pprint(max_width=-1)

SEL = Table(_sel_rows)
SEL.write(SELECTION_FITS, overwrite=True)
print(f"\nselection table: {len(SEL)} galaxies over {len(np.unique(SEL['snap']))} snapshots "
      f"-> {SELECTION_FITS}")

In [ ]:
# ── figures: selection funnel + gas-particle content + AGN classes ──
_zs   = list(RESULTS.keys())
_tags = [RESULTS[z]["A"]["tag"] for z in _zs]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# funnel per anchor
_steps = ["pool_massive_passive", "with_any_gas", "gas_gt20", "selected", "with_event"]
_slbl  = ["massive+passive", "any gas", "gas>20", "all cuts", "SFT/QT found"]
_x = np.arange(len(_zs)); _w = 0.16
for j, (st, sl) in enumerate(zip(_steps, _slbl)):
    axes[0].bar(_x + (j - 2) * _w, [STATS[st][i] for i in range(len(STATS))], width=_w, label=sl)
axes[0].set_xticks(_x); axes[0].set_xticklabels(_tags)
axes[0].set_ylabel("N galaxies"); axes[0].set_title("selection funnel")
axes[0].legend(fontsize=9)

# gas-particle histograms (pool), with the >20 floor
for z in _zs:
    ng = RESULTS[z]["H"]["P"]["ngas"][0]
    ng = ng[np.isfinite(ng) & (ng > 0)]
    if ng.size:
        axes[1].hist(np.log10(ng), bins=25, histtype="step", lw=2, label=RESULTS[z]["A"]["tag"])
axes[1].axvline(np.log10(NGAS_MIN), color="k", ls=":", label=f"ngas={NGAS_MIN}")
axes[1].set_xlabel("log10 ngas (anchor)"); axes[1].set_ylabel("N")
axes[1].set_title("gas-particle content of the pool"); axes[1].legend(fontsize=9)

# AGN classes among the selected
_classes = ["strong", "intermediate", "weak", "no_AGN", "no_event", "unclassified"]
_bot = np.zeros(len(_zs))
for cl in _classes:
    v = np.array([STATS[cl][i] for i in range(len(STATS))], float)
    axes[2].bar(_x, v, bottom=_bot, label=cl)
    _bot += v
axes[2].set_xticks(_x); axes[2].set_xticklabels(_tags)
axes[2].set_ylabel("N selected"); axes[2].set_title("AGN-coupling classes (quench window)")
axes[2].legend(fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "sample_statistics.png"), dpi=150, bbox_inches="tight")
plt.show()

# Part 3b — mass–size QC: flag sources too large (or too small) for the apertures

Fixed **physical** apertures implicitly assume every source has a similar size — the mass–size
relation is the check on that. Anchor-epoch CAESAR radii come from the histories (row 0;
`radii.*` are **comoving kpc** — verified `unit: 'kpccm'` in the m25n512 catalogs — converted
with $1/(1+z)$). CAESAR $R_{50}$ is the 3D half-**mass** radius; the van der Wel+2014 quiescent
relations (projected half-light $R_e$) are drawn for context only.

Flags (written back into `SELECTION_FITS`; Part 7 carries them into every catalog):

- **`flag_too_large`** — `SIZE_FACTOR·R50 > R_CUTOUT_KPC`: the 100 pkpc cutout/grid truncates
  the stellar envelope → the "≈ total" 100 kpc aperture (and any CIGALE mass) biases low;
- **`flag_unresolved`** — `R50 < N_EPS_MIN·ε` (softening): the size is not trusted and the
  1 kpc "central" aperture is not meaningfully sub-galactic.

Nothing is dropped — the flags are one boolean away in any downstream cut. The per-rung print
shows for how much of the sample each aperture is sub-galactic (< R50) vs effectively total
(> 3 R50).


In [ ]:
# ── Part 3b — mass-size QC: flag too-large / unresolved sources for the aperture ladder ──
# Self-contained after Parts 0/0b: reads SELECTION_FITS + the anchor histories in SFHDIR.
SIZE_FACTOR    = 5.0     # envelope proxy: SIZE_FACTOR*R50 beyond the cutout -> truncated
N_EPS_MIN      = 2.0     # resolved if R50 >= N_EPS_MIN * softening
EPS_MIN_CKPC_H = 0.25    # m25n512 minimum gravitational softening [comoving kpc/h]
SIMBA_H        = 0.68

SEL, SNAPS, IDS = load_selection()
_rdb = anchor_row0(("radii.stellar_half_mass", "radii.gas_half_mass"))
_zz0 = row0_arr(_rdb, SNAPS, IDS, "z0")                        # row 0 = anchor epoch
# radii are comoving in the catalogs (kpccm) -> proper kpc with each anchor's own z
_r50s = row0_arr(_rdb, SNAPS, IDS, "radii.stellar_half_mass") / (1.0 + _zz0)
_r50g = row0_arr(_rdb, SNAPS, IDS, "radii.gas_half_mass") / (1.0 + _zz0)

_zsel = np.asarray(SEL["z_snap"], float)
_eps_pkpc = EPS_MIN_CKPC_H / SIMBA_H / (1.0 + _zsel)           # softening, proper kpc
flag_too_large  = SIZE_FACTOR * _r50s > R_CUTOUT_KPC
flag_unresolved = _r50s < N_EPS_MIN * _eps_pkpc

SEL["r50_star_kpc"]    = _r50s
SEL["r50_gas_kpc"]     = _r50g
SEL["flag_too_large"]  = flag_too_large.astype(int)
SEL["flag_unresolved"] = flag_unresolved.astype(int)
SEL.write(SELECTION_FITS, overwrite=True)

_nok = int(np.isfinite(_r50s).sum())
print(f"R50 matched: {_nok}/{len(SEL)} | median R50 = {np.nanmedian(_r50s):.2f} pkpc | "
      f"too large (R50 > {R_CUTOUT_KPC/SIZE_FACTOR:.0f} kpc): {int(flag_too_large.sum())} | "
      f"unresolved (R50 < {N_EPS_MIN:g} eps): {int(flag_unresolved.sum())}")
print("aperture rung vs the sample sizes:")
_rungs = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]
for _t, _rr in zip(APERTURE_LABELS, _rungs):
    _sub = 100 * np.nanmean(_rr < _r50s); _tot = 100 * np.nanmean(_rr > 3 * _r50s)
    print(f"  {_t:>9s} ({_rr:6.2f} kpc): sub-galactic (<R50) for {_sub:4.0f}%  |  "
          f"~total (>3 R50) for {_tot:4.0f}%")
for _k in np.where(flag_too_large | flag_unresolved)[0]:
    _why = "TOO LARGE" if flag_too_large[_k] else "unresolved"
    print(f"  [{_why}] snap {int(SEL['snap'][_k])} gal {int(SEL['gal_id'][_k])}: "
          f"R50={_r50s[_k]:.2f} pkpc, logM*={float(SEL['log_mstar'][_k]):.2f}, "
          f"z={_zsel[_k]:.2f}")

# ── mass-size relation vs the aperture ladder ──
_fig, _ax = plt.subplots(figsize=(7.4, 5.6))
_zt_colors = plt.cm.viridis(np.linspace(0, 0.9, len(TARGET_REDSHIFTS)))
# van der Wel+2014 Table 5, early types: R_e = A*(M*/5e10)^alpha [kpc] (context only)
_VDW = {0.25: (10**0.60, 0.75), 0.75: (10**0.42, 0.71), 1.25: (10**0.22, 0.76),
        1.75: (10**0.09, 0.76), 2.25: (10**-0.05, 0.79)}
_lm = np.asarray(SEL["log_mstar"], float)
_xmax = max(11.4, np.nanmax(_lm) + 0.15)
_mm = np.logspace(10, _xmax, 40)
for _c, _zt in zip(_zt_colors, TARGET_REDSHIFTS):
    _m = np.isclose(np.asarray(SEL["z_target"], float), _zt)
    if not _m.any():
        continue
    _ax.scatter(_lm[_m], _r50s[_m], s=22, color=_c, label=f"z\u2248{_zt:g}", zorder=3)
    _A, _al = _VDW[min(_VDW, key=lambda z: abs(z - _zt))]
    _ax.plot(np.log10(_mm), _A * (_mm / 5e10) ** _al, "--", color=_c, lw=1.1, alpha=0.7)
for _k in np.where(flag_too_large | flag_unresolved)[0]:
    _ax.scatter([_lm[_k]], [_r50s[_k]], s=95, facecolor="none",
                edgecolor="crimson", lw=1.4, zorder=4)
for _rr, _t in zip(_rungs, APERTURE_LABELS):
    _ax.axhline(_rr, color="0.78", lw=0.7, zorder=1)
    _ax.text(_xmax - 0.03, _rr * 1.04, _t, fontsize=7, va="bottom", ha="right", color="0.45")
_ax.axhline(R_CUTOUT_KPC / SIZE_FACTOR, color="crimson", ls=":", lw=1.3)
_ax.text(10.02, R_CUTOUT_KPC / SIZE_FACTOR * 1.05,
         f"too large ({SIZE_FACTOR:g}\u00b7R50 > {R_CUTOUT_KPC:.0f} kpc cutout)",
         fontsize=8, color="crimson", va="bottom")
_ax.set_yscale("log"); _ax.set_xlim(9.98, _xmax)
_ax.set_xlabel(r"$\log_{10}\,M_*/M_\odot$")
_ax.set_ylabel(r"stellar $R_{50}$ [proper kpc]")
_ax.set_title("mass\u2013size QC: CAESAR 3D half-mass radii vs the aperture ladder\n"
              "(dashed: van der Wel+2014 quiescent $R_e$, projected half-light \u2014 context only)",
              fontsize=9)
_ax.legend(fontsize=8, frameon=False, loc="lower right")
plt.savefig(os.path.join(PLOTDIR, "mass_size_aperture_qc.png"), dpi=140, bbox_inches="tight")
plt.show()


# Part 4 — Stage 0: extract the per-galaxy particle files (cluster)

One HDF5 per galaxy (gas + stars + BHs; gas keeps `Dust_Masses`, BHs carry
`BH_Mass`/`BH_Mdot` for the `agn_on` run) under
`hydro_dir_base/snap_NNN/<PREFIX>_snap<NNN>_gal<ID>.h5` — now a **100 pkpc spherical region
cutout** around each galaxy centre (CGM + satellites included; periodic-wrap safe), NOT just
the caesar member particles. Identical for all runs (`dust_on` / `dust_off` / `agn_on` — the dust treatment and the
AGN sources live in the parameter masters). Cutouts extracted before 2026-07-28 have no
`PartType5` group — re-run this part (`EXTRACT_OVERWRITE = True`) before launching `agn_on`. Reads `SELECTION_FITS`, so it can run in a fresh session once Part 3
has been executed.

⚠ Region files reuse the plist filenames, so `EXTRACT_OVERWRITE = True` below **replaces** any
old galaxy-member-only files — intended, since mixed hydro inputs would corrupt the sample.

In [ ]:
from simbanator.analysis import extract_particles

SEL, SNAPS, IDS = load_selection()
print(f"{len(SNAPS)} sources over snapshots {sorted(set(SNAPS.tolist()))}")

EXTRACT_OVERWRITE = True    # region cutouts REPLACE the old plist files (same names)
EXTRACT_PTYPES    = ("PartType0", "PartType4", "PartType5")   # gas (Dust_Masses) + stars + BHs (agn_on)

bad_snaps = []
for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    try:
        _cs = sim.load_catalog(snap=_snap)
        extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, radius=R_CUTOUT_KPC,
                          ptypes=EXTRACT_PTYPES, sim_name=sim.name, prefix=PARTICLE_PREFIX,
                          overwrite=EXTRACT_OVERWRITE, verbose=1)
        del _cs
    except (OSError, KeyError) as e:
        print(f"  [SKIP] snap {_snap}: {type(e).__name__}: {str(e).splitlines()[0]}")
        bad_snaps.append((_snap, len(_ids_here)))

print("\nparticle extraction complete ->", hydro_dir_base)
if bad_snaps:
    print(f"{len(bad_snaps)} snapshot(s) unreadable: {bad_snaps} — re-stage those files and re-run.")

# Part 4b — annulus sampling QC: particle counts per projected annulus

How well can powderday sample an **annular** SED? Each Hyperion aperture is a circle in the
**image plane**, so for every sightline the star (emission sources) and gas (dust carriers)
particles of each 100 pkpc cutout are projected perpendicular to the viewing direction and
counted in the annuli between consecutive rungs (`ann1kpc` = the 0→1 kpc disc, then 1→3.16,
3.16→10, 10→31.6, 31.6→100 kpc; as in Part 7a, the **outer** rung names the annulus).
Counts span the whole LOS depth through the sphere — exactly the geometry the RT sees (the
outermost annulus is depth-truncated like its aperture). Centres are the **exact RT grid
centres** (`code_coods` from the Stage-1 selection HDF5; caesar fallback if Part 5 has not
run yet).

Reading the numbers:

- **stars = intrinsic emitters.** `nstar = 0` → the annular flux is scattered/re-emitted
  light only and a CIGALE fit of it is meaningless; a few tens of star particles → the
  annular SED is shot-noise dominated (a handful of SSP ages/metallicities).
- **gas → dust grid.** Gas is smoothed onto the octree, so counts are indicative; `ndust`
  (gas with `Dust_Masses > 0`) counts the particles actually carrying dust.

One row per (galaxy, sightline) → `tables/annulus_particle_counts.fits`
(`nstar_/ngas_/ndust_/ntot_<annulus>` + `A_V_glob`/`dusty`); Part 7c reads it to flag
star-free annular catalogs.

**Dusty vs non-dusty split.** If Part 7a's `attenuation_vs_ism.fits` exists, each galaxy is
flagged **dusty** (global $A_V > 0.1$, same threshold as 7a Fig 3, fiducial aperture +
sightline) or non-dusty, the figure highlights the two subsamples (red vs gray lines, separate
medians) and the summary prints their per-annulus median counts — the dusty galaxies are the
ones whose annular attenuation/CIGALE fits matter, so their sampling is the QC that counts.
Part 7a needs the RT fluxes, so on a fresh pipeline this cell first runs without the split
(`dusty = -1`) — **re-run it after Part 7a** to get the highlighted version.

In [ ]:
# ── Part 4b — annulus sampling QC: star/gas counts per projected annulus & sightline ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (Part 4).
_R_MID = np.where(R_EDGES[:-1] > 0, np.sqrt(R_EDGES[:-1] * R_EDGES[1:]), R_EDGES[1:] / 2.0)

SEL, SNAPS, IDS = load_selection()
_centers = rt_centers(SNAPS, IDS)

_rows, _skipped = [], []
for _s, _g in zip(SNAPS, IDS):
    _c  = _centers.get((int(_s), int(_g)))
    _st = read_cutout(_s, _g, _c, "PartType4")
    _gs = read_cutout(_s, _g, _c, "PartType0", fields=("Dust_Masses",))
    if _st is None or _gs is None:
        _skipped.append((int(_s), int(_g)))
        continue
    _dm = (_gs["Dust_Masses"] if _gs["Dust_Masses"] is not None
           else np.zeros(len(_gs["pos"])))
    _d = {"star": _st["pos"], "gas": _gs["pos"],
          "dust": _gs["pos"][np.asarray(_dm) > 0]}                # dust-carrying gas
    for _j, _il in enumerate(INCL_LABELS):
        _row = {"snap": int(_s), "gal_id": int(_g), "incl": _il}
        for _pt, _pos in _d.items():                              # projected radius wrt LOS
            _cnt, _ = np.histogram(projected_radius(_pos, NHAT[_j]), R_EDGES)
            for _k, _al in enumerate(ANNULUS_LABELS):
                _row[f"n{_pt}_{_al}"] = int(_cnt[_k])
        for _al in ANNULUS_LABELS:
            _row[f"ntot_{_al}"] = _row[f"nstar_{_al}"] + _row[f"ngas_{_al}"]
        _rows.append(_row)

QC_COUNTS = Table(_rows)
QC_COUNTS.meta["R_EDGES"] = list(np.round(R_EDGES, 3))             # proper kpc

# dusty split from Part 7a (global A_V, fiducial aperture/sightline):
# dusty = 1 (A_V > AV_DUSTY), 0 (transparent), -1 (no A_V yet -> re-run after Part 7a)
_avg, QC_COUNTS["dusty"] = dusty_flags(QC_COUNTS["snap"], QC_COUNTS["gal_id"])
QC_COUNTS["A_V_glob"] = _avg
_have_av = bool(np.isfinite(_avg).any())

_out = os.path.join(TABLEDIR, "annulus_particle_counts.fits")
QC_COUNTS.write(_out, overwrite=True)
print(f"{len(QC_COUNTS)} rows ({len(QC_COUNTS)//N_INCL} galaxies x {N_INCL} sightlines) -> {_out}")
if _skipped:
    print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")
_dg = np.asarray(QC_COUNTS["dusty"], int)[::N_INCL]   # per galaxy (same on all sightlines)
if _have_av:
    print(f"dusty split (Part 7a global A_V > {AV_DUSTY:g}): {int((_dg == 1).sum())} dusty / "
          f"{int((_dg == 0).sum())} non-dusty / {int((_dg == -1).sum())} unmatched galaxies")

# ── summary: how well is each annulus sampled? ──
print(f"\n{'annulus':>10s} {'r [pkpc]':>13s} | {'nstar p16/50/84':>17s} {'=0':>4s} {'<10':>4s} "
      f"{'<100':>5s} | {'ngas p50':>8s} {'=0':>4s} | {'ndust p50':>9s} {'=0':>4s}")
for _k, _al in enumerate(ANNULUS_LABELS):
    _ns = np.asarray(QC_COUNTS[f"nstar_{_al}"], int)
    _ng = np.asarray(QC_COUNTS[f"ngas_{_al}"],  int)
    _nd = np.asarray(QC_COUNTS[f"ndust_{_al}"], int)
    _p  = np.percentile(_ns, [16, 50, 84]).astype(int)
    print(f"{_al:>10s} {R_EDGES[_k]:5.1f}-{R_EDGES[_k+1]:6.1f} | "
          f"{_p[0]:5d}/{_p[1]:5d}/{_p[2]:5d} {np.mean(_ns == 0)*100:3.0f}% {np.mean(_ns < 10)*100:3.0f}% "
          f"{np.mean(_ns < 100)*100:4.0f}% | {int(np.median(_ng)):8d} {np.mean(_ng == 0)*100:3.0f}% | "
          f"{int(np.median(_nd)):9d} {np.mean(_nd == 0)*100:3.0f}%")
_nfree = int(sum((np.asarray(QC_COUNTS[f"nstar_{_al}"], int) == 0).sum()
                 for _al in ANNULUS_LABELS))
print(f"\nstar-free (annulus, galaxy, sightline) triples: {_nfree} "
      f"/ {len(QC_COUNTS) * len(ANNULUS_LABELS)} — those annular SEDs have NO intrinsic emitters")

if _have_av:                     # median counts split by the Part 7a dusty flag
    _dm_all = np.asarray(QC_COUNTS["dusty"], int)
    print(f"\nmedian counts, dusty (D, n={int((_dg == 1).sum())} gals) "
          f"vs non-dusty (N, n={int((_dg == 0).sum())}):")
    print(f"{'annulus':>10s} | {'nstar D':>8s} {'nstar N':>8s} | {'ngas D':>8s} {'ngas N':>8s} "
          f"| {'ndust D':>8s} {'ndust N':>8s}")
    for _al in ANNULUS_LABELS:
        _vals = []
        for _cc in ("nstar", "ngas", "ndust"):
            _v = np.asarray(QC_COUNTS[f"{_cc}_{_al}"], int)
            for _dd in (1, 0):
                _m = _dm_all == _dd
                _vals.append(int(np.median(_v[_m])) if _m.any() else -1)
        print(f"{_al:>10s} | {_vals[0]:8d} {_vals[1]:8d} | {_vals[2]:8d} {_vals[3]:8d} "
              f"| {_vals[4]:8d} {_vals[5]:8d}")

# ── figure: count distributions per annulus, dusty vs non-dusty highlighted ──
_dm_all = np.asarray(QC_COUNTS["dusty"], int)
_have_split = _have_av and (_dm_all >= 0).any()
_fig, _axs = plt.subplots(1, 3, figsize=(13.5, 4.4), sharey=True)
for _ax, _pt, _ttl in zip(_axs, ("star", "gas", "dust"),
                          ("star particles (emitters)", "gas particles",
                           "dust-carrying gas (Dust_Masses > 0)")):
    _M = np.column_stack([np.asarray(QC_COUNTS[f"n{_pt}_{_al}"], int)
                          for _al in ANNULUS_LABELS])
    if _have_split:                # per-(gal,sightline) lines colored by the Part 7a split
        for _rowv, _dd in zip(_M, _dm_all):
            _ax.plot(_R_MID, _rowv, color={1: "#c0392b", 0: "0.75"}.get(_dd, "0.88"),
                     lw=0.5, alpha=0.45, zorder=1)
        for _dd, _col, _mk, _lab in ((1, "#c0392b", "o-", f"dusty ($A_V>{AV_DUSTY:g}$)"),
                                     (0, "#2980b9", "s--", "non-dusty")):
            _mrows = _dm_all == _dd
            if _mrows.any():
                _ax.plot(_R_MID, np.median(_M[_mrows], axis=0), _mk, color=_col, lw=2,
                         zorder=3, label=f"{_lab} median (n={int(_mrows.sum()) // N_INCL} gals)")
    else:
        for _rowv in _M:                                          # one line per (gal, sightline)
            _ax.plot(_R_MID, _rowv, color="0.75", lw=0.5, alpha=0.5, zorder=1)
        _ax.fill_between(_R_MID, np.percentile(_M, 16, axis=0), np.percentile(_M, 84, axis=0),
                         color="#2980b9", alpha=0.25, zorder=2, label="16–84%")
        _ax.plot(_R_MID, np.median(_M, axis=0), "o-", color="#2980b9", lw=2, zorder=3,
                 label="median")
    for _thr, _ls in ((10, ":"), (100, "--")):
        _ax.axhline(_thr, color="0.3", ls=_ls, lw=0.9)
        _ax.text(_R_MID[0] * 0.9, _thr * 1.15, f"N={_thr}", color="0.3", fontsize=7)
    _ax.set_xscale("log"); _ax.set_yscale("symlog", linthresh=1)
    _ax.set_xticks(_R_MID); _ax.set_xticklabels([l[3:] for l in ANNULUS_LABELS], fontsize=8)
    _ax.set_xlabel("annulus (outer-rung label)"); _ax.set_title(_ttl, fontsize=10)
    _ax.set_ylim(bottom=-0.5)
_axs[0].set_ylabel("particles per projected annulus (full LOS depth)")
_axs[0].legend(fontsize=8, frameon=False, loc="upper left")
_fig.suptitle("annulus sampling QC — all galaxies x 4 sightlines"
              + (" — dusty split: Part 7a global $A_V$" if _have_split else ""), fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(PLOTDIR, "annulus_particle_counts.png"), dpi=140, bbox_inches="tight")
plt.show()


# Part 4c — SIMBA metallicities per aperture & annulus (CIGALE priors)

Mass-weighted **stellar** and **gas** metallicities (total metal mass fraction,
`Metallicity[:, 0]`) of each cutout, measured in the **same projected geometry as the SEDs**:
per sightline, cumulative within each aperture rung (`ap1kpc…ap100kpc`) and in each annulus
between rungs (`ann3kpc…ann100kpc`; `ann1kpc`≡`ap1kpc`). Same centres/projection as Part 4b.

These are the **metallicity priors for the CIGALE runs** (Part 7e): CIGALE's bc03
`metallicity` and nebular `zgas` are strict grids, so each galaxy's SIMBA value is snapped to
the **nearest allowed grid value in log space** (bc03: 0.0001, 0.0004, 0.004, 0.008, 0.02,
0.05 — verified against the cluster's CIGALE 2025.1 sources) and the catalog is split into
per-metallicity sub-runs: one per bc03 node, each fitted with that single stellar Z and a
`zgas` grid restricted to the group members' snapped values. The summary below shows how
the sample maps onto the bc03 nodes per aperture — i.e. how many sub-runs Part 7e will
create. Empty apertures/annuli (no particles) → NaN → the `Zsfree` group (default Z grid).

One row per (galaxy, sightline) → `tables/aperture_metallicities.fits`
(`Zstar_<label>`, `Zgas_<label>` for the 9 labels).

In [ ]:
# ── Part 4c — mass-weighted Z_star / Z_gas per projected aperture & annulus ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts; same geometry as Part 4b.
from simbanator.sed.cigale import grid_options, nearest_option

Z_LABELS = list(APERTURE_LABELS) + ANNULUS_LABELS[1:]   # cumulative rungs + true annuli

SEL, SNAPS, IDS = load_selection()
_centers = rt_centers(SNAPS, IDS)

def _mwz(z, m, sel):
    # mass-weighted metallicity over a particle selection (NaN if empty)
    if not sel.any():
        return np.nan
    mm = m[sel]
    return float(np.sum(mm * z[sel]) / np.sum(mm)) if mm.sum() > 0 else np.nan

_rows, _skipped = [], []
for _s, _g in zip(SNAPS, IDS):
    _c = _centers.get((int(_s), int(_g)))
    _P = {}
    for _pt, _nm in (("PartType4", "star"), ("PartType0", "gas")):
        _cut = read_cutout(_s, _g, _c, _pt, fields=("Masses", "Metallicity"))
        if _cut is None:
            _P = None
            break
        _met = _cut["Metallicity"]
        _z = np.asarray(_met[:, 0] if _met.ndim == 2 else _met, float)  # col 0 = total Z
        _P[_nm] = (_cut["pos"], np.asarray(_cut["Masses"], float), _z)
    if _P is None:
        _skipped.append((int(_s), int(_g)))
        continue
    for _j, _il in enumerate(INCL_LABELS):
        _row = {"snap": int(_s), "gal_id": int(_g), "incl": _il}
        for _nm, (_pos, _m, _z) in _P.items():
            _R = projected_radius(_pos, NHAT[_j])
            for _k, _lab in enumerate(APERTURE_LABELS):              # cumulative
                _row[f"Z{_nm}_{_lab}"] = _mwz(_z, _m, _R <= R_EDGES[_k + 1])
            for _k, _lab in enumerate(ANNULUS_LABELS[1:], start=1):  # annular
                _row[f"Z{_nm}_{_lab}"] = _mwz(_z, _m,
                                              (_R > R_EDGES[_k]) & (_R <= R_EDGES[_k + 1]))
        _rows.append(_row)

ZTAB = Table(_rows)
ZTAB.meta["R_EDGES"] = list(np.round(R_EDGES, 3))
_out = os.path.join(TABLEDIR, "aperture_metallicities.fits")
ZTAB.write(_out, overwrite=True)
print(f"{len(ZTAB)} rows ({len(ZTAB)//N_INCL} galaxies x {N_INCL} sightlines) -> {_out}")
if _skipped:
    print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")

# ── summary: sample vs the CIGALE grids (how many Z sub-runs Part 7e will make) ──
ZSUN = 0.0134
_zs_grid = grid_options("bc03", "metallicity")
print(f"\nbc03 metallicity nodes: {_zs_grid}")
print(f"{'label':>10s} | {'med Z*/Zsun':>11s} {'med Zgas/Zsun':>13s} | bc03 node counts (stars)")
for _lab in Z_LABELS:
    _zsv = np.asarray(ZTAB[f"Zstar_{_lab}"], float)
    _zgv = np.asarray(ZTAB[f"Zgas_{_lab}"], float)
    _near = nearest_option(_zsv, _zs_grid)
    _cnt = {f"{_n:g}": int(np.sum(_near == _n)) for _n in _zs_grid
            if np.sum(_near == _n)}
    _nnan = int(np.sum(~np.isfinite(_near)))
    if _nnan:
        _cnt["free"] = _nnan
    print(f"{_lab:>10s} | {np.nanmedian(_zsv)/ZSUN:11.2f} {np.nanmedian(_zgv)/ZSUN:13.2f} | {_cnt}")


# Part 5 — Stage 1: selection HDF5 + Slurm masters (all three runs)

## ⚠ REQUIRED once per powderday install: the multi-aperture patch

Stock powderday gives the peeled SED a **single infinite aperture**; the parameter masters in this
repo now carry `SED_APERTURE_NAP / SED_APERTURE_MIN_KPC / SED_APERTURE_MAX_KPC`, but powderday must
be taught to read them (**already applied** in this cluster's `powderday/front_end_tools.py`,
both SED branches — redo only on a fresh install). On the cluster, locate the peeled-image setup:

```bash
grep -rn "add_peeled_images" $(python -c "import powderday, os; print(os.path.dirname(powderday.__file__))")
```

and immediately after the image-configuration lines (`set_viewing_angles` / `set_track_origin` /
`set_uncertainties`), insert (adapt `image` / `cfg.par` to the local variable names in that file):

```python
# --- multi-aperture SEDs (analize_simba_cgm patch) ---
try:
    from hyperion.util.constants import kpc as _kpc
    _nap = int(getattr(cfg.par, 'SED_APERTURE_NAP', 0))
    if _nap > 0:
        image.set_aperture_range(_nap,
                                 float(cfg.par.SED_APERTURE_MIN_KPC) * _kpc,
                                 float(cfg.par.SED_APERTURE_MAX_KPC) * _kpc)
        image.set_uncertainties(True)   # Monte-Carlo SED errors -> <filter>_err columns
except Exception as _e:
    print('[aperture patch] skipped:', _e)
```

Hyperion **log-spaces** the apertures between min and max: 1→100 kpc with `NAP=5` gives the
10^(k/2) ladder **1, 3.16, 10, 31.6, 100 kpc** (central → outskirts).
Viewing angles need **no extra patch**: stock powderday already reads
`MANUAL_ORIENTATION / THETA / PHI` from the parameter master. Verify with the Part 6 QC cell
after the first galaxy finishes.

Then run this cell, launch `submit_all_snaps.sh` under each run's `powderday_sed_out/`, and come
back to Part 6/7 when the `.rtout.sed` files exist.

In [ ]:
# ── MakeSED handles (constructor only — cheap; Parts 6–7 need just this cell, not the next) ──
from simbanator.sed.makesed import MakeSED

makeseds = {
    key: MakeSED(sim, nnodes=1, model_run_name=cfg['run_tag'],
                 hydro_dir_base=hydro_dir_base, selection_file=selection_file,
                 output_dir=sed_output_dir, run_tag=cfg['run_tag'])
    for key, cfg in RUNS.items()
}
for key, ms in makeseds.items():
    print(f"{key:9s} -> run_tag='{ms.run_tag}', master='{RUNS[key]['paramf']}'")

In [ ]:
# ── write the selection HDF5 + generate the Slurm masters (run once per sample change) ──
SEL, SNAPS, IDS = load_selection()

for key, cfg in RUNS.items():
    ms = makeseds[key]
    print(f"\n=== {key} (run_tag='{ms.run_tag}', master='{cfg['paramf']}') ===")
    ms.selection_gals(snaps=SNAPS, galaxyID=IDS)                 # same sources for both runs
    ms.create_master('cluster', 'region', radius=R_CUTOUT_KPC,
                     partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
                     prefix=PARTICLE_PREFIX, paramf=cfg['paramf'], snaps_to_run=None)

# Part 6 — Aperture QC (run after the first `.rtout.sed` exists)

Confirms the powderday patch took effect **before** burning time on the full extraction:
reads the aperture layout stored in one output file, probes every aperture index, and prints
the expected index → radius mapping.

In [ ]:
from simbanator.sed.makesed import list_sed_apertures, _read_sed

_pat = os.path.join(makeseds['dust_on'].model_dir_base, 'snap_*', 'gal_*', '*.rtout.sed')
_cands = sorted(glob.glob(_pat))
if not _cands:
    raise FileNotFoundError(f"no .rtout.sed yet under {makeseds['dust_on'].model_dir_base} — "
                            "run the RT jobs first")
_probe = _cands[0]
print("probing:", _probe, "\n")

for gname, entry in list_sed_apertures(_probe).items():
    print(f"[{gname}] seds shape = {entry.get('seds_shape')}")
    for k, v in entry.get('seds_attrs', {}).items():
        print(f"    seds.attrs[{k!r}] = {v}")
    for k, v in entry['group_attrs'].items():
        print(f"    group.attrs[{k!r}] = {v}")

print("\nexpected mapping (log-spaced, from the parameter master):")
for i, r in enumerate(APERTURE_RADII_KPC):
    mark = (f"   <- {APERTURE_LABELS[WANTED_AP_IDX.index(i)]}"
            if i in WANTED_AP_IDX else "")
    print(f"  aperture={i:2d} -> {r:7.2f} kpc{mark}")

n_ok = 0
for i in range(N_AP):
    try:
        wav, flx, unc = _read_sed(_probe, aperture=i, uncertainties=True)
        assert np.shape(flx)[0] == N_INCL, (
            f"{np.shape(flx)[0]} inclination(s) in the rtout but N_INCL={N_INCL} — "
            "THETA/PHI here disagree with the parameter master the jobs copied")
        has_unc = unc is not None and np.isfinite(np.asarray(unc)).any()
        print(f"  aperture={i}: OK  flux shape={np.shape(flx)}  MC uncertainties={'yes' if has_unc else 'NO'}")
        n_ok += 1
    except Exception as e:
        print(f"  aperture={i}: FAILED ({type(e).__name__}: {e})")
assert n_ok == N_AP, (
    f"only {n_ok}/{N_AP} apertures readable — the powderday aperture patch is NOT active "
    "(or N_AP here disagrees with SED_APERTURE_NAP in the parameter master the jobs copied)")
print(f"\nOK — {N_AP} apertures present.")

# Part 7 — Stage 2: per-aperture flux extraction → catalogs

Filter set (2026-07-10): **Subaru/HSC, CFHT/MegaCam, HST/WFC3, JWST/NIRCam+MIRI,
Spitzer/MIPS (24/70/160 µm) + Herschel/PACS+SPIRE (70–500 µm — the dust-emission
peak), VISTA/VIRCAM, JCMT/SCUBA-2, ALMA band 6** (custom 211–275 GHz top-hat,
`ALMA_band6.res`) + the Johnson V/U & 2MASS J locals kept for Part 7a.

For each dust run × aperture: convolve the SED (and its Monte-Carlo uncertainty) with the filter
set, then join the sample metadata. **One catalog per aperture per dust mode** under
`output/cis25/sed_aperture_catalogs/`, columns: `gal_id, snap, redshift`, sample metadata
(`agn_class, log_mstar, ngas, t_sft, t_qt, tau_q, xstr_quench`, …) and per-filter
`<filter>` / `<filter>_err` fluxes (mJy, rest-frame convolution as in `test_powderday.ipynb`).

In [ ]:
# ── filter set (user-chosen mock-observation instruments, 2026-07-09) ──
# optical: Subaru/HSC, CFHT/MegaCam, HST/WFC3; near-IR: JWST/NIRCam, VISTA
# (SVO lists it as Paranal/VIRCAM; CIGALE names it paranal.vircam.*);
# mid-IR: JWST/MIRI; far-IR: Spitzer/MIPS 24/70/160 um (samples the dust
# emission peak that MIRI + the sub-mm bands only straddle); sub-mm/mm:
# Herschel/PACS (70/100/160 um) + SPIRE (250/350/500 um) bracket the cold-dust
# peak (added 2026-07-10, user request); sub-mm/mm: JCMT/SCUBA-2 + ALMA band 6
# (local top-hat, 211-275 GHz — the SED grid is 0.001-1000 um REST, so
# observed-frame band 6 is covered at every anchor; at z=0.3 its red edge is
# truncated at 1.31 mm).
FACILITIES  = ['Subaru', 'CFHT', 'HST', 'JWST', 'JWST', 'Spitzer',
               'Herschel', 'Herschel', 'Paranal', 'JCMT']
INSTRUMENTS = ['HSC', 'MegaCam', 'WFC3', 'NIRCam', 'MIRI', 'MIPS',
               'PACS', 'SPIRE', 'VIRCAM', 'SCUBA2']

local_filters = {
    # Johnson V/U + 2MASS J stay for Part 7a A_V (not fitted: 7e drops them)
    '2MASS':   {'J': {'J': REMOTE_HOME + '/2MASS_J.res'}},
    'Johnson': {'V': {'V': REMOTE_HOME + '/maiz-apellaniz_Johnson_V.res'}},
    # separate top-level key required: dict cannot hold two entries under 'Johnson'
    'Johnson2': {'U': {'U': REMOTE_HOME + '/maiz-apellaniz_Johnson_U.res'}},
    # custom: not in SVO; the matching ALMA_band6_cigale.dat must be added to
    # the CIGALE env once: pcigale-filters add ALMA_band6_cigale.dat
    'ALMA': {'ALMA': {'band6': REMOTE_HOME + '/ALMA_band6.res'}},
}

SEL, SNAPS, IDS = load_selection()

def extract_flux_set(redshift, prefix):
    """extract_flux_batch over every (dust mode, aperture rung, sightline).

    redshift=False -> rest-frame (this part's catalogs); True -> observed
    frame (the Part 7b CIGALE inputs). One pass = 2 x 5 x 4 = 40 extractions;
    returns {(dust key, aperture label, incl label): flux-table path}.
    """
    files = {}
    for key in RUNS:
        ms = makeseds[key]
        for i, label in zip(WANTED_AP_IDX, APERTURE_LABELS):
            for j, ilab in enumerate(INCL_LABELS):
                print(f"\n=== extract: {key} / {label} / {ilab} (aperture {i}, "
                      f"inclination {j}, {'observed' if redshift else 'rest'} frame) ===")
                flux_file, _ = ms.extract_flux_batch(
                    SNAPS, IDS, FACILITIES, INSTRUMENTS,
                    filters=None, local_filters=local_filters, wave_unit='micron',
                    findx=j, aperture=i, uncertainties=True, redshift=redshift,
                    outname=f"{prefix}_{key}_{label}_{ilab}.fits")
                files[(key, label, ilab)] = flux_file
    return files

FLUX_FILES = extract_flux_set(redshift=False, prefix="fluxes")   # rest frame


In [ ]:
# ── final catalogs: fluxes+errors ⨝ sample metadata; one file per (dust mode, aperture) ──
_META = ["gal_id", "snap", "z_snap", "z_target", "agn_class", "xstr_quench",
         "log_mstar", "ngas", "nstar", "ssfr", "t_sft", "t_qt", "tau_q", "tau_q_over_tH",
         "r50_star_kpc", "flag_too_large", "flag_unresolved"]
SEL, SNAPS, IDS = load_selection()
_META = [c for c in _META if c in SEL.colnames]   # size-QC columns exist after Part 3b

CATALOGS = {}
for (key, label, ilab), ff in FLUX_FILES.items():
    t = Table.read(ff)
    if len(t) == 0:
        print(f"[{key}/{label}/{ilab}] EMPTY flux table — skipped"); continue
    t.rename_column('gal_id_at_snap', 'gal_id')
    cat = join(t, SEL[_META], keys=['snap', 'gal_id'], join_type='left')
    flux_cols = [c for c in t.colnames if c not in ('gal_id', 'snap', 'redshift')]
    cat = cat[['gal_id', 'snap', 'redshift'] + [c for c in _META if c not in ('gal_id', 'snap')]
              + flux_cols]
    out = os.path.join(CATDIR, f"catalog_{key}_{label}_{ilab}.fits")
    _k = APERTURE_LABELS.index(label)
    cat.meta['APERTURE'] = label
    cat.meta['APIDX'] = WANTED_AP_IDX[_k]
    cat.meta['APKPC'] = float(APERTURE_RADII_KPC[WANTED_AP_IDX[_k]])   # true rung radius
    cat.meta['INCL'] = ilab
    cat.meta['THETA'] = THETA_DEG[INCL_LABELS.index(ilab)]
    cat.meta['PHI'] = PHI_DEG[INCL_LABELS.index(ilab)]
    cat.meta['DUSTRUN'] = key
    cat.write(out, overwrite=True)
    CATALOGS[(key, label, ilab)] = out
    n_err = sum(1 for c in cat.colnames if c.endswith('_err'))
    print(f"[{key}/{label}/{ilab}] {len(cat)} galaxies, {n_err} error columns -> {out}")

# ── cross-check: per aperture, every run must contain the same sources as dust_on ──
print()
for label in APERTURE_LABELS:
    for ilab in INCL_LABELS:
        fon = FLUX_FILES.get(('dust_on', label, ilab))
        if fon is None:
            continue
        t_on = Table.read(fon)
        s_on = set(zip(np.asarray(t_on['snap'], int), np.asarray(t_on['gal_id_at_snap'], int)))
        for key in (k for k in RUNS if k != 'dust_on'):
            fk = FLUX_FILES.get((key, label, ilab))
            if fk is None:
                continue
            t_k = Table.read(fk)
            s_k = set(zip(np.asarray(t_k['snap'], int), np.asarray(t_k['gal_id_at_snap'], int)))
            status = "OK" if s_on == s_k else f"MISMATCH on={sorted(s_on - s_k)} {key}={sorted(s_k - s_on)}"
            print(f"{label:>10s}/{ilab}: dust_on={len(s_on)} {key}={len(s_k)} -> {status}")

# Part 7a — Dust attenuation $A_V$ from the matched dust_on / dust_off fluxes

We already have dust_on **and** dust_off fluxes for the *same* galaxies, so the rest-frame
band attenuation is a direct differential measurement — no SED fit needed:

$$A_\lambda = -2.5\,\log_{10}\!\left(\frac{F_{\rm dust\_on}}{F_{\rm dust\_off}}\right)\ \ [\mathrm{mag}]$$

measured per aperture from the Part-7 catalogs (rest-frame `Johnson.V.V` for $A_V$, plus
`Johnson2.U.U`/`2MASS.J.J` for the curve slope $A_U\!-\!A_V$). The **radial** attenuation
is built the observational way: annular fluxes $F(<r_{\rm out})-F(<r_{\rm in})$ between
consecutive aperture rungs give $A_V$ per annulus (annuli whose differential flux goes
non-positive from MC noise are masked).

$A_V$ is correlated against

- the **anchor-epoch ISM**: $f_{\rm mol}$, $M_{H_2}/M_\star$, $f_{\rm gas}$, dust-to-gas and
  $f_{\rm dust}=M_{\rm dust}/M_\star$ (histories, row 0), plus $\kappa_{\rm rot}$ of the H$_2$
  gas disk — the H$_2$-mass-weighted Sales+2012 $\kappa_{\rm rot}$ inside 20 pkpc, computed
  from the Stage-0 region cutouts (caesar's all-gas $\kappa_{\rm rot}$ kept for comparison);
- the **quench diagnostics + stellar structure at the observation epoch**: sSFR, $M_\star$,
  mass-weighted age, $\log Z_\star/Z_\odot$, stellar $B/T$ (caesar `rotation.stellar_BoverT`),
  $\tau_q$, $\tau_q/t_H$ and the AGN class.

Outputs: `tables/attenuation_vs_ism.fits` (per-galaxy $A_\lambda$ + ISM + structure +
quench/AGN, with $A_V$ per aperture **and** per annulus), a Spearman-ranked correlation
table, and three figures (`attenuation_vs_ism.png`, `attenuation_vs_quench.png`,
`attenuation_aperture_curve.png`).

*Needs Parts 0–3 (histories under `SFHDIR`), the Part-4 region cutouts (for
$\kappa_{\rm rot}^{H_2}$), the anchor caesar catalogs (for $Z$, $B/T$, $\kappa_{\rm rot}$)
and Part 7 (flux catalogs); CIGALE is **not** required.*

In [ ]:
# ── Part 7a — Dust attenuation (A_λ) from dust_on/dust_off vs ISM & quenching ──
# Since we already have matched dust_on / dust_off fluxes for the SAME galaxies,
# the rest-frame band attenuation follows directly (no SED fit needed):
#       A_λ = -2.5 log10( F_dust_on / F_dust_off )   [mag]
# measured per aperture from the Part-7 catalogs. The RADIAL profile is built the
# observational way: annular fluxes F(<r_out) - F(<r_in) between consecutive
# aperture rungs -> A_V per annulus. A_V is then correlated against the
# anchor-epoch ISM content (H2/HI/gas/dust from the histories + kappa_rot of the
# H2 disk from the Stage-0 region cutouts) and the quench diagnostics + stellar
# structure (mass, age, metallicity, B/T) at the observation epoch.
from scipy.stats import spearmanr
from simbanator.sed.flux_extraction import attenuation_mag as _atten

ATTEN_BANDS = {"A_U": "Johnson2.U.U", "A_V": "Johnson.V.V", "A_J": "2MASS.J.J"}
AGN_COLORS  = {"strong": "#c0392b", "intermediate": "#e67e22", "weak": "#2980b9",
               "no_AGN": "#27ae60", "no_event": "#7f8c8d", "unclassified": "#bdc3c7"}
ZSUN       = 0.0134     # Asplund+2009 total-Z scale (SIMBA's Solar reference)
R_KROT_KPC = 20.0       # H2-disk kappa_rot measured inside this proper radius

# ── 1. anchor-epoch ISM masses + stellar age (row 0 of each history) ──
GAS_KEYS = ["masses.H2", "masses.HI", "masses.gas", "masses.dust", "masses.stellar",
            "ages.mass_weighted"]
_gasdb = anchor_row0(GAS_KEYS)                                   # row 0 = anchor epoch
_missing_gas = [k for k in GAS_KEYS if not any(k in v for v in _gasdb.values())]
if _missing_gas:
    print("WARNING: ISM fields absent from histories (dropped at build):", _missing_gas)
print(f"ISM masses: {len(_gasdb)} (snap,gal) rows from the anchor histories")

def _gp_arr(tab, key):
    """anchor-epoch mass `key` aligned to a catalog table's (snap, gal_id) rows."""
    return row0_arr(_gasdb, tab["snap"], tab["gal_id"], key)

# ── 1b. anchor-epoch structure from the caesar catalogs (direct h5py read) ──
# metallicities / stellar B/T / gas kappa_rot are not tracked in the histories;
# GroupID == row index in the caesar files, so a plain dataset read suffices.
STRUCT_KEYS = {"Z_star": "metallicities.stellar", "Z_gas": "metallicities.mass_weighted",
               "BT_star": "rotation.stellar_BoverT", "kappa_gas": "rotation.gas_kappa_rot"}
_structdb, _posdb = {}, {}            # (snap,gid) -> {props} / (pos[kpccm], a, h)
_need = {}
for _s, _g in _gasdb:
    _need.setdefault(_s, set()).add(_g)
for _snap, _gids in sorted(_need.items()):
    _cf = sim.get_caesar_file(_snap)
    if not os.path.exists(_cf):
        print(f"WARNING: no caesar file for snap {_snap} -> structure props stay NaN")
        continue
    with h5py.File(_cf, "r") as f:
        _dcts = f["galaxy_data/dicts"]
        _vals = {k: _dcts[v][:] for k, v in STRUCT_KEYS.items() if v in _dcts}
        _pos  = f["galaxy_data/pos"][:]                          # kpccm
        _sa   = f["simulation_attributes"].attrs
        _a, _h = float(_sa["scale_factor"]), float(_sa["hubble_constant"])
    _absent = [v for k, v in STRUCT_KEYS.items() if k not in _vals]
    if _absent:
        print(f"WARNING: snap {_snap} caesar file lacks {_absent}")
    for _g in _gids:
        if _g < len(_pos):
            _structdb[(_snap, _g)] = {k: float(_arr[_g]) for k, _arr in _vals.items()}
            _posdb[(_snap, _g)]    = (np.asarray(_pos[_g], float), _a, _h)
print(f"structure props: {len(_structdb)} (snap,gal) rows from the anchor caesar catalogs")

def _sp_arr(tab, key):
    """anchor-epoch structure prop `key` aligned to a catalog table's rows."""
    return np.array([_structdb.get((int(s), int(g)), {}).get(key, np.nan)
                     for s, g in zip(tab["snap"], tab["gal_id"])])

# ── 1c. kappa_rot of the H2 gas disk from the Stage-0 region cutouts ──
def _kappa_rot_h2(snap, gid):
    """Sales+12 kappa_rot of the H2-mass-weighted gas within R_KROT_KPC (proper).

    Cutout Coordinates are code units (ckpc/h) unwrapped around the caesar centre,
    so pos_kpccm*h recovers that centre exactly; the uniform sqrt(a) factor of the
    code velocities cancels in the K_rot/K ratio.
    """
    rec = _posdb.get((int(snap), int(gid)))
    if rec is None:
        return np.nan
    pos_kpccm, a_scale, hub = rec
    cut = read_cutout(snap, gid, pos_kpccm * hub, "PartType0",
                      fields=("Velocities", "Masses", "FractionH2"))
    if cut is None or cut["FractionH2"] is None:
        return np.nan
    r   = cut["pos"]                                            # proper kpc, gal frame
    v   = np.asarray(cut["Velocities"], float)
    mh2 = np.asarray(cut["Masses"], float) * np.asarray(cut["FractionH2"], float)
    sel = (np.sqrt(np.sum(r**2, axis=1)) < R_KROT_KPC) & (mh2 > 0) & np.isfinite(mh2)
    if sel.sum() < 10:
        return np.nan
    r, vv, w = r[sel], v[sel], mh2[sel]
    r  = r  - np.average(r,  axis=0, weights=w)                 # recentre on the H2 body
    vv = vv - np.average(vv, axis=0, weights=w)
    j  = np.cross(r, vv)
    L  = np.sum(w[:, None] * j, axis=0)
    if not np.isfinite(L).all() or np.linalg.norm(L) == 0:
        return np.nan
    zhat = L / np.linalg.norm(L)
    jz   = j @ zhat
    Rcyl = np.sqrt(np.maximum(np.sum(r**2, axis=1) - (r @ zhat)**2, 0.0))
    ok   = Rcyl > 1e-3
    Krot = 0.5 * np.sum(w[ok] * (jz[ok] / Rcyl[ok])**2)
    Ktot = 0.5 * np.sum(w * np.sum(vv**2, axis=1))
    return float(Krot / Ktot) if Ktot > 0 else np.nan

# ── 2. per-aperture attenuation table (dust_on ⨝ dust_off on snap+gal_id) ──
ATTEN_INCL = INCL_LABELS[0]     # fiducial sightline for the A_λ analysis
def _load_atten(label, incl=None):
    incl = ATTEN_INCL if incl is None else incl
    fon  = os.path.join(CATDIR, f"catalog_dust_on_{label}_{incl}.fits")
    foff = os.path.join(CATDIR, f"catalog_dust_off_{label}_{incl}.fits")
    if not (os.path.exists(fon) and os.path.exists(foff)):
        return None
    on, off = Table.read(fon), Table.read(foff)
    off_cols = ["snap", "gal_id"] + [c for c in ATTEN_BANDS.values() if c in off.colnames]
    m = join(on, off[off_cols], keys=["snap", "gal_id"],
             table_names=["on", "off"], metadata_conflicts="silent")
    for aname, col in ATTEN_BANDS.items():
        m[aname] = (_atten(m[f"{col}_on"], m[f"{col}_off"])
                    if f"{col}_on" in m.colnames and f"{col}_off" in m.colnames
                    else np.full(len(m), np.nan))
    return m

_atab = {lab: _load_atten(lab) for lab in APERTURE_LABELS}
_atab = {k: v for k, v in _atab.items() if v is not None and len(v)}
if not _atab:
    raise FileNotFoundError(f"no dust_on/dust_off catalogs in {CATDIR}; run Part 7 first")
FID_AP = APERTURE_LABELS[-1] if APERTURE_LABELS[-1] in _atab else list(_atab)[-1]
print(f"apertures with catalogs: {list(_atab)}  |  fiducial (global A_V) = {FID_AP}"
      f"  |  sightline = {ATTEN_INCL}")

# ── 3. fiducial-aperture analysis table: A_λ + ISM tracers + structure + quench ──
base  = _atab[FID_AP]
_MH2  = _gp_arr(base, "masses.H2");   _MHI = _gp_arr(base, "masses.HI")
_Mgas = _gp_arr(base, "masses.gas");  _Md  = _gp_arr(base, "masses.dust")
_Mst  = _gp_arr(base, "masses.stellar")
with np.errstate(all="ignore"):
    f_mol       = _MH2 / (_MH2 + _MHI)          # molecular fraction of neutral gas
    f_H2_star   = _MH2 / _Mst                    # specific molecular content
    f_gas       = _Mgas / (_Mgas + _Mst)         # gas fraction
    DGR         = _Md / _Mgas                     # dust-to-gas ratio
    f_dust_star = _Md / _Mst                      # f_dust: specific dust content

ATTEN = Table()
ATTEN["snap"]     = np.asarray(base["snap"], int)
ATTEN["gal_id"]   = np.asarray(base["gal_id"], int)
ATTEN["z_target"] = np.asarray(base["z_target"], float)
for aname in ATTEN_BANDS:
    ATTEN[aname] = np.asarray(base[aname], float)
ATTEN["S_UV"] = ATTEN["A_U"] - ATTEN["A_V"]      # attenuation-curve slope proxy (mag)
for lab, tt in _atab.items():                    # enclosed A_V at every aperture
    idx = {(int(s), int(g)): k for k, (s, g) in enumerate(zip(tt["snap"], tt["gal_id"]))}
    col = np.full(len(base), np.nan)
    for k, (s, g) in enumerate(zip(base["snap"], base["gal_id"])):
        j = idx.get((int(s), int(g)))
        if j is not None:
            col[k] = tt["A_V"][j]
    ATTEN[f"A_V_{lab}"] = col
with np.errstate(all="ignore"):
    ATTEN["log_MH2"]   = np.log10(np.where(_MH2 > 0, _MH2, np.nan))
    ATTEN["log_Mgas"]  = np.log10(np.where(_Mgas > 0, _Mgas, np.nan))
    ATTEN["log_Mdust"] = np.log10(np.where(_Md > 0, _Md, np.nan))
ATTEN["f_mol"] = f_mol; ATTEN["f_H2_star"] = f_H2_star; ATTEN["f_gas"] = f_gas
ATTEN["DGR"] = DGR; ATTEN["f_dust_star"] = f_dust_star
for c in ("log_mstar", "ssfr", "tau_q", "tau_q_over_tH", "xstr_quench"):
    ATTEN[c] = np.asarray(base[c], float)
_agn = np.asarray(base["agn_class"])
ATTEN["agn_class"] = np.array([a.decode() if isinstance(a, (bytes, np.bytes_)) else str(a)
                               for a in _agn])

# anchor-epoch stellar structure + gas-disk rotation (observation time)
ATTEN["age_mw"] = _gp_arr(base, "ages.mass_weighted")          # Gyr, mass-weighted
with np.errstate(all="ignore"):
    ATTEN["logZ_star"] = np.log10(_sp_arr(base, "Z_star") / ZSUN)
    ATTEN["logZ_gas"]  = np.log10(_sp_arr(base, "Z_gas") / ZSUN)
ATTEN["BT_star"]   = _sp_arr(base, "BT_star")
ATTEN["kappa_gas"] = _sp_arr(base, "kappa_gas")
ATTEN["kappa_H2"]  = np.array([_kappa_rot_h2(s, g)
                               for s, g in zip(base["snap"], base["gal_id"])])
print(f"kappa_H2 (<{R_KROT_KPC:g} pkpc): measured for "
      f"{int(np.isfinite(np.asarray(ATTEN['kappa_H2'])).sum())}/{len(ATTEN)} galaxies "
      f"(needs the Stage-0 cutouts + >=10 H2-bearing gas particles)")

# ── 3b. annular A_V — the observational radial profile: F(<r_out) - F(<r_in) ──
_labs_all = [l for l in APERTURE_LABELS if l in _atab]
_r_out    = np.array([APERTURE_RADII_KPC[WANTED_AP_IDX[APERTURE_LABELS.index(l)]]
                      for l in _labs_all])                     # TRUE rung radii [pkpc]
_r_in     = np.concatenate([[0.0], _r_out[:-1]])
_r_mid    = np.where(_r_in > 0, np.sqrt(_r_in * _r_out), _r_out / 2.0)

def _band_matrix(col):
    """(n_gal, n_ap) matrix of catalog column `col`, aligned to the base rows."""
    M = np.full((len(base), len(_labs_all)), np.nan)
    for k, lab in enumerate(_labs_all):
        tt = _atab[lab]
        if col not in tt.colnames:
            continue
        idx = {(int(s), int(g)): j for j, (s, g) in enumerate(zip(tt["snap"], tt["gal_id"]))}
        for i, (s, g) in enumerate(zip(base["snap"], base["gal_id"])):
            j = idx.get((int(s), int(g)))
            if j is not None:
                M[i, k] = tt[col][j]
    return M

_Von, _Voff = _band_matrix("Johnson.V.V_on"), _band_matrix("Johnson.V.V_off")
_dVon  = np.column_stack([_Von[:, :1],  np.diff(_Von,  axis=1)])   # annular fluxes
_dVoff = np.column_stack([_Voff[:, :1], np.diff(_Voff, axis=1)])
AV_ANN = _atten(_dVon, _dVoff)
for k, lab in enumerate(_labs_all):
    ATTEN[f"A_V_ann_{lab}"] = AV_ANN[:, k]
_nneg = int(np.sum(((_dVon <= 0) | (_dVoff <= 0)) & np.isfinite(_Von) & np.isfinite(_Voff)))
print(f"annular A_V: {len(_labs_all)} annuli/galaxy; {_nneg} annuli with non-positive "
      f"differential flux (MC noise / empty annulus) -> NaN")

_out = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
ATTEN.write(_out, overwrite=True)
_av = np.asarray(ATTEN["A_V"], float)
print(f"attenuation table: {len(ATTEN)} galaxies ({FID_AP}) -> {_out}")
print(f"A_V [{FID_AP}]  median={np.nanmedian(_av):.3f}  90th={np.nanpercentile(_av,90):.3f}  "
      f"max={np.nanmax(_av):.3f}   (A_V>0.1 mag: {int(np.nansum(_av>0.1))}/{len(ATTEN)})")

# ── 4. Spearman correlations of the global A_V with everything ──
_targets = [("f_mol","f_mol"), ("f_H2_star","M_H2/M*"), ("f_gas","f_gas"),
            ("DGR","dust/gas"), ("f_dust_star","f_dust"), ("log_MH2","log M_H2"),
            ("log_Mgas","log M_gas"), ("kappa_H2","kappa_H2"), ("kappa_gas","kappa_gas"),
            ("log_mstar","log M*"), ("age_mw","age_mw"), ("logZ_star","log Z*/Zsun"),
            ("logZ_gas","log Zg/Zsun"), ("BT_star","B/T"), ("ssfr","sSFR"),
            ("tau_q","tau_q"), ("tau_q_over_tH","tau_q/t_H"), ("S_UV","A_U-A_V")]
_ranked = []
for col, lbl in _targets:
    x = np.asarray(ATTEN[col], float); ok = np.isfinite(_av) & np.isfinite(x)
    if ok.sum() >= 5:
        rho, p = spearmanr(_av[ok], x[ok]); _ranked.append((lbl, rho, p, int(ok.sum())))
_ranked.sort(key=lambda r: -abs(r[1]))
print("\nSpearman  A_V  vs …   (fiducial aperture, sorted by |rho|)")
print(f"  {'quantity':12s} {'rho':>7s} {'p':>10s} {'n':>4s}")
for lbl, rho, p, n in _ranked:
    flag = "***" if p < 0.01 else "** " if p < 0.05 else "*  " if p < 0.1 else ""
    print(f"  {lbl:12s} {rho:+7.3f} {p:10.2e} {n:4d}  {flag}")

# per-aperture robustness of the two headline ISM correlations
print("\nrobustness across apertures  (rho[p]):")
for lab in _labs_all:
    tt = _atab[lab]; av = np.asarray(tt["A_V"], float)
    with np.errstate(all="ignore"):
        fh2 = _gp_arr(tt, "masses.H2") / _gp_arr(tt, "masses.stellar")
        dgr = _gp_arr(tt, "masses.dust") / _gp_arr(tt, "masses.gas")
    def _rp(x):
        ok = np.isfinite(av) & np.isfinite(x)
        return spearmanr(av[ok], x[ok]) if ok.sum() >= 5 else (np.nan, np.nan)
    (r1, p1), (r2, p2) = _rp(fh2), _rp(dgr)
    print(f"  {lab:16s}  M_H2/M*: {r1:+.2f}[{p1:.2g}]   dust/gas: {r2:+.2f}[{p2:.2g}]")

# ── 5. figures ──
_cls_present = [c for c in ["strong","intermediate","weak","no_AGN","no_event","unclassified"]
                if c in set(ATTEN["agn_class"])]
def _scatter(ax, xcol, xlabel, xlog=False):
    x = np.asarray(ATTEN[xcol], float); y = _av
    for cls in _cls_present:
        s = ATTEN["agn_class"] == cls
        ax.scatter(x[s], y[s], s=28, c=AGN_COLORS.get(cls, "#333"), label=cls,
                   edgecolor="k", linewidth=0.3, alpha=0.85)
    ok = np.isfinite(x) & np.isfinite(y)
    if xlog: ok &= x > 0
    if ok.sum() >= 5:
        rho, p = spearmanr(x[ok], y[ok])
        ax.text(0.04, 0.95, f"$\\rho$={rho:+.2f}\np={p:.2g}", transform=ax.transAxes,
                va="top", fontsize=8.5, bbox=dict(fc="white", ec="0.7", alpha=0.85, pad=1.6))
    if xlog:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel(r"$A_V$ [mag]")

# Fig 1 — attenuation vs ISM / dust content (the H2 connection)
_p1 = [("f_mol", r"$f_{\rm mol}=M_{H_2}/(M_{H_2}\!+\!M_{HI})$", False),
       ("f_H2_star", r"$M_{H_2}/M_\star$", True),
       ("f_gas", r"$f_{\rm gas}=M_{\rm gas}/(M_{\rm gas}\!+\!M_\star)$", False),
       ("DGR", r"dust-to-gas $M_{\rm dust}/M_{\rm gas}$", True),
       ("f_dust_star", r"$f_{\rm dust}=M_{\rm dust}/M_\star$", True),
       ("kappa_H2", r"$\kappa_{\rm rot}^{H_2}$ ($<$%g pkpc)" % R_KROT_KPC, False)]
fig, axes = plt.subplots(2, 3, figsize=(15, 8.6))
for ax, (c, xl, xlog) in zip(axes.flat, _p1):
    _scatter(ax, c, xl, xlog)
axes.flat[0].legend(fontsize=7.5, loc="upper right", framealpha=0.9)
fig.suptitle(f"Rest-frame $A_V$ (dust_on/dust_off, {FID_AP}) vs anchor-epoch ISM content",
             fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
_f1 = os.path.join(PLOTDIR, "attenuation_vs_ism.png")
fig.savefig(_f1, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f1)

# Fig 2 — attenuation vs quenching + stellar structure at the observation epoch
_p2 = [("ssfr", "sSFR [yr$^{-1}$]", True),
       ("log_mstar", r"$\log_{10} M_\star\,[M_\odot]$", False),
       ("age_mw", "mass-weighted age [Gyr]", False),
       ("logZ_star", r"$\log_{10} Z_\star/Z_\odot$", False),
       ("BT_star", r"stellar $B/T$", False),
       ("tau_q", r"$\tau_q$ [yr]", False),
       ("tau_q_over_tH", r"$\tau_q/t_H$", False)]
fig, axes = plt.subplots(2, 4, figsize=(18.5, 8.4))
for ax, (c, xl, xlog) in zip(axes.flat[:7], _p2):
    _scatter(ax, c, xl, xlog)
ax = axes.flat[7]
for i, cls in enumerate(_cls_present):
    s = ATTEN["agn_class"] == cls; yv = _av[np.asarray(s)]
    xj = i + np.random.uniform(-0.16, 0.16, size=int(np.sum(s)))
    ax.scatter(xj, yv, c=AGN_COLORS.get(cls, "#333"), edgecolor="k", linewidth=0.3, s=28)
    if np.isfinite(yv).any():
        ax.hlines(np.nanmedian(yv), i - 0.3, i + 0.3, color="k", lw=2)
ax.set_xticks(range(len(_cls_present)))
ax.set_xticklabels(_cls_present, rotation=30, ha="right", fontsize=8.5)
ax.set_ylabel(r"$A_V$ [mag]"); ax.set_title("by AGN class", fontsize=9)
fig.suptitle(f"Rest-frame $A_V$ ({FID_AP}) vs quenching & stellar structure "
             f"at the observation epoch", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
_f2 = os.path.join(PLOTDIR, "attenuation_vs_quench.png")
fig.savefig(_f2, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f2)

# Fig 3 — radial A_V from annular fluxes + attenuation-curve slope
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.8))
_AVap  = np.vstack([np.asarray(ATTEN[f"A_V_{l}"], float) for l in _labs_all]).T
_AVann = np.vstack([np.asarray(ATTEN[f"A_V_ann_{l}"], float) for l in _labs_all]).T
_dusty = _av > AV_DUSTY                                   # most quenched gals are transparent;
for row in _AVann[_dusty]:                           # the radial trend only matters there
    axL.plot(_r_mid, row, "-", color="0.7", lw=0.8, alpha=0.7, zorder=1)
axL.plot(_r_mid, np.nanmedian(_AVann[_dusty], axis=0), "o-", color="#c0392b", lw=2,
         label=f"annular median, $A_V\\!>\\!0.1$ (n={int(_dusty.sum())})", zorder=3)
axL.plot(_r_mid, np.nanmedian(_AVann, axis=0), "s--", color="#2980b9",
         label=f"annular median, all (n={len(_av)})", zorder=2)
axL.plot(_r_out, np.nanmedian(_AVap[_dusty], axis=0), ":", color="0.35", lw=1.5,
         label=r"enclosed $A_V(<r)$, $A_V\!>\!0.1$", zorder=2)
axL.set_xscale("log"); axL.set_xlabel("radius [pkpc]")
axL.set_ylabel(r"$A_V$ [mag]")
axL.set_title(r"radial $A_V$: annuli $F(<r_{\rm out})-F(<r_{\rm in})$")
axL.legend(fontsize=8)
for cls in _cls_present:
    s = ATTEN["agn_class"] == cls
    axR.scatter(_av[np.asarray(s)], np.asarray(ATTEN["S_UV"])[np.asarray(s)], s=28,
                c=AGN_COLORS.get(cls, "#333"), edgecolor="k", linewidth=0.3, alpha=0.85, label=cls)
axR.axhline(0, color="0.6", lw=0.8, ls="--")
axR.set_xlabel(r"$A_V$ [mag]"); axR.set_ylabel(r"$A_U-A_V$ [mag]  (curve slope)")
axR.set_title("attenuation depth vs UV-optical reddening"); axR.legend(fontsize=7.5)
fig.tight_layout()
_f3 = os.path.join(PLOTDIR, "attenuation_aperture_curve.png")
fig.savefig(_f3, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f3)

# Part 7a-agn — AGN contribution to the observed SEDs (`agn_on` vs `dust_on`)

The `agn_on` run re-radiates the **same galaxies through the same live-dust grid** with the
SIMBA BHs added as point sources (`parameters_master-agn.py`: Hopkins+2007 intrinsic quasar
template, `BH_var=False` so $L_{\rm bol}=0.1\,\dot M_{\rm BH}c^2$ follows the SIMBA accretion
rates). Differencing the matched Part-7 catalogs per (band, aperture, sightline) isolates the
dust-reprocessed AGN contribution — no SED fit involved:

- $f_{\rm AGN} = 1 - F_{\rm on}/F_{\rm agn}$ — fractional AGN contribution per band (0 = none);
- $\Delta m = -2.5\log_{10}(F_{\rm agn}/F_{\rm on}) \le 0$ — the AGN brightening [mag]. This is
  **exactly the bias on the Part-7a differential $A_V$** if the dusty frame contains an
  unrecognized AGN (the `dust_off` reference cancels): an AGN mimics *negative* extra
  attenuation, strongest in the central apertures and blue bands.

Residual significance uses the propagated MC errors of both runs. Per-galaxy rows (fiducial
sightline) with per-aperture $f_{\rm AGN}$/$\Delta m$ in the V band and the U/V/J spectral
shape at the central + global apertures → `tables/agn_flux_contribution.fits` +
`plots/.../agn_sed_contribution.png`. The CIGALE-side counterpart (how the unmodeled AGN
shifts $M_*$, ages, SFH, $M_{\rm dust}$) is Part 7h's `agn_on` pairing.


In [ ]:
# ── Part 7a-agn — AGN contribution to the observed SEDs (agn_on vs dust_on) ──
# Self-contained after Part 0 (+ Part 7 catalogs on disk). See the markdown above:
# f_AGN = 1 - F_on/F_agn per band; dm = -2.5 log10(F_agn/F_on) = the A_V bias.
from scipy.stats import spearmanr as _spearman_agn

AGN_BANDS = {"U": "Johnson2.U.U", "V": "Johnson.V.V", "J": "2MASS.J.J"}  # rest-frame
AGN_INCL  = INCL_LABELS[0]              # fiducial sightline (matches Part 7a)
_AGN_COLORS = {"strong": "#c0392b", "intermediate": "#e67e22", "weak": "#2980b9",
               "no_AGN": "#27ae60", "no_event": "#7f8c8d", "unclassified": "#bdc3c7"}

def _load_agn_pair(label, incl=AGN_INCL):
    """catalog_agn_on ⨝ catalog_dust_on on (snap, gal_id) + f_AGN/dm/S_N per band."""
    fa = os.path.join(CATDIR, f"catalog_agn_on_{label}_{incl}.fits")
    fo = os.path.join(CATDIR, f"catalog_dust_on_{label}_{incl}.fits")
    if not (os.path.exists(fa) and os.path.exists(fo)):
        return None
    agn, on = Table.read(fa), Table.read(fo)
    _keep = set(AGN_BANDS.values()) | {f"{c}_err" for c in AGN_BANDS.values()}
    m = join(on, agn[["snap", "gal_id"] + [c for c in agn.colnames if c in _keep]],
             keys=["snap", "gal_id"], table_names=["on", "agn"],
             metadata_conflicts="silent")
    for _bn, _col in AGN_BANDS.items():
        _ca, _co = f"{_col}_agn", f"{_col}_on"
        _Fa = (np.asarray(m[_ca], float) if _ca in m.colnames
               else np.full(len(m), np.nan))
        _Fo = (np.asarray(m[_co], float) if _co in m.colnames
               else np.full(len(m), np.nan))
        with np.errstate(all="ignore"):
            _ok = np.isfinite(_Fa) & np.isfinite(_Fo) & (_Fa > 0) & (_Fo > 0)
            m[f"fagn_{_bn}"] = np.where(_ok, 1.0 - _Fo / _Fa, np.nan)
            m[f"dm_{_bn}"]   = np.where(_ok, -2.5 * np.log10(_Fa / _Fo), np.nan)
            _ea, _eo = f"{_col}_err_agn", f"{_col}_err_on"
            if _ea in m.colnames and _eo in m.colnames:
                _sig = np.sqrt(np.asarray(m[_ea], float) ** 2
                               + np.asarray(m[_eo], float) ** 2)
                m[f"sn_{_bn}"] = np.where(_ok & (_sig > 0), (_Fa - _Fo) / _sig, np.nan)
            else:
                m[f"sn_{_bn}"] = np.full(len(m), np.nan)
    return m

_agntab = {lab: _load_agn_pair(lab) for lab in APERTURE_LABELS}
_agntab = {k: v for k, v in _agntab.items() if v is not None and len(v)}
if not _agntab:
    raise FileNotFoundError(f"no agn_on/dust_on catalog pairs in {CATDIR}; "
                            "run the agn_on RT + Part 7 first")
_labs_agn = [l for l in APERTURE_LABELS if l in _agntab]
_r_agn    = np.array([APERTURE_RADII_KPC[WANTED_AP_IDX[APERTURE_LABELS.index(l)]]
                      for l in _labs_agn])                  # true rung radii [pkpc]
AGN_CEN, AGN_GLOB = _labs_agn[0], _labs_agn[-1]
print(f"apertures with agn_on pairs: {_labs_agn}  |  central = {AGN_CEN}, "
      f"global = {AGN_GLOB}  |  sightline = {AGN_INCL}")

# ── per-galaxy table (base = global aperture; per-aperture V-band columns) ──
_gbase = _agntab[AGN_GLOB]
AGNC = Table()
AGNC["snap"]     = np.asarray(_gbase["snap"], int)
AGNC["gal_id"]   = np.asarray(_gbase["gal_id"], int)
AGNC["z_target"] = np.asarray(_gbase["z_target"], float)
for _c in ("log_mstar", "ssfr", "xstr_quench"):
    AGNC[_c] = np.asarray(_gbase[_c], float)
AGNC["agn_class"] = np.array([a.decode() if isinstance(a, (bytes, np.bytes_)) else str(a)
                              for a in _gbase["agn_class"]])
for _lab in _labs_agn:                      # V band along the aperture ladder
    _tt  = _agntab[_lab]
    _idx = {(int(s), int(g)): j for j, (s, g) in enumerate(zip(_tt["snap"], _tt["gal_id"]))}
    for _q in ("fagn_V", "dm_V", "sn_V"):
        _col = np.full(len(_gbase), np.nan)
        for _i, (_s, _g) in enumerate(zip(_gbase["snap"], _gbase["gal_id"])):
            _j = _idx.get((int(_s), int(_g)))
            if _j is not None:
                _col[_i] = _tt[_q][_j]
        AGNC[f"{_q}_{_lab}"] = _col
for _bn in AGN_BANDS:                       # U/V/J spectral shape, central + global
    for _lab, _sfx in ((AGN_CEN, "cen"), (AGN_GLOB, "glob")):
        _tt  = _agntab[_lab]
        _idx = {(int(s), int(g)): j for j, (s, g) in enumerate(zip(_tt["snap"], _tt["gal_id"]))}
        _col = np.full(len(_gbase), np.nan)
        for _i, (_s, _g) in enumerate(zip(_gbase["snap"], _gbase["gal_id"])):
            _j = _idx.get((int(_s), int(_g)))
            if _j is not None:
                _col[_i] = _tt[f"fagn_{_bn}"][_j]
        AGNC[f"fagn_{_bn}_{_sfx}"] = _col
_outf = os.path.join(TABLEDIR, "agn_flux_contribution.fits")
AGNC.write(_outf, overwrite=True)
print(f"{len(AGNC)} galaxies -> {_outf}\n")

# ── stats: per class, central vs global AGN contribution + significance ──
_cls_order = [c for c in ("strong", "intermediate", "weak", "no_AGN", "no_event",
                          "unclassified") if c in set(AGNC["agn_class"])]
print(f"{'class':>14s} | {'N':>3s} {'med f_AGN(V) cen':>17s} {'glob':>7s} "
      f"{'S/N>3 cen':>10s}")
for _cl in _cls_order:
    _m  = AGNC["agn_class"] == _cl
    _fc = np.asarray(AGNC[f"fagn_V_{AGN_CEN}"], float)[_m]
    _fg = np.asarray(AGNC[f"fagn_V_{AGN_GLOB}"], float)[_m]
    _sn = np.asarray(AGNC[f"sn_V_{AGN_CEN}"], float)[_m]
    print(f"{_cl:>14s} | {int(_m.sum()):3d} {np.nanmedian(_fc):17.4f} "
          f"{np.nanmedian(_fg):7.4f} {int(np.nansum(_sn > 3)):10d}")
_okc = (np.isfinite(np.asarray(AGNC["xstr_quench"], float))
        & np.isfinite(np.asarray(AGNC[f"fagn_V_{AGN_CEN}"], float)))
if _okc.sum() > 5:
    _rho, _p = _spearman_agn(np.asarray(AGNC["xstr_quench"], float)[_okc],
                             np.asarray(AGNC[f"fagn_V_{AGN_CEN}"], float)[_okc])
    print(f"\nSpearman f_AGN(V, {AGN_CEN}) vs xstr_quench: rho={_rho:+.2f} p={_p:.3g}"
          "   (coupling strength is jet-mode — low f_Edd — so a NEGATIVE or flat"
          "\n   trend is expected: radiatively bright AGN live in the weak-coupling class)")

# ── figure: radial dilution + coupling scatter + spectral shape ──
_fig, _axs = plt.subplots(1, 3, figsize=(15, 4.6))
_ax = _axs[0]                                       # (a) f_AGN(V) vs aperture radius
for _s2, _g2, _row in zip(AGNC["snap"], AGNC["gal_id"],
                          zip(*[AGNC[f"fagn_V_{l}"] for l in _labs_agn])):
    _ax.plot(_r_agn, _row, color="0.85", lw=0.6, zorder=0)
for _cl in _cls_order:
    _m = AGNC["agn_class"] == _cl
    _med = [np.nanmedian(np.asarray(AGNC[f"fagn_V_{l}"], float)[_m]) for l in _labs_agn]
    _ax.plot(_r_agn, _med, "o-", color=_AGN_COLORS.get(_cl, "k"), label=_cl, lw=2)
_ax.set_xscale("log"); _ax.set_xlabel("aperture radius [pkpc]")
_ax.set_ylabel(r"$f_{\rm AGN}$ (V band)")
_ax.set_title("AGN contribution vs aperture"); _ax.legend(fontsize=8, frameon=False)
_ax = _axs[1]                                       # (b) central f_AGN vs coupling
for _cl in _cls_order:
    _m = AGNC["agn_class"] == _cl
    _ax.scatter(np.asarray(AGNC["xstr_quench"], float)[_m],
                np.asarray(AGNC[f"fagn_V_{AGN_CEN}"], float)[_m],
                s=18, color=_AGN_COLORS.get(_cl, "k"), alpha=0.75, label=_cl)
_ax.set_xlabel(r"$x_{\rm str,quench}$ (AGN–ISM coupling)")
_ax.set_ylabel(rf"$f_{{\rm AGN}}$ (V, {AGN_CEN})")
_ax.set_title("central AGN light vs quench-window coupling")
_ax = _axs[2]                                       # (c) U/V/J spectral shape
_xb = np.arange(len(AGN_BANDS))
for _cl in _cls_order:
    _m = AGNC["agn_class"] == _cl
    _ax.plot(_xb, [np.nanmedian(np.asarray(AGNC[f"fagn_{b}_cen"], float)[_m])
                   for b in AGN_BANDS], "o-", color=_AGN_COLORS.get(_cl, "k"),
             label=f"{_cl} ({AGN_CEN})", lw=2)
    _ax.plot(_xb, [np.nanmedian(np.asarray(AGNC[f"fagn_{b}_glob"], float)[_m])
                   for b in AGN_BANDS], "s--", color=_AGN_COLORS.get(_cl, "k"),
             alpha=0.5, lw=1.2)
_ax.set_xticks(_xb); _ax.set_xticklabels(list(AGN_BANDS))
_ax.set_ylabel(r"median $f_{\rm AGN}$")
_ax.set_title(f"spectral shape (solid {AGN_CEN}, dashed {AGN_GLOB})")
_ax.legend(fontsize=7, frameon=False)
_fig.tight_layout()
plt.savefig(os.path.join(PLOTDIR, "agn_sed_contribution.png"),
            dpi=140, bbox_inches="tight")
plt.show()


# Part 7b — CIGALE input files (one per dust mode × aperture)

Writes `output/cis25/sed_aperture_catalogs/cigale/cigale_{dust_on|dust_off}_{ap…}.fits` in the
exact input format of **CIGALE 2025.0**. All the format/mapping logic lives in
**`simbanator.sed.cigale`** (band names verified against the 2025.0 filter database):

- columns `id` (`snapNNN_galID`), `redshift`, `distance` (Mpc, Planck13 — the same D_L used to
  normalize the fluxes), then per band the flux **in mJy** + its `<band>_err`;
- band names match the CIGALE DB exactly (`jwst.nircam.F200W`, `hst.wfc3.ir.F160W`,
  `spitzer.irac.I1`, `herschel.pacs.green`, `2mass.J`, `generic.johnson.U/V`, …); bands with no
  CIGALE counterpart (grisms, quad filters) are dropped and reported;
- missing fluxes are NaN; make an error negative by hand for upper-limit treatment.

Two deliberate choices:

1. **Observed frame.** CIGALE compares redshifted models to observed photometry, so the
   extraction reruns with `redshift=True` (the Part 7 catalogs stay rest-frame).
2. **Raw MC errors (`err_floor=0`).** CIGALE itself adds `additionalerror` (10 % by default,
   set in Part 7es `prepare_run`) in quadrature at fit time — a floor here too would be
   double-counted. The Hyperion MC error alone is just RT convergence noise.

In [ ]:
# ── Part 7b: CIGALE 2025.0 input files — observed-frame fluxes+errors, one per (dust mode, aperture, sightline) ──
# Format + band mapping live in simbanator.sed.cigale (verified against the 2025.0 filter DB).
# Needs the Part 5 MakeSED handles + the Part 7 filter/extractor cell in this session.
from simbanator.sed.cigale import write_cigale_input

os.makedirs(CIGALE_DIR, exist_ok=True)

# CIGALE compares redshifted models to observed photometry -> observed frame
# (the Part 7 catalogs stay rest-frame)
CIGALE_FLUX_FILES = extract_flux_set(redshift=True, prefix="cigale_fluxes")

CIGALE_FILES = {}
for (key, label, ilab), flux_file in CIGALE_FLUX_FILES.items():
    # err_floor=0: CIGALE adds its own 10% 'additionalerror' in quadrature at fit time
    CIGALE_FILES[(key, label, ilab)] = write_cigale_input(
        flux_file, os.path.join(CIGALE_DIR, f"cigale_{key}_{label}_{ilab}.fits"),
        err_floor=0.0)

print(f"\n{len(CIGALE_FILES)} CIGALE input files -> {CIGALE_DIR}")


# Part 7c — annular CIGALE inputs: $F(<r_{\rm out}) - F(<r_{\rm in})$ between consecutive rungs

The Part 7b catalogs are **cumulative** (each aperture contains all the inner light), so any
radial trend fitted from them is a curve-of-growth. This part builds true **annular** catalogs
the observational way — differencing the observed-frame per-aperture fluxes band by band
(`simbanator.sed.flux_extraction.annular_flux_table`), the same construction as Part 7a's
annular $A_V$. As there, the **outer** rung names the annulus: `ann3kpc` = 1→3.16 kpc, …,
`ann100kpc` = 31.6→100 kpc. `ann1kpc` (0→1 kpc) **is** the `ap1kpc` aperture and is not
duplicated — the radial set downstream is `ap1kpc + ann3kpc…ann100kpc`.

- **Non-positive annular flux** (MC noise / empty annulus) → NaN flux+error = missing band to
  CIGALE. Objects losing most bands this way will fit poorly — watch the printed counts.
- **Errors**: `sqrt(err_out² − err_in²)` — the photons inside $r_{\rm in}$ are counted in both
  rungs, so the cumulative variances subtract; where MC noise makes the difference
  non-positive, the conservative quadrature **sum** is used instead.
- **Sampling QC**: with Part 4b's `annulus_particle_counts.fits` present, sources with **zero
  star particles** in an annulus are listed per catalog — their annular flux is scattered
  light only and the CIGALE fit is meaningless (drop them via the QC table when interpreting).
- Files land next to the Part 7b ones (`cigale/cigale_{dust}_{ann…}_{incl}.fits`), so
  **Part 7e picks them up automatically** (+32 runs = 2 dust modes × 4 annuli × 4 sightlines).
  Part 7g compares annular runs against the **same-annulus** SIMBA truth
  (Part 7f measures every annulus too; annular $A_V$ truth by differencing
  the adjacent aperture catalogs).

Reads the Part 7b per-aperture `cigale_fluxes_*.fits` intermediates from disk (run
Part 7b once first); no MakeSED handles needed.

In [ ]:
# ── Part 7c: annular CIGALE inputs — F(<r_out) − F(<r_in) between consecutive rungs ──
from simbanator.sed.flux_extraction import annular_flux_table
from simbanator.sed.cigale import write_cigale_input

os.makedirs(CIGALE_DIR, exist_ok=True)

_qcf = os.path.join(TABLEDIR, "annulus_particle_counts.fits")
_qc = Table.read(_qcf) if os.path.exists(_qcf) else None
if _qc is None:
    print("[QC] annulus_particle_counts.fits not found — run Part 4b to flag star-free annuli")

CIGALE_ANN_FILES = {}
for key in RUNS:
    _fluxdir = os.path.join(sed_output_dir, RUNS[key]['run_tag'], 'sed_fluxes')
    for ilab in INCL_LABELS:
        _files = [os.path.join(_fluxdir, f"cigale_fluxes_{key}_{lab}_{ilab}.fits")
                  for lab in APERTURE_LABELS]
        _miss = [os.path.basename(p) for p in _files if not os.path.exists(p)]
        if _miss:
            raise FileNotFoundError(f"{key}/{ilab}: run Part 7b first — missing {_miss}")
        for k in range(1, len(APERTURE_LABELS)):     # k=0 (ann1kpc) == ap1kpc, already fit
            alab = ANNULUS_LABELS[k]
            print(f"\n=== annularize: {key} / {alab} "
                  f"({APERTURE_LABELS[k-1]} -> {APERTURE_LABELS[k]}) / {ilab} ===")
            _tann = annular_flux_table(_files[k - 1], _files[k])
            CIGALE_ANN_FILES[(key, alab, ilab)] = write_cigale_input(
                _tann, os.path.join(CIGALE_DIR, f"cigale_{key}_{alab}_{ilab}.fits"),
                err_floor=0.0)   # CIGALE adds its own 10% additionalerror at fit time
            if _qc is not None and key == 'dust_on':          # QC once per (annulus, incl)
                _m = np.char.strip(np.asarray(_qc["incl"], str)) == ilab
                _z = np.asarray(_qc[f"nstar_{alab}"], int)[_m] == 0
                if _z.any():
                    _who = [f"snap{int(s):03d}_gal{int(g)}"
                            for s, g in zip(np.asarray(_qc["snap"], int)[_m][_z],
                                            np.asarray(_qc["gal_id"], int)[_m][_z])]
                    print(f"    [QC] {int(_z.sum())} source(s) with ZERO star particles in "
                          f"{alab}/{ilab} (annular flux = scattered light only): {_who}")

print(f"\n{len(CIGALE_ANN_FILES)} annular CIGALE input files -> {CIGALE_DIR}")

# Part 7d — SFH-derived priors: true parameter ranges & delayed+bq fits

Before running CIGALE, check what the **simulation's own** star-formation histories imply for the fit grid. For each selected galaxy this cell:

- **smooths + resamples** the SFH first (Gaussian kernel ≈ the snapshot cadence → uniform 25 Myr grid; `simbanator.analysis.sfh_utils.smooth_resample_sfh`) — SIMBA's per-snapshot SFR is instantaneous and bursty, so the smooth CIGALE form is fitted to the smoothed track;
- tabulates the true ranges of `M*`, `sSFR`, the quench duration `tau_q`, and the quench lookback `age_bq = t_obs − t_qt`;
- fits the **delayed-τ + burst/quench** form (CIGALE `sfhdelayedbq`) to `SFR(t)`, giving the empirical priors for `tau_main`, `age_main`, `age_bq`, `r_sfr` — the only form fitted, because it is what CIGALE itself fits (it cannot reproduce SIMBA's bursty tracks in detail; the per-galaxy `r2` in `sfh_delayedbq_fits.fits` quantifies that mismatch);
- prints the p5/p50/p95 of each fitted parameter next to the **current CIGALE grid nodes**, and plots example SFHs + fits, the median stacked SFH, and the fitted-parameter distributions against those nodes.

Use the printed *suggested grid* and the dashed-node histograms to decide where to extend/refine the `sfhdelayedbq` grid in Part 7e. Requires the anchor histories (Part 1, `BUILD_MULTI_Z`) and the selection catalog (Parts 2–3).

In [ ]:
# ── Part 7d — Priors from the TRUE SFHs: parameter ranges + delayed+bq fits ──
# Self-contained after Part 1 (needs ANCHORS + load_anchor_history) and the
# selection catalog (SELECTION_FITS). From SIMBA's own star-formation histories
# it (1) tabulates the parameter ranges the CIGALE grid should span and (2) fits
# the sfhdelayedbq form to each galaxy. Plots example SFHs + fits, the median
# stacked SFH, and the parameter distributions against the current CIGALE nodes.
# (delayed+bq cannot reproduce SIMBA's bursty tracks in detail — it is kept as
# the single fit form because it is what CIGALE itself fits.)
from simbanator.sed.cigale import DEFAULT_MODULE_PARAMS
from simbanator.analysis.sfh_utils import (smooth_resample_sfh,
                                            sfr_delayed_bq, fit_delayed_bq)

# SIMBA's per-snapshot SFR is instantaneous and bursty: smooth + resample each
# track before fitting the smooth CIGALE form.
SFH_SMOOTH_KERNEL_MYR = None    # Gaussian sigma [Myr]; None -> median snapshot spacing
SFH_RESAMPLE_DT_MYR   = 25.0    # uniform output grid step [Myr]

# ── gather the selected sample's true SFHs (SFR vs cosmic time) ──
SEL, SNAPS, IDS = load_selection()
SFH = []
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        continue
    H = load_anchor_history(A)
    gids = np.asarray(H["galaxy_ids"])
    t_gyr = np.asarray(H["t_cosmic_yr"], float) / 1e9
    sfr_all = np.asarray(H["P"]["sfr"], float)          # (n_snap, n_gal)
    order = np.argsort(t_gyr)
    gid_to_col = {int(g): i for i, g in enumerate(gids)}
    for row in SEL[SEL["snap"] == int(A["snap"])]:
        col = gid_to_col.get(int(row["gal_id"]))
        if col is None:
            continue
        tg = t_gyr[order]; sg = sfr_all[order, col]
        ts, ss = smooth_resample_sfh(tg, sg, dt_myr=SFH_RESAMPLE_DT_MYR,
                                     kernel_myr=SFH_SMOOTH_KERNEL_MYR)
        t_obs = float(tg[-1])
        tqt = float(row["t_qt"]) if np.isfinite(row["t_qt"]) else np.nan
        age_bq_emp = (t_obs * 1e9 - tqt) / 1e6 if np.isfinite(tqt) else np.nan
        SFH.append(dict(
            gal_id=int(row["gal_id"]), snap=int(A["snap"]), z=float(A["z"]),
            t=ts, sfr=ss, t_raw=tg, sfr_raw=sg,
            t_obs=t_obs, log_mstar=float(row["log_mstar"]),
            ssfr=float(row["ssfr"]),
            tau_q_gyr=(float(row["tau_q"]) / 1e9 if np.isfinite(row["tau_q"]) else np.nan),
            age_bq_emp_myr=age_bq_emp))

if not SFH:
    raise RuntimeError("no SFHs to fit — build the anchor histories "
                       "(BUILD_MULTI_Z, Part 1) and the selection catalog "
                       "(Parts 2-3) first")

for g in SFH:
    _ab0 = (g["age_bq_emp_myr"] / 1e3 if np.isfinite(g["age_bq_emp_myr"]) else None)
    g["bq"] = fit_delayed_bq(g["t"], g["sfr"], g["t_obs"], age_bq0=_ab0)  # smoothed
n_bq = sum(g["bq"] is not None for g in SFH)
_kern = ("median-cadence" if SFH_SMOOTH_KERNEL_MYR is None
         else f"{SFH_SMOOTH_KERNEL_MYR:g} Myr")
print(f"[SFH priors] {len(SFH)} selected galaxies | smoothed (kernel={_kern}, "
      f"dt={SFH_RESAMPLE_DT_MYR:g} Myr) | delayedbq fit ok: {n_bq}")

# persist the per-galaxy CIGALE-model (delayed+bq) fits: Part 7g uses them as
# the SFH-parameter truth (same functional form CIGALE fits, so the comparison
# is apples-to-apples — the SFT/QT clocks are a different quantity).
SFH_FITS_TABLE = os.path.join(TABLEDIR, "sfh_delayedbq_fits.fits")
_ft = Table(dict(
    snap=[g["snap"] for g in SFH], gal_id=[g["gal_id"] for g in SFH],
    z=[g["z"] for g in SFH],
    tau_main_myr=[g["bq"]["tau_main_myr"] if g["bq"] else np.nan for g in SFH],
    age_main_myr=[g["bq"]["age_main_myr"] if g["bq"] else np.nan for g in SFH],
    age_bq_myr=[g["bq"]["age_bq_myr"] if g["bq"] else np.nan for g in SFH],
    r_sfr=[g["bq"]["r_sfr"] if g["bq"] else np.nan for g in SFH],
    r2=[g["bq"]["r2"] if g["bq"] else np.nan for g in SFH],
))
_ft.write(SFH_FITS_TABLE, overwrite=True)
print(f"[SFH priors] per-galaxy delayed+bq fits -> {SFH_FITS_TABLE}")


def _pc(vals, q=(5, 50, 95)):
    v = np.asarray(vals, float); v = v[np.isfinite(v)]
    return tuple(np.percentile(v, q)) if v.size else (np.nan,) * len(q)


_log_mstar = [g["log_mstar"] for g in SFH]
_log_ssfr = [np.log10(g["ssfr"]) for g in SFH if g["ssfr"] > 0]
_tau_q = [g["tau_q_gyr"] * 1e3 for g in SFH]
_age_bq_emp = [g["age_bq_emp_myr"] for g in SFH]
_fit_tau = [g["bq"]["tau_main_myr"] for g in SFH if g["bq"]]
_fit_agem = [g["bq"]["age_main_myr"] for g in SFH if g["bq"]]
_fit_agebq = [g["bq"]["age_bq_myr"] for g in SFH if g["bq"]]
_fit_rsfr = [g["bq"]["r_sfr"] for g in SFH if g["bq"]]

print("\n TRUE sample parameter ranges  (p5 / p50 / p95):")
def _line(name, vals, unit="", nd=2):
    a, b, c = _pc(vals)
    print(f"   {name:<26s} {a:9.{nd}f} {b:9.{nd}f} {c:9.{nd}f}  {unit}")
_line("log10 M*/Msun", _log_mstar)
_line("log10 sSFR/yr", _log_ssfr)
_line("tau_q (quench duration)", _tau_q, "Myr", 0)
_line("age_bq = t_obs - t_qt", _age_bq_emp, "Myr", 0)
print("   --- delayed-tau + burst/quench fits: ---")
_line("tau_main", _fit_tau, "Myr", 0)
_line("age_main", _fit_agem, "Myr", 0)
_line("age_bq", _fit_agebq, "Myr", 0)
_line("r_sfr", _fit_rsfr, "", 3)

_cur = DEFAULT_MODULE_PARAMS["sfhdelayedbq"]
print("\n suggested sfhdelayedbq grid (p5..p95 of the fits) vs current nodes:")
for par, vals in [("tau_main", _fit_tau), ("age_main", _fit_agem),
                  ("age_bq", _fit_agebq), ("r_sfr", _fit_rsfr)]:
    a, b, c = _pc(vals)
    print(f"   {par:<9s} data p5/50/95 = {a:8.1f} /{b:8.1f} /{c:8.1f}    "
          f"current nodes = {_cur.get(par)}")

# ── figure 1: example galaxy SFHs with the delayed+bq fit ──
_good = [g for g in SFH if g["bq"]]
_good = sorted(_good, key=lambda g: g["log_mstar"])
_pick = ([_good[int(x)] for x in np.linspace(0, len(_good) - 1, min(6, len(_good)))]
         if _good else [])
if _pick:
    _nc = min(3, len(_pick)); _nr = int(np.ceil(len(_pick) / _nc))
    fig1, ax1 = plt.subplots(_nr, _nc, figsize=(4.6 * _nc, 3.4 * _nr),
                             squeeze=False)
    for k, g in enumerate(_pick):
        ax = ax1[k // _nc][k % _nc]
        ax.plot(g["t_raw"], g["sfr_raw"], "o", ms=3.5, color="0.65",
                label="SIMBA snapshots")
        ax.plot(g["t"], g["sfr"], "-", color="0.25", lw=1.4,
                label="smoothed+resampled")
        tt = np.linspace(g["t"][0], g["t"][-1], 200)
        b = g["bq"]
        ax.plot(tt, sfr_delayed_bq(tt, b["A"], b["tau_main_myr"] / 1e3,
                b["age_main_myr"] / 1e3, b["age_bq_myr"] / 1e3, b["r_sfr"],
                b["t_obs"]), "-", color="#D55E00", lw=1.8,
                label="delayed+bq")
        ax.axvline(g["t_obs"] - b["age_bq_myr"] / 1e3, color="0.6", ls=":", lw=1)
        ax.set_title(f"id {g['gal_id']} z={g['z']:.2f}  "
                     r"$\tau$=%.0f age$_{bq}$=%.0f r=%.2f  $R^2$=%.2f"
                     % (b["tau_main_myr"], b["age_bq_myr"], b["r_sfr"], b["r2"]),
                     fontsize=8)
        ax.set_xlabel("cosmic time [Gyr]"); ax.set_ylabel(r"SFR [$M_\odot$/yr]")
        if k == 0:
            ax.legend(fontsize=7, frameon=False)
    for k in range(len(_pick), _nr * _nc):
        ax1[k // _nc][k % _nc].set_axis_off()
    fig1.suptitle("example true SFHs with the delayed+bq fit", y=1.0)
    fig1.tight_layout()
    fig1.savefig(os.path.join(PLOTDIR, "sfh_prior_examples.png"),
                 dpi=150, bbox_inches="tight")

# ── figure 2: median stacked SFH + fitted-parameter distributions vs grid ──
_L = np.linspace(0, 8, 60)          # lookback from observation [Gyr]
_stack = []
for g in SFH:
    look = g["t_obs"] - g["t"]; pk = np.nanmax(g["sfr"])
    if pk <= 0:
        continue
    o = np.argsort(look)
    _stack.append(np.interp(_L, look[o], (g["sfr"] / pk)[o], left=np.nan,
                            right=np.nan))
_stack = np.array(_stack)
fig2, ax2 = plt.subplots(2, 3, figsize=(14.5, 8)); ax2 = ax2.ravel()
if _stack.size:
    med = np.nanmedian(_stack, 0)
    q16, q84 = np.nanpercentile(_stack, [16, 84], 0)
    ax2[0].fill_between(_L, q16, q84, color="#0072B2", alpha=0.25,
                        label="16-84%")
    ax2[0].plot(_L, med, "-", color="#0072B2", lw=2, label="median")
    _tobs_med = float(np.median([g["t_obs"] for g in SFH]))
    _fb = fit_delayed_bq(_tobs_med - _L[np.isfinite(med)][::-1],
                         med[np.isfinite(med)][::-1], _tobs_med)
    if _fb:
        _tt = _tobs_med - _L
        ax2[0].plot(_L, sfr_delayed_bq(_tt, _fb["A"], _fb["tau_main_myr"] / 1e3,
                    _fb["age_main_myr"] / 1e3, _fb["age_bq_myr"] / 1e3,
                    _fb["r_sfr"], _tobs_med), "-", color="#D55E00", lw=1.8,
                    label="delayed+bq fit")
    ax2[0].set_xlabel("lookback from observation [Gyr]")
    ax2[0].set_ylabel("SFR / peak")
    ax2[0].set_title("median stacked SFH"); ax2[0].legend(fontsize=7,
                                                          frameon=False)

# histograms: delayed+bq params with the current grid nodes overlaid
for ax, par, vals, unit, nodes in [
        (ax2[1], "tau_main", _fit_tau, "Myr", _cur.get("tau_main")),
        (ax2[2], "age_main", _fit_agem, "Myr", _cur.get("age_main")),
        (ax2[3], "age_bq", _fit_agebq, "Myr", _cur.get("age_bq")),
        (ax2[4], "r_sfr", _fit_rsfr, "", _cur.get("r_sfr"))]:
    v = np.asarray(vals, float); v = v[np.isfinite(v)]
    if v.size:
        ax.hist(v, bins=20, color="0.7", edgecolor="0.4")
    for node in (nodes or []):
        ax.axvline(node, color="#D55E00", ls="--", lw=1.2)
    ax.set_xlabel(f"{par} [{unit}]" if unit else par)
    ax.set_ylabel("galaxies")
    ax.set_title(f"{par}: fits vs grid nodes", fontsize=9)
ax2[5].set_axis_off()
fig2.suptitle("delayed+bq parameter distributions (bars) vs current grid "
              "nodes (dashed)", y=1.0)
fig2.tight_layout()
fig2.savefig(os.path.join(PLOTDIR, "sfh_prior_distributions.png"),
             dpi=150, bbox_inches="tight")
print(f"\n[SFH priors] figures -> {PLOTDIR}/sfh_prior_examples.png, "
      "sfh_prior_distributions.png")
plt.show()

# Part 7d2 — aperture-matched smoothed SFHs for the `sfhfromfile` injection

Part 7d fits SIMBA's SFHs with the `sfhdelayedbq` form; Part 7e used to hand CIGALE a
Cartesian grid over those parameters. That grid is the single largest source of freedom in
the fit, and it is freedom we do not need: we **know** each galaxy's star-formation history.
Following `analogues_specphot_almac11.ipynb` Part 9b, Part 7e now **injects** the SFH
directly (`sfhfromfile`), leaving $A_V$ as the free measurand instead of one parameter among
thousands.

This cell builds the archive those runs read. For every selected galaxy it takes the
**archaeological** SFH — mass formed per 100 Myr from the star-particle formation times in
the Stage-0 cutout, i.e. the exact particles the RT saw — smooths it
(`smooth_resample_sfh`, 25 Myr grid, 150 Myr Gaussian kernel) and stores it as
`cigale/sfh_smoothed_aperture.h5`, keyed `snapNNN_galID/<sightline>/<aperture>`.

**Why per aperture and per sightline.** The photometry Part 7b extracts is aperture- and
sightline-resolved, so the stellar population inside `ap3kpc` along `i0p0` is *not* the one
the global history track describes — inner apertures are older and more quenched. Injecting
the global SFH into an aperture fit would import exactly the age–dust degeneracy this
exercise is trying to remove. The binning here reuses the Part 7f geometry
(`read_cutout` → `projected_radius` → cumulative rungs), so archive and
`aperture_truth.fits` describe the same particles.

**Caveats.** The archaeological SFH weights by *current* particle masses, so old bins sit low
by the return fraction (≲30%) — a shape tilt only, and `normalise=True` in `sfhfromfile`
means no mass or normalization is ever taken from these curves ($M_*$ stays the fitted
quantity, which is what makes the Part 7g recovery check honest). Apertures holding fewer
than `SFH_NSTAR_MIN` star particles are left out of the archive; Part 7e drops those members
from the SFH family but still fits their photometry against the other members' shapes.
Times are on the notebook's `COSMO` (Planck15); Part 7e re-caps the age against pcigale's own
Planck18 ceiling.


In [ ]:
# ── Part 7d2 — aperture-matched smoothed SFH archive (sfhfromfile injection) ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (cluster). Writes one
# smoothed archaeological SFH per (galaxy, sightline, cumulative aperture) —
# the SAME projected geometry Part 7b extracted the photometry in, and the same
# particles Part 7f measures its truth from. Part 7e injects these into CIGALE
# via sfhfromfile (shape only, normalise=True), which removes the sfhdelayedbq
# grid entirely and leaves Av_ISM as the free measurand.
from simbanator.analysis.sfh_utils import smooth_resample_sfh

SFH_APERTURE_H5       = os.path.join(CIGALE_DIR, "sfh_smoothed_aperture.h5")
OVERWRITE_SFH_ARCHIVE = False
SFH_ARCH_BIN_MYR      = 100.0    # archaeological bin width before smoothing
SFH_ARCH_DT_MYR       = 25.0     # smoothed output grid step
SFH_ARCH_KERNEL_MYR   = 150.0    # Gaussian kernel sigma
SFH_NSTAR_MIN         = 20       # fewer star particles -> shot-noise, not an SFH

os.makedirs(CIGALE_DIR, exist_ok=True)


def load_sfh_archive(path=None):
    """{id: {incl: {aperture: (t_gyr, sfr, t_obs_gyr)}}} from the archive."""
    path = path or SFH_APERTURE_H5
    out = {}
    with h5py.File(path, "r") as f:
        for sid in f:
            for il in f[sid]:
                for ap in f[sid][il]:
                    d = f[sid][il][ap]
                    arr = np.asarray(d[:], float)
                    out.setdefault(str(sid), {}).setdefault(str(il), {})[str(ap)] = (
                        arr[:, 0], arr[:, 1], float(d.attrs["t_obs_gyr"]))
    return out


if os.path.exists(SFH_APERTURE_H5) and not OVERWRITE_SFH_ARCHIVE:
    _arch = load_sfh_archive()
    _n = sum(len(a) for g in _arch.values() for a in g.values())
    print(f"cached: {_n} aperture SFHs for {len(_arch)} galaxies -> "
          f"{SFH_APERTURE_H5}  (OVERWRITE_SFH_ARCHIVE=True rebuilds)")
else:
    SEL, SNAPS, IDS = load_selection()
    _cen = rt_centers(SNAPS, IDS)
    _ag = np.linspace(0.02, 1.0, 4096)               # a -> cosmic time grid
    _tg = COSMO.age(1.0 / _ag - 1.0).value           # Gyr
    _apr = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]

    _nw = _ngal_ok = _nskip = 0
    with h5py.File(SFH_APERTURE_H5, "w") as f5:
        f5.attrs["bin_myr"] = SFH_ARCH_BIN_MYR
        f5.attrs["dt_myr"] = SFH_ARCH_DT_MYR
        f5.attrs["kernel_myr"] = SFH_ARCH_KERNEL_MYR
        f5.attrs["nstar_min"] = SFH_NSTAR_MIN
        f5.attrs["cosmology"] = COSMO.name
        for _snap in np.unique(SNAPS):
            _z = float(sim.get_z_from_snap(int(_snap)))
            _t_obs = float(COSMO.age(_z).value)
            _edges = np.arange(0.0, _t_obs + SFH_ARCH_BIN_MYR / 1e3,
                               SFH_ARCH_BIN_MYR / 1e3)
            _tc = 0.5 * (_edges[:-1] + _edges[1:])
            _ns = 0
            for _gid in np.unique(IDS[SNAPS == _snap]):
                _cut = read_cutout(_snap, _gid,
                                   _cen.get((int(_snap), int(_gid))), "PartType4",
                                   fields=("Masses", "StellarFormationTime"))
                if _cut is None or _cut.get("StellarFormationTime") is None:
                    print(f"  [skip] snap {_snap} gal {_gid}: no cutout/centre/stars")
                    _nskip += 1
                    continue
                _a = np.asarray(_cut["StellarFormationTime"], float)
                _ok = np.isfinite(_a) & (_a > 0) & (_a <= 1)
                if int(_ok.sum()) < SFH_NSTAR_MIN:
                    _nskip += 1
                    continue
                _tf = np.interp(np.clip(_a[_ok], _ag[0], 1.0), _ag, _tg)   # Gyr
                _mm = np.asarray(_cut["Masses"], float)[_ok] * 1e10 / _cut["h"]
                _pos = _cut["pos"][_ok]
                _sid = f"snap{int(_snap):03d}_gal{int(_gid)}"
                _ngal_ok += 1
                _ns += 1
                for _il, _nv in zip(INCL_LABELS, NHAT):
                    _rp = projected_radius(_pos, _nv)
                    for _lab, _r in zip(APERTURE_LABELS, _apr):
                        _msk = _rp <= _r
                        _nap = int(_msk.sum())
                        if _nap < SFH_NSTAR_MIN:
                            continue
                        _h, _ = np.histogram(_tf[_msk], bins=_edges,
                                             weights=_mm[_msk])
                        _ts, _ss = smooth_resample_sfh(
                            _tc, _h / (SFH_ARCH_BIN_MYR * 1e6),
                            dt_myr=SFH_ARCH_DT_MYR, kernel_myr=SFH_ARCH_KERNEL_MYR)
                        _d = f5.create_dataset(f"{_sid}/{_il}/{_lab}",
                                               data=np.column_stack([_ts, _ss]))
                        _d.attrs["t_obs_gyr"] = _t_obs
                        _d.attrs["nstar"] = _nap
                        _nw += 1
            print(f"snap {_snap:3d} (z={_z:.2f}, t_obs={_t_obs:.2f} Gyr): "
                  f"{_ns} galaxies archived")
    print(f"\n[SFH archive] {_nw} aperture SFHs for {_ngal_ok} galaxies "
          f"({_nskip} skipped) -> {SFH_APERTURE_H5}")
    _arch = load_sfh_archive()

# ── coverage report: how many members each Part 7e run will have an SFH for ──
_cov = {}
for _sid, _g in _arch.items():
    for _il, _aps in _g.items():
        for _lab in _aps:
            _cov[(_il, _lab)] = _cov.get((_il, _lab), 0) + 1
print("\naperture SFH coverage (galaxies with an archived SFH):")
print("   " + "".join(f"{l:>10s}" for l in APERTURE_LABELS))
for _il in INCL_LABELS:
    print(f"{_il:>7s}" + "".join(f"{_cov.get((_il, l), 0):>10d}"
                                 for l in APERTURE_LABELS))

# ── figure: the radial SFH gradient the injection preserves ──
_pick = [s for s in sorted(_arch) if len(_arch[s].get(INCL_LABELS[0], {}))
         == len(APERTURE_LABELS)][:3]
if _pick:
    fig, axs = plt.subplots(1, len(_pick), figsize=(4.8 * len(_pick), 3.6),
                            squeeze=False)
    _cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(APERTURE_LABELS)))
    for _k, _sid in enumerate(_pick):
        ax = axs[0][_k]
        for _lab, _c in zip(APERTURE_LABELS, _cmap):
            _e = _arch[_sid][INCL_LABELS[0]].get(_lab)
            if _e is None:
                continue
            ax.plot(_e[0], _e[1], "-", lw=1.5, color=_c, label=_lab)
        ax.set(xlabel="cosmic time [Gyr]", ylabel=r"SFR [$M_\odot$/yr]",
               title=f"{_sid}  ({INCL_LABELS[0]})")
        ax.grid(alpha=0.3)
        if _k == 0:
            ax.legend(fontsize=7, frameon=False, title="aperture",
                      title_fontsize=7)
    fig.suptitle("aperture-matched smoothed SFHs injected into CIGALE "
                 "(inner apertures quench earlier)", y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "sfh_aperture_archive.png"), dpi=150,
                bbox_inches="tight")
    plt.show()


# Part 7e — prior-constrained CIGALE runs as ONE SLURM job array (cluster)

CIGALE never fits inside the notebook: this cell **prepares** every run directory
(preflight error repair + `simbanator.sed.cigale.prepare_run`, which writes a complete,
validated `pcigale.ini` + `.spec` — no `pcigale init`/`genconf` by hand) and writes **one
SLURM job array with one task per run** (`cigale.write_slurm_array`); you then `sbatch` it
from a login node.

## 2026-08-07 redesign — making $A_V$ a measurand

The earlier grids fitted a free `sfhdelayedbq` SFH (6500 parameter combinations) *and* a
free attenuation *and* a partly-free dust emission simultaneously. In that setup
`bayes.attenuation.Av_ISM` is not a measurement of the mock's dust — it is one degenerate
parameter among thousands, free to trade against stellar age. This part now follows
`analogues_specphot_almac11.ipynb` **Part 9b**: everything we know from the simulation is
pinned, and the attenuation is the only thing left free.

| axis | how it is constrained now | why |
|---|---|---|
| **SFH** | **injected** via `sfhfromfile` — each run's `sfh.fits` carries its members' own aperture-matched smoothed SFHs from Part 7d2, shape-only (`normalise=True`) | removes the parametric-form mismatch entirely; $M_*$ stays the *fitted* normalization, so Part 7g's recovery check is still honest |
| **metallicity** | `split_by_metallicity` pins one bc03 node per run from the Part 4c aperture $Z_\star$; gas $Z$ restricts `zgas` to ≤ 3 nodes | age–metallicity–dust all redden the optical; two of the three are now known |
| **redshift / age** | fixed at the snapshot $z$; `sfh.age` capped at **pcigale's own Planck18 ceiling** at $z$ rounded to `redshift_decimals = 2`, with the `sfh.fits` time axis shifted so the *recent* end survives | CIGALE 2025.1 silently masks every model older than that ceiling ("No suitable model found") — it cost 20/72 objects in the almac11 runs before it was found |
| **dust emission** | **freed**: `umin` 11 nodes, `gamma` 4, `qpah` 2 | with the SFH fixed, any FIR mismatch a pinned grid cannot absorb leaks straight into $A_V$ through energy balance. Dust emission never touches the UV/optical, so the freedom reopens no age–dust degeneracy — it trades a little IR leverage (variance) against the pin's bias |
| **AGN** | `skirtor2016` in the chain for **all three arms**, `fracAGN` free with **0.0 in the grid**, `i = 30°` (type 1) | identical estimator across arms. `dust_on`/`dust_off` were rendered without AGN sources, so their `fracAGN` must return ~0 — that null test is **Part 7e2**. `i` is pinned face-on because the m25 `agn_on` RT injects the BH as a point source obscured only by the galaxy's own dust grid; a torus-obscured geometry would double-count the obscuration |
| **attenuation** | **the measurand, free**: `Av_ISM` on 18 nodes over 0–3 mag, `slope_ISM` ∈ {−0.7, −0.48, −0.3}, `mu` ∈ {0.2, 0.44, 0.7}, `slope_BC` ∈ {−1.3, −0.7} | `Av_ISM` is normalized at V, so it keeps its V-band meaning even with a freed curve shape. Low `mu` lets the fit power the FIR from birth clouds without reddening the diffuse optical — the failure mode that pinned `Av_ISM` at 0 in the almac11 $\chi^2$ tail |

**Scope.** Three arms × **5 cumulative apertures** × 4 sightlines × 4 snapshots × Z groups.
The annuli of Part 7c are deliberately **not** fitted: an annulus does not contain the dust
heated by the light it emits, so CIGALE's energy balance is ill-posed there — and energy
balance is exactly the pathway that corrupts $A_V$. Run dirs carry a `_sfhinj` suffix, so
the previous `sfhdelayedbq` runs stay on disk for the generation-over-generation comparison.

**`dust_off` is lighter by design.** `_DROP_FIR` removes MIPS/PACS/SPIRE/SCUBA-2/ALMA (pure
stellar Rayleigh–Jeans there), which leaves `dl2014` unconstrained — so that axis alone is
collapsed to one combo. The **attenuation grid stays identical to `dust_on`**, otherwise the
"does the estimator return $A_V \approx 0$ when there is no dust?" null test would be rigged.

## Memory — why the 2026-08-07 array died

Job `12828014` was SIGKILLed on the `INTEL_PHI` nodes, always on the largest grids. The
cause is in `/etc/slurm/slurm.conf`: this cluster schedules with
`SelectTypeParameters=CR_Core` — **memory is not a consumable resource**, so `--mem-per-cpu`
reserves nothing and only sets a cgroup ceiling. With `ExclusiveUser=YES` and no array
throttle, ~32 eight-core tasks packed onto one 256-core / 190 GB phi node and exhausted node
RAM; the kernel OOM killer took the biggest processes.

The fix is per-task footprint, not partition avoidance (PHI is usually the only partition
actually available, so it is kept):

- `MAX_BLOCK_MODELS` → `prepare_run` sets pcigale's `blocks` so no block's shared model
  arrays exceed ~0.7 GB (they live in `/dev/shm` and count against the job cgroup;
  overrunning *that* dies with SIGBUS rather than SIGKILL).
- `ARRAY_THROTTLE` caps simultaneous tasks, so the worst case on any single node is bounded.
- Injecting the SFH helps here too: the old `sfhdelayedbq` grid also made each pcigale
  worker cache 6500 SFH module instances; `sfhfromfile` caches one per family member.

**One-time env setup.** CIGALE 2025 lives in its **own** conda env — do NOT install it into
`pd39` (powderday pins numpy/astropy):

```bash
conda create -n cigale python=3.12 -y
conda activate cigale
pip install <path to the cigale-v2025 tarball from cigale.lam.fr>   # or `pip install .`
```

The kernel stays `pd39`: only the env's `pcigale` executable is needed, and
`cigale.find_pcigale()` locates it by absolute path (no `conda activate` at runtime, so the
same path works inside the SLURM tasks). `cg.describe_run()` prints the full genconf-style
docs; `cg.describe_run(run_dir, docs=False)` echoes one prepared run's grids.

**Other knobs.** `BROADBANDS_ONLY` drops `F###N`/`F###M`, HSC/VIRCAM narrowbands and the
duplicate `spire *_ext` curves from the inputs (93 → ~40 bands); `FIT_BANDS` then pins which
of those are actually **fitted** — everything else is still **predicted** (`bayes.<band>`).
`SKIP_IF_DONE = False` (default) re-fits everything, with CIGALE renaming any pre-existing
`out/` to a timestamped `<YYYYMMDDHHMM>_out/` backup; set it `True` and re-run this cell
before resubmitting after a partial failure.

⚠️ **Downstream:** `bayes.sfh.tau_main` / `age_main` / `age_bq` / `r_sfr` no longer exist —
there is no parametric SFH to report; `sfh.index` (which member's injected SFH won) replaces
them. Part 7g degrades gracefully — `compare_results` only compares truth columns that have a
matching `bayes.*`, so those four panels simply disappear while $M_*$, mass-weighted age, SFR,
$A_V^{\rm ISM}$, dust mass and metallicity are still checked against the SIMBA truth.

Workflow: **Part 7d2** → this cell → `sbatch output/cis25/cigale_runs/submit_cigale_array.job`
→ **Part 7e2** (AGN null test) → Parts 7g–7k.


In [ ]:
# ── Part 7e: prior-constrained CIGALE runs + ONE SLURM job array (cluster) ──
# CIGALE never runs in this kernel: the cell only writes run dirs + the job file.
# 2026-08-07 redesign, porting analogues_specphot_almac11.ipynb Part 9b so that
# bayes.attenuation.Av_ISM is a MEASURAND and not one free parameter among
# thousands. What changed vs the sfhdelayedbq grids:
#   (1) SFH INJECTED — each run's sfh.fits carries its members' own smoothed
#       aperture-matched archaeological SFHs (Part 7d2), 1 Myr grid, shape-only
#       (normalise=True) so M* stays the fitted normalization and the Part 7g
#       recovery check stays honest. The 6500-combo sfhdelayedbq grid is gone.
#   (2) METALLICITY PINNED FOR EVERY ARM — split_by_metallicity snaps the Part
#       4c aperture Z_star to one bc03 node per run and restricts zgas to <=3
#       nodes. The old tag regex did not match 'agn_on_*', so those runs
#       silently kept the default 3-metallicity grid (3x larger, and not
#       comparable to their dust_on twins). Fixed here.
#   (3) REDSHIFT/AGE CAPPED AT PCIGALE'S OWN CEILING — CIGALE 2025.1 rejects
#       every model with sfh.age above the Planck18 universe age at the observed
#       z rounded to redshift_decimals=2, silently masking objects ("No suitable
#       model found"). The sfh.fits time axis is shifted so the RECENT end
#       survives the cap (sfhfromfile truncates at the recent end, so shrinking
#       age alone would cut the newest SFR).
#   (4) DUST EMISSION FREED — umin on an 11-node ladder, gamma 4 nodes, qpah 2.
#       With the SFH fixed, any FIR mismatch a pinned grid cannot absorb leaks
#       straight into Av_ISM through energy balance; dust emission never touches
#       the UV/optical, so the freedom reopens no age-dust degeneracy.
#   (5) ATTENUATION = THE MEASURAND, LEFT FREE — denser Av_ISM grid, slope_ISM /
#       mu / slope_BC freed. Av_ISM is normalized at V, so it keeps its V-band
#       meaning even with a freed curve shape.
#   (6) skirtor2016 IN THE CHAIN FOR ALL THREE ARMS with fracAGN free INCLUDING
#       0.0 — identical estimator across arms, and 'no AGN' stays a reachable
#       solution. dust_on/dust_off were built without AGN sources, so their
#       recovered fracAGN must come back ~0: that is the null test Part 7e2 runs.
#       i is pinned to a type-1 view (30 deg < 90-oa): the m25 agn_on RT injects
#       the BH as a point source obscured only by the galaxy's own dust grid, so
#       a torus-obscured geometry would double-count the obscuration.
#   (7) CUMULATIVE APERTURES ONLY — annuli are dropped. An annulus does not
#       contain the dust heated by the light it emits, so CIGALE's energy
#       balance is ill-posed there and that is exactly the pathway that
#       corrupts Av. All 4 sightlines are kept.
#   (8) MEMORY — the 2026-08-07 array died with SIGKILL on INTEL_PHI: this
#       cluster schedules on cores only (SelectTypeParameters=CR_Core), so
#       --mem-per-cpu reserves nothing and 32 x 8-core tasks packed onto one
#       256-core / 190 GB phi node exhausted node RAM. Fixed by bounding each
#       task's shared model arrays (MAX_BLOCK_MODELS -> pcigale 'blocks') and
#       capping simultaneous tasks (ARRAY_THROTTLE), NOT by dropping partitions.
from astropy.cosmology import Planck18
from simbanator.sed import cigale as cg

PCIGALE_CMD    = cg.find_pcigale()   # dedicated conda env; see the markdown above
CORES_PER_TASK = 8
PLOT_SEDS      = True
SKIP_IF_DONE   = False               # True -> resubmits only run what is missing
USE_Z_PRIORS   = True
RUN_SUFFIX     = "_sfhinj"           # keeps the old sfhdelayedbq runs on disk
ARMS           = ("dust_on", "dust_off", "agn_on")
BROADBANDS_ONLY = True
# --- memory/scheduling (see note 8) ---
MAX_BLOCK_MODELS = 1_000_000   # ~0.7 GB of shared model arrays per block
ARRAY_THROTTLE   = 24          # worst case 24 x ~2 GB on one node
PARTITIONS       = "INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL"
WALLTIME         = "1-00:00"   # PHI cores are ~3-4x slower than Cascade

_DROP_BAND = re.compile(r"\.F\d+[NM]$|\.NB\d+$|_ext$|^generic\.|^2mass\.")
# dust_off inputs additionally drop the FIR/sub-mm bands (pure stellar
# Rayleigh-Jeans there): with no FIR data the dl2014 axis is unconstrained, so
# it is collapsed to a single combo for that arm — the ATTENUATION grid stays
# identical to dust_on, or the Av null test would be rigged.
_DROP_FIR = re.compile(r"^spitzer\.mips\.|^herschel\.|^jcmt\.|^alma\.")
# cumulative apertures only (note 7); agn_on now matches too (note 2)
_TAG_RE = re.compile(r"^(dust_on|dust_off|agn_on)_(ap[0-9]+kpc)_(i\d+p\d+)$")

# manual fitted-band list (unselected bands are still PREDICTED as bayes.<band>)
FIT_BANDS = [
    "subaru.hsc.g", "subaru.hsc.r", "subaru.hsc.i", "subaru.hsc.z",
    "subaru.hsc.Y",
    "paranal.vircam.Y", "paranal.vircam.J", "paranal.vircam.H",
    "paranal.vircam.Ks",
    "hst.wfc3.uvis1.F606W", "hst.wfc3.uvis1.F814W",
    "jwst.nircam.F070W", "jwst.nircam.F090W", "jwst.nircam.F115W",
    "jwst.nircam.F150W", "jwst.nircam.F150W2", "jwst.nircam.F200W",
    "jwst.nircam.F277W", "jwst.nircam.F322W2", "jwst.nircam.F356W",
    "jwst.nircam.F444W",
    "jwst.miri.F560W", "jwst.miri.F770W", "jwst.miri.F1000W",
    "jwst.miri.F1130W", "jwst.miri.F1280W", "jwst.miri.F1500W",
    "jwst.miri.F1800W", "jwst.miri.F2100W", "jwst.miri.F2550W",
    "spitzer.mips.24mu", "spitzer.mips.70mu", "spitzer.mips.160mu",
    "herschel.pacs.blue", "herschel.pacs.green", "herschel.pacs.red",
    "herschel.spire.PSW", "herschel.spire.PMW",
    "jcmt.scuba2.450GHz", "jcmt.scuba2.850GHz",
    "alma.band6",
]

SED_MODULES = ("sfhfromfile", "bc03", "nebular", "dustatt_modified_CF00",
               "dl2014", "skirtor2016", "restframe_parameters", "redshifting")

# ── the measurand: attenuation, free ──
AV_ISM_GRID    = [0.0, 0.02, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.55,
                  0.7, 0.9, 1.1, 1.4, 1.7, 2.1, 2.6, 3.0]
SLOPE_ISM_GRID = [-0.7, -0.48, -0.3]      # CF00 default + greyer ISM curves
MU_GRID        = [0.2, 0.44, 0.7]         # low mu = FIR powered by birth clouds
SLOPE_BC_GRID  = [-1.3, -0.7]             # CF00 default + a greyer BC curve
# ── nuisance: dust emission, freed so the IR bands stop torquing Av ──
QPAH_GRID  = [2.50, 5.95]                 # powderday PAH_frac['usg'] + 1 lower
UMIN_GRID  = sorted({float(v) for v in cg.nearest_option(
    [0.1, 0.2, 0.35, 0.6, 1.0, 1.7, 3.0, 5.0, 8.0, 12.0, 25.0],
    cg.grid_options("dl2014", "umin"), log=True)})
GAMMA_GRID = [0.01, 0.02, 0.05, 0.1]
# ── nuisance: AGN. fracAGN MUST contain 0.0 (Part 7e2 null test) ──
FRAC_AGN_GRID = [0.0, 0.1, 0.2, 0.4]
SKIRTOR_I     = 30                        # type-1 view (i < 90 - oa = 50 deg)

# rest-frame line EWs (CIGALE 2025.1 label/blue/line/red format)
EW_SPEC = ("OIII5007/497.7/499.7/499.7/501.7/501.7/503.7 & "
           "Halpha/653.3/655.3/655.3/657.3/657.3/659.3 & "
           "HdeltaA/404.160/407.975/408.350/412.225/412.850/416.100")
# sfh.tau_main/age_main/age_bq/r_sfr are GONE (no parametric SFH any more):
# sfh.index records which member's injected SFH won instead. Part 7g drops
# those four panels by itself (compare_results keeps only truth columns
# that have a matching bayes.* in results.fits) — nothing to rewire there.
VARIABLES = ["stellar.m_star", "stellar.metallicity", "stellar.age_m_star",
             "sfh.sfr", "sfh.sfr10Myrs", "sfh.sfr100Myrs", "sfh.index",
             "attenuation.Av_ISM", "attenuation.Av_BC",
             "attenuation.generic.bessell.V", "attenuation.generic.bessell.B",
             "dust.luminosity", "dust.mass", "dust.umean",
             "agn.fracAGN",
             "param.Dn4000", "param.EW(OIII5007)", "param.EW(Halpha)",
             "param.EW(HdeltaA)",
             "param.restframe_Lnu(galex.FUV)",
             "param.restframe_Lnu(generic.bessell.V)",
             "param.restframe_generic.johnson.U-generic.johnson.V",
             "param.restframe_generic.johnson.V-generic.johnson.J",
             "param.restframe_galex.FUV-galex.NUV",
             "param.restframe_galex.NUV-sloan.sdss.r"]
ANALYSIS_PARAMS = {"variables": VARIABLES, "save_best_sed": True}

print("pcigale:", PCIGALE_CMD)
print(f"[grid] Av_ISM {len(AV_ISM_GRID)} x slope_ISM {len(SLOPE_ISM_GRID)} x "
      f"mu {len(MU_GRID)} x slope_BC {len(SLOPE_BC_GRID)} = "
      f"{len(AV_ISM_GRID)*len(SLOPE_ISM_GRID)*len(MU_GRID)*len(SLOPE_BC_GRID)}"
      f" attenuation combos")
print(f"[grid] dl2014 qpah {QPAH_GRID} | umin {UMIN_GRID} | gamma {GAMMA_GRID}")
print(f"[grid] skirtor2016 fracAGN {FRAC_AGN_GRID} @ i={SKIRTOR_I} deg (type 1)")

# ── the injected SFHs (Part 7d2) — kernel-restart safe ──
SFH_APERTURE_H5 = globals().get(
    "SFH_APERTURE_H5", os.path.join(CIGALE_DIR, "sfh_smoothed_aperture.h5"))
if "load_sfh_archive" not in globals():
    def load_sfh_archive(path=None):
        """{id: {incl: {aperture: (t_gyr, sfr, t_obs_gyr)}}} — see Part 7d2."""
        out = {}
        with h5py.File(path or SFH_APERTURE_H5, "r") as f:
            for sid in f:
                for il in f[sid]:
                    for ap in f[sid][il]:
                        d = f[sid][il][ap]
                        arr = np.asarray(d[:], float)
                        out.setdefault(str(sid), {}).setdefault(
                            str(il), {})[str(ap)] = (arr[:, 0], arr[:, 1],
                                                     float(d.attrs["t_obs_gyr"]))
        return out
if not os.path.exists(SFH_APERTURE_H5):
    raise RuntimeError(f"{SFH_APERTURE_H5} missing — run Part 7d2 first "
                       "(7e injects the aperture-matched smoothed SFHs)")
SFH_ARCH = load_sfh_archive()
print(f"[sfh] archive: {len(SFH_ARCH)} galaxies x up to "
      f"{len(INCL_LABELS)*len(APERTURE_LABELS)} (sightline, aperture) SFHs")


def _sfh_file(run_dir, ids, incl, aperture, age_myr, shift_myr=0):
    """Write run_dir/sfh.fits for sfhfromfile; return the ids actually written.

    col 0 = time [Myr], 0..age_myr in STRICT 1 Myr steps (CIGALE raises
    otherwise); one SFR column per member that has an archived SFH for THIS
    (sightline, aperture). Interpolated from the 25 Myr archive grid; SFR = 0
    before the first archived sample, edge-held after the last (the recent edge
    drives the UV/nebular fluxes). shift_myr > 0 maps table time t to cosmic
    time t + shift_myr, i.e. drops the OLDEST shift_myr so that t = age_myr
    still lands on the snapshot epoch.
    """
    t1 = np.arange(int(age_myr) + 1)
    tab = Table()
    tab["time"] = t1.astype(np.int64)
    kept = []
    for gid in ids:
        entry = SFH_ARCH.get(str(gid), {}).get(incl, {}).get(aperture)
        if entry is None:
            continue
        t_gyr, sfr = entry[0], entry[1]
        tab[str(gid)] = np.clip(np.interp(t1 + shift_myr, t_gyr * 1e3, sfr,
                                          left=0.0), 0.0, None)
        kept.append(str(gid))
    if not kept:
        return None
    os.makedirs(run_dir, exist_ok=True)
    tab.write(os.path.join(run_dir, "sfh.fits"), overwrite=True)
    return kept


# ── per-galaxy metallicity priors (Part 4c), keyed by (aperture, sightline) ──
_ztabf = os.path.join(TABLEDIR, "aperture_metallicities.fits")
_ztab = Table.read(_ztabf) if (USE_Z_PRIORS and os.path.exists(_ztabf)) else None
if USE_Z_PRIORS and _ztab is None:
    print(f"[Z priors] {_ztabf} missing — run Part 4c first; fitting WITHOUT "
          "Z priors")


def _z_maps(label, ilab):
    _m = np.char.strip(np.asarray(_ztab["incl"], str)) == ilab
    _ids = [f"snap{int(s):03d}_gal{int(g)}" for s, g in
            zip(np.asarray(_ztab["snap"], int)[_m],
                np.asarray(_ztab["gal_id"], int)[_m])]
    return (dict(zip(_ids, np.asarray(_ztab[f"Zstar_{label}"], float)[_m])),
            dict(zip(_ids, np.asarray(_ztab[f"Zgas_{label}"], float)[_m])))


PERSNAP_DIR = os.path.join(CIGALE_DIR, "persnap")
os.makedirs(PERSNAP_DIR, exist_ok=True)

_run_dirs, _skipped, _nmodels = [], [], []
for data_file in sorted(glob.glob(os.path.join(CIGALE_DIR, "cigale_*.fits"))):
    tag = os.path.basename(data_file)[len("cigale_"):-len(".fits")]
    _mt = _TAG_RE.match(tag)
    if _mt is None:
        continue                      # annuli, Part 7b intermediates, per-snap files
    arm, ap_label, incl = _mt.group(1), _mt.group(2), _mt.group(3)
    if arm not in ARMS:
        continue

    cg.sanitize_input_errors(data_file)     # legacy all-negative-error repair
    _t = Table.read(data_file)
    _zs_map, _zg_map = _z_maps(ap_label, incl) if _ztab is not None else (None, None)

    _tsnaps = np.array([int(str(i).split("_")[0][4:]) for i in _t["id"]])
    for _sn in sorted(set(_tsnaps.tolist())):
        _tsn = _t[_tsnaps == _sn]
        if BROADBANDS_ONLY:
            _tsn = _tsn[[c for c in _tsn.colnames
                         if not _DROP_BAND.search(c.removesuffix("_err"))]]
        if arm == "dust_off":
            _tsn = _tsn[[c for c in _tsn.colnames
                         if not _DROP_FIR.search(c.removesuffix("_err"))]]
        # NaN errors -> 10% of the flux (CIGALE adds additionalerror in quadrature)
        _nfix = 0
        for _b in [c for c in _tsn.colnames
                   if c not in ("id", "redshift", "distance")
                   and not c.endswith("_err")]:
            _f, _e = (np.asarray(_tsn[_b], float),
                      np.asarray(_tsn[f"{_b}_err"], float))
            _bad = np.isfinite(_f) & ~np.isfinite(_e)
            if _bad.any():
                _tsn[f"{_b}_err"][_bad] = 0.1 * np.abs(_f[_bad])
                _nfix += int(_bad.sum())
        _sntag = f"{tag}_snap{_sn:03d}"
        _snapf = os.path.join(PERSNAP_DIR, f"cigale_{_sntag}.fits")
        _tsn.write(_snapf, overwrite=True)

        # age: SIMBA's own t(z), capped at pcigale's Planck18 ceiling (note 3)
        _z = float(np.asarray(_tsn["redshift"], float)[0])
        _age_cos = int(round(COSMO.age(_z).to(u.Myr).value))
        _age_cig = int(np.floor(min(Planck18.age(_z).to(u.Myr).value,
                                    Planck18.age(round(_z, 2)).to(u.Myr).value))) - 1
        _age_myr = min(_age_cos, _age_cig)
        _shift = _age_cos - _age_myr

        groups = (cg.split_by_metallicity(_snapf, _zs_map, zgas=_zg_map)
                  if _zs_map is not None
                  else [{"tag": "", "path": _snapf, "module_params": None}])
        for grp in groups:
            _gt = Table.read(grp["path"])
            _ids = [str(i) for i in _gt["id"]]
            run_dir = os.path.join(
                RUN_BASE,
                _sntag + (f"_{grp['tag']}" if grp["tag"] else "") + RUN_SUFFIX)
            kept = _sfh_file(run_dir, _ids, incl, ap_label, _age_myr, _shift)
            if kept is None:
                _skipped.append((os.path.basename(run_dir), len(_ids)))
                continue
            mp = {
                "sfhfromfile": {
                    "filename": os.path.join(run_dir, "sfh.fits"),
                    "sfr_column": list(range(1, len(kept) + 1)),
                    "age": [_age_myr], "normalise": True},
                "bc03": {"imf": 1, "metallicity": [0.008, 0.02, 0.05]},
                "dustatt_modified_CF00": {"Av_ISM": AV_ISM_GRID,
                                          "slope_ISM": SLOPE_ISM_GRID,
                                          "mu": MU_GRID,
                                          "slope_BC": SLOPE_BC_GRID},
                # dust_off has no FIR data -> dl2014 unconstrained, collapse it
                "dl2014": ({"qpah": QPAH_GRID, "umin": UMIN_GRID,
                            "gamma": GAMMA_GRID} if arm != "dust_off" else
                           {"qpah": [2.5], "umin": [5.0], "gamma": [0.02]}),
                "skirtor2016": {"fracAGN": FRAC_AGN_GRID, "i": SKIRTOR_I},
                "restframe_parameters": {"Dn4000": True, "EW": EW_SPEC},
            }
            for _k, _v in (grp["module_params"] or {}).items():
                mp[_k] = {**mp.get(_k, {}), **_v}      # bc03 Z pin + zgas nodes
            cg.prepare_run(run_dir, grp["path"], sed_modules=SED_MODULES,
                           module_params=mp, analysis_params=ANALYSIS_PARAMS,
                           cores=CORES_PER_TASK, fit_bands=FIT_BANDS,
                           max_block_models=MAX_BLOCK_MODELS, verbose=False)
            _run_dirs.append(run_dir)

            def _n(mod, par, default=1):
                v = mp.get(mod, {}).get(par)
                return len(v) if isinstance(v, (list, tuple)) else default
            _nmodels.append(
                len(kept) * _n("bc03", "metallicity") * _n("nebular", "zgas")
                * _n("dustatt_modified_CF00", "Av_ISM")
                * _n("dustatt_modified_CF00", "slope_ISM")
                * _n("dustatt_modified_CF00", "mu")
                * _n("dustatt_modified_CF00", "slope_BC")
                * _n("dl2014", "qpah") * _n("dl2014", "umin")
                * _n("dl2014", "gamma") * _n("skirtor2016", "fracAGN"))
            if len(kept) < len(_ids):
                print(f"  [note] {os.path.basename(run_dir)}: "
                      f"{len(_ids) - len(kept)} member(s) without an archived "
                      f"SFH — family = {len(kept)}")

if not _run_dirs:
    raise RuntimeError("no run dirs prepared — check the Part 7b catalogs and "
                       "the Part 7d2 SFH archive")

_nm = np.asarray(_nmodels, float)
print(f"\n{len(_run_dirs)} run dirs prepared under {RUN_BASE}")
print(f"   models/run: min {_nm.min():,.0f} | median {np.median(_nm):,.0f} | "
      f"max {_nm.max():,.0f}  (blocks = ceil(n/{MAX_BLOCK_MODELS:,}))")
for _arm in ARMS:
    _k = [d for d in _run_dirs if os.path.basename(d).startswith(_arm)]
    print(f"   {_arm:9s}: {len(_k):4d} runs")
if _skipped:
    print(f"   [skipped] {len(_skipped)} run(s) with no archived SFH for any "
          f"member: {[s[0] for s in _skipped[:4]]}"
          f"{' ...' if len(_skipped) > 4 else ''}")

JOB_FILE = cg.write_slurm_array(
    _run_dirs, os.path.join(RUN_BASE, "submit_cigale_array.job"),
    pcigale_cmd=PCIGALE_CMD, partition=PARTITIONS, cores=CORES_PER_TASK,
    time=WALLTIME, array_throttle=ARRAY_THROTTLE, plots=PLOT_SEDS,
    skip_if_done=SKIP_IF_DONE)
print(f"\nsbatch {JOB_FILE}")
print(f"  partitions {PARTITIONS}")
print(f"  <= {ARRAY_THROTTLE} tasks at once x {CORES_PER_TASK} cores "
      f"(memory is NOT a scheduled resource on this cluster — the throttle and "
      f"the {MAX_BLOCK_MODELS:,}-model blocks are what keep a 256-core / 190 GB "
      f"phi node from running out of RAM)")
print("  re-running this cell regenerates the script; flip SKIP_IF_DONE=True "
      "to resubmit only the missing runs")


# Part 7e2 — the AGN null test (run after the array drains)

All three arms are now fitted with the **same** chain, `skirtor2016` included and `fracAGN`
free with `0.0` in the grid. That makes a falsifiable prediction:

- `dust_on` and `dust_off` were rendered **without** AGN sources → their recovered
  `bayes.agn.fracAGN` must be consistent with zero.
- `agn_on` carries SIMBA's black holes as point sources (`parameters_master-agn.py`,
  Hopkins+2007 template, $L_{\rm bol} = 0.1\,\dot M_{\rm BH}c^2$) → it must not.

This matters for the measurand, not just for tidiness. If the AGN-free arms come back with a
finite `fracAGN`, the torus is absorbing light that belongs to the stars or the diffuse dust,
and the `Av_ISM` this redesign exists to measure is contaminated at exactly the wavelengths
that set it. The per-aperture breakdown localizes any leak — a torus can only plausibly steal
light in the central rungs, so `fracAGN > 0` in `ap100kpc` for `dust_on` would point at the
mid-IR calibration rather than at a genuine compact source.

For `agn_on` the cell also correlates the recovered `fracAGN` against the RT truth
(`fagn_V_<aperture>` from Part 7a-agn). This is deliberately **not** a 1:1 identity test:
CIGALE's `fracAGN` is defined over the total dust luminosity (`lambda_fracAGN = 0/0`) while
the RT column is a V-band flux ratio. The correlation is the test — a fit that recovers an
AGN where SIMBA put a bright one, and none where it put a faint one, is tracking the real
thing.

Outputs: `tables/cigale_sfhinj_fits.fits` (every fitted object, all arms) and
`plots/powderday_quenched/cigale_agn_null_test.png`.


In [ ]:
# ── Part 7e2 — AGN null test: does the fit invent an AGN that is not there? ──
# Self-contained after Part 0 (+ Part 7e for the naming constants). Every arm is
# now fitted with the SAME chain, skirtor2016 included and fracAGN free with 0.0
# in the grid. dust_on / dust_off were rendered WITHOUT AGN sources, so their
# recovered fracAGN must come back consistent with zero; agn_on carries SIMBA's
# BHs as point sources, so it must not. If dust_on came back with a finite AGN
# fraction, the torus would be soaking up light that belongs to the stars or the
# dust — and the Av_ISM this whole redesign is meant to measure would be
# contaminated. This cell is the check.
FRACAGN_NULL_TOL = 0.05     # |fracAGN| below this counts as "no AGN"
NSIG_NULL        = 2.0      # ... or consistent with 0 within this many sigma
RUN_SUFFIX = globals().get("RUN_SUFFIX", "_sfhinj")   # kernel-restart safe
_RUNRE = re.compile(r"^(dust_on|dust_off|agn_on)_(ap\d+kpc)_(i\d+p\d+)"
                    r"_snap(\d+)(?:_(Zs\w+))?" + re.escape(RUN_SUFFIX) + r"$")

_rows = []
for _rd in sorted(glob.glob(os.path.join(RUN_BASE, f"*{RUN_SUFFIX}"))):
    _m = _RUNRE.match(os.path.basename(_rd))
    _res = os.path.join(_rd, "out", "results.fits")
    if _m is None or not os.path.exists(_res):
        continue
    _t = Table.read(_res)
    if "bayes.agn.fracAGN" not in _t.colnames:
        print(f"  [skip] {os.path.basename(_rd)}: no bayes.agn.fracAGN "
              "(pre-skirtor run?)")
        continue
    for _r in _t:
        _rows.append(dict(
            arm=_m.group(1), aperture=_m.group(2), incl=_m.group(3),
            snap=int(_m.group(4)), zgrp=_m.group(5) or "",
            id=str(_r["id"]),
            gal_id=int(str(_r["id"]).split("_gal")[1]),
            fracAGN=float(_r["bayes.agn.fracAGN"]),
            fracAGN_err=float(_r["bayes.agn.fracAGN_err"]),
            Av_ISM=float(_r["bayes.attenuation.Av_ISM"]),
            Av_ISM_err=float(_r["bayes.attenuation.Av_ISM_err"]),
            chi2=float(_r["best.reduced_chi_square"])))

if not _rows:
    raise RuntimeError(f"no finished {RUN_SUFFIX} runs with results.fits under "
                       f"{RUN_BASE} — submit the Part 7e array first")
FIT = Table(rows=_rows)
FIT.write(os.path.join(TABLEDIR, "cigale_sfhinj_fits.fits"), overwrite=True)
print(f"{len(FIT)} fitted objects from "
      f"{len(set(zip(FIT['arm'], FIT['aperture'], FIT['incl'], FIT['snap'])))} "
      f"run configurations\n")


def _null_stats(sub):
    f, e = np.asarray(sub["fracAGN"], float), np.asarray(sub["fracAGN_err"], float)
    ok = np.isfinite(f)
    f, e = f[ok], e[ok]
    if not f.size:
        return None
    consistent = (f <= FRACAGN_NULL_TOL) | (f <= NSIG_NULL * np.where(e > 0, e, np.inf))
    return dict(n=f.size, p50=np.median(f), p84=np.percentile(f, 84),
                p95=np.percentile(f, 95), fmax=f.max(),
                frac_null=consistent.mean())


print(f"AGN null test  (|fracAGN| <= {FRACAGN_NULL_TOL:g} or consistent with 0 "
      f"within {NSIG_NULL:g} sigma)")
print(f"{'arm':<9s}{'N':>6s}{'p50':>9s}{'p84':>9s}{'p95':>9s}{'max':>9s}"
      f"{'no-AGN':>9s}   verdict")
_verdict = {}
for _arm in ("dust_on", "dust_off", "agn_on"):
    _s = _null_stats(FIT[np.asarray(FIT["arm"]) == _arm])
    if _s is None:
        print(f"{_arm:<9s}{'—':>6s}   (no finished runs)")
        continue
    _expect_null = _arm != "agn_on"
    _pass = (_s["frac_null"] >= 0.95) if _expect_null else (_s["frac_null"] < 0.95)
    _verdict[_arm] = _pass
    print(f"{_arm:<9s}{_s['n']:>6d}{_s['p50']:>9.3f}{_s['p84']:>9.3f}"
          f"{_s['p95']:>9.3f}{_s['fmax']:>9.3f}{100*_s['frac_null']:>8.1f}%   "
          + ("PASS" if _pass else "FAIL")
          + ("  (expected no AGN)" if _expect_null else "  (expected an AGN)"))
if _verdict.get("dust_on") is False or _verdict.get("dust_off") is False:
    print("\n  ** the AGN-free arms recovered a finite AGN fraction: the torus "
          "is absorbing stellar/dust light, so Av_ISM from those runs is NOT "
          "clean. Inspect the offending runs before using Parts 7g-7k. **")

# per-aperture breakdown: an AGN-free arm can only leak in the central rungs
print(f"\nmedian fracAGN per arm x aperture (AGN-free arms should stay ~0):")
print(f"{'arm':<9s}" + "".join(f"{l:>10s}" for l in APERTURE_LABELS))
for _arm in ("dust_on", "dust_off", "agn_on"):
    _line = f"{_arm:<9s}"
    for _lab in APERTURE_LABELS:
        _s = FIT[(np.asarray(FIT["arm"]) == _arm)
                 & (np.asarray(FIT["aperture"]) == _lab)]
        _line += (f"{np.median(np.asarray(_s['fracAGN'], float)):>10.3f}"
                  if len(_s) else f"{'—':>10s}")
    print(_line)

# ── does the recovered AGN track the RT truth where an AGN WAS injected? ──
_agnf = os.path.join(TABLEDIR, "agn_flux_contribution.fits")
if os.path.exists(_agnf):
    _AG = Table.read(_agnf)
    _key = {(int(r["snap"]), int(r["gal_id"])): r for r in _AG}
    _sub = FIT[np.asarray(FIT["arm"]) == "agn_on"]
    _x, _y = [], []
    for _r in _sub:
        _tr = _key.get((int(_r["snap"]), int(_r["gal_id"])))
        _col = f"fagn_V_{_r['aperture']}"
        if _tr is None or _col not in _AG.colnames:
            continue
        _v = float(_tr[_col])
        if np.isfinite(_v):
            _x.append(_v)
            _y.append(float(_r["fracAGN"]))
    if len(_x) > 5:
        _x, _y = np.asarray(_x), np.asarray(_y)
        _rho = np.corrcoef(_x, _y)[0, 1]
        print(f"\n[agn_on] CIGALE fracAGN vs RT V-band AGN flux fraction: "
              f"Pearson r = {_rho:.2f} over {len(_x)} (galaxy, aperture) pairs")
        print("   (not a 1:1 identity — CIGALE's fracAGN is defined over the "
              "total dust luminosity, the RT truth is a V-band flux ratio; "
              "the correlation is the test)")
else:
    _x = _y = None
    print(f"\n[agn_on] {_agnf} missing — run Part 7a-agn for the RT truth "
          "comparison")

# ── figures ──
_narm = [a for a in ("dust_on", "dust_off", "agn_on")
         if (np.asarray(FIT["arm"]) == a).any()]
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
_col = {"dust_on": "#0072B2", "dust_off": "#009E73", "agn_on": "#E69F00"}
for _arm in _narm:
    _s = FIT[np.asarray(FIT["arm"]) == _arm]
    axs[0].hist(np.asarray(_s["fracAGN"], float), bins=np.linspace(0, 0.5, 26),
                histtype="step", lw=1.8, color=_col[_arm], label=_arm)
axs[0].axvline(FRACAGN_NULL_TOL, color="0.4", ls=":", lw=1.2)
axs[0].set(xlabel="bayes.agn.fracAGN", ylabel="objects",
           title="AGN fraction per arm")
axs[0].legend(fontsize=8, frameon=False)

for _arm in _narm:
    _s = FIT[np.asarray(FIT["arm"]) == _arm]
    axs[1].hist(np.asarray(_s["Av_ISM"], float), bins=np.linspace(0, 2.0, 26),
                histtype="step", lw=1.8, color=_col[_arm], label=_arm)
axs[1].set(xlabel=r"bayes.attenuation.$A_V^{\rm ISM}$", ylabel="objects",
           title="the measurand (dust_off should pile at 0)")
axs[1].legend(fontsize=8, frameon=False)

if _x is not None and len(_x) > 5:
    axs[2].plot(_x, _y, "o", ms=4, color=_col["agn_on"], alpha=0.7, mec="none")
    axs[2].set(xlabel="RT V-band AGN flux fraction (truth)",
               ylabel="CIGALE bayes.agn.fracAGN",
               title=f"agn_on: recovered vs injected (r={_rho:.2f})")
    axs[2].grid(alpha=0.3)
else:
    axs[2].set_axis_off()
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "cigale_agn_null_test.png"), dpi=150,
            bbox_inches="tight")
plt.show()


# Part 7f — aperture-matched SIMBA truth (cluster, cached once)

CIGALE's estimates describe the stars inside **one projected aperture along one sightline** —
the global caesar/history M\*/SFR/age are only the right truth for the largest aperture. One
pass over the Stage-0 region cutouts measures, per (galaxy, sightline, aperture rung
**and annulus between rungs** — `ann3kpc…ann100kpc`, so the annular CIGALE runs are compared
against the stars of the SAME annulus):

- **M\*** — current stellar mass in the projected cylinder (matching Hyperion's peeled
  apertures: radius r perpendicular to the (θ, φ) sightline, full depth);
- **archaeological SFR** over the last 25 / 100 Myr — mass formed in the window from
  `StellarFormationTime` (current masses, so ≲10–15 % mass-loss bias — noted, not corrected);
- **mass-weighted age and total Z** of the same stars (→ `stellar.age_m_star`,
  `stellar.metallicity`);
- a **delayed+bq fit to the aperture's own archaeological SFH** (50 Myr bins,
  `simbanator.analysis.sfh_utils.fit_delayed_bq` — the same form CIGALE fits), giving the
  per-aperture `sfh.tau_main / age_main / age_bq / r_sfr` truth.

Everything is cached to `tables/aperture_truth.fits` (one row per galaxy × sightline ×
aperture/annulus; `OVERWRITE_APERTURE_TRUTH=True` rebuilds, and a cache without annulus rows
is rebuilt automatically). Part 7g overlays this cache on its truth
table per run; the global values remain the fallback when the cache is absent. Requires the
Stage-0 particle files and the Stage-1 selection HDF5 (centres = the RT-grid `x_cent`), so it
runs on the **cluster**.

In [ ]:
# ── Part 7f — cache SIMBA properties in the SAME apertures/annuli/sightlines as the RT ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (cluster).
from simbanator.analysis.sfh_utils import fit_delayed_bq

APERTURE_TRUTH_FITS      = os.path.join(TABLEDIR, "aperture_truth.fits")
OVERWRITE_APERTURE_TRUTH = False
SFR_WINDOWS_MYR = (25.0, 100.0)     # -> sfh.sfr / sfh.sfr100Myrs truth
ARCH_BIN_MYR    = 50.0              # archaeological-SFH bin for the delayed+bq fit
NSTAR_AP_MIN    = 20                # ages/Z/fits need at least this many star particles

APERTURE_TRUTH = None
if os.path.exists(APERTURE_TRUTH_FITS) and not OVERWRITE_APERTURE_TRUTH:
    APERTURE_TRUTH = Table.read(APERTURE_TRUTH_FITS)
    if not any(str(a).startswith("ann") for a in APERTURE_TRUTH["aperture"]):
        print("cache has no annulus rows (pre-annuli version) -> rebuilding")
        APERTURE_TRUTH = None
    else:
        print(f"cached ({len(APERTURE_TRUTH)} rows) -> {APERTURE_TRUTH_FITS}  "
              "(OVERWRITE_APERTURE_TRUTH=True rebuilds)")
if APERTURE_TRUTH is None:
    SEL, SNAPS, IDS = load_selection()
    _cen = rt_centers(SNAPS, IDS)               # RT-grid centres (code units)

    _ag = np.linspace(0.02, 1.0, 4096)          # a -> t interpolation grid
    _tg = COSMO.age(1.0 / _ag - 1.0).value                      # Gyr

    _rows = []
    _apr = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]        # true rung radii [pkpc]
    for _snap in np.unique(SNAPS):
        _zs = float(sim.get_z_from_snap(int(_snap)))
        _t_obs = float(COSMO.age(_zs).value)
        _n_gal = 0
        for _gid in np.unique(IDS[SNAPS == _snap]):
            _cut = read_cutout(_snap, _gid, _cen.get((int(_snap), int(_gid))),
                               "PartType4",
                               fields=("Masses", "StellarFormationTime", "Metallicity"))
            if _cut is None:
                print(f"  [skip] snap {_snap} gal {_gid}: no cutout/centre or no stars")
                continue
            _pos = _cut["pos"]
            _mst = np.asarray(_cut["Masses"], float) * 1e10 / _cut["h"]
            _af  = np.asarray(_cut["StellarFormationTime"], float)
            _Zst = np.asarray(_cut["Metallicity"], float)
            _Zst = _Zst[:, 0] if _Zst.ndim == 2 else _Zst
            _tf = np.interp(np.clip(_af, _ag[0], 1.0), _ag, _tg)    # formation time [Gyr]
            _n_gal += 1
            for _il, _nv in zip(INCL_LABELS, NHAT):
                _rproj = projected_radius(_pos, _nv)

                def _measure(_msk, _lab, _r):
                    """Truth of the stars in one projected selection (aperture OR annulus)."""
                    _nap = int(_msk.sum())
                    _row = dict(snap=int(_snap), gal_id=int(_gid), incl=_il,
                                aperture=_lab, ap_kpc=float(_r), nstar_ap=_nap,
                                mstar=np.nan, sfr25=np.nan, sfr100=np.nan,
                                age_m_star_myr=np.nan, met_star=np.nan,
                                tau_main_myr=np.nan, age_main_myr=np.nan,
                                age_bq_myr=np.nan, r_sfr=np.nan, fit_r2=np.nan)
                    if _nap:
                        _mm, _tt, _zz = _mst[_msk], _tf[_msk], _Zst[_msk]
                        _row["mstar"] = float(_mm.sum())
                        for _w, _key in zip(SFR_WINDOWS_MYR, ("sfr25", "sfr100")):
                            _row[_key] = float(_mm[_tt >= _t_obs - _w / 1e3].sum()
                                               / (_w * 1e6))
                        if _nap >= NSTAR_AP_MIN:
                            _row["age_m_star_myr"] = float(
                                np.sum(_mm * (_t_obs - _tt)) / _mm.sum() * 1e3)
                            _row["met_star"] = float(np.sum(_mm * _zz) / _mm.sum())
                            _bins = np.arange(_tt.min(), _t_obs + ARCH_BIN_MYR / 1e3,
                                              ARCH_BIN_MYR / 1e3)
                            if _bins.size >= 8:
                                _hm, _ = np.histogram(_tt, bins=_bins, weights=_mm)
                                _tc = 0.5 * (_bins[1:] + _bins[:-1])
                                _fit = fit_delayed_bq(_tc, _hm / (ARCH_BIN_MYR * 1e6),
                                                      _t_obs)
                                if _fit:
                                    for _k2 in ("tau_main_myr", "age_main_myr",
                                                "age_bq_myr", "r_sfr"):
                                        _row[_k2] = float(_fit[_k2])
                                    _row["fit_r2"] = float(_fit["r2"])
                    return _row

                for _lab, _r in zip(APERTURE_LABELS, _apr):     # cumulative rungs
                    _rows.append(_measure(_rproj <= _r, _lab, float(_r)))
                for _ki in range(1, len(_apr)):                 # true annuli (outer rung
                    _rows.append(_measure((_rproj > _apr[_ki - 1])   # names the annulus)
                                          & (_rproj <= _apr[_ki]),
                                          ANNULUS_LABELS[_ki], float(_apr[_ki])))
        print(f"snap {_snap:3d} (z={_zs:.2f}): aperture+annulus truth for {_n_gal} galaxies")
    APERTURE_TRUTH = Table(rows=_rows)
    APERTURE_TRUTH.write(APERTURE_TRUTH_FITS, overwrite=True)
    print(f"{len(APERTURE_TRUTH)} rows ({len(INCL_LABELS)} sightlines x "
          f"{len(APERTURE_LABELS)} apertures + {len(APERTURE_LABELS) - 1} annuli) "
          f"-> {APERTURE_TRUTH_FITS}")


# Part 7g — SIMBA truth vs CIGALE-recovered properties (per run)

For every finished run, `simbanator.sed.cigale.compare_results` joins `out/results.fits` with
the Part 3 selection catalog, **prints the per-galaxy table** (true vs recovered, log where
appropriate) with median-offset/NMAD stats, draws one-to-one panels (colored by anchor
redshift), and **saves both into that run's output folder**:
`<run_dir>/out/simba_vs_cigale.fits` + `.png`.

Compared properties (truth columns are named exactly like the CIGALE variables — add a column
here and it is compared automatically as long as the variable is in `prepare_run`'s
`variables`):

**Aperture-matched truth.** When `tables/aperture_truth.fits` exists (Part 7f), every
truth property below except $A_V$ is replaced, per run, by the SIMBA value measured **inside
that run's projected aperture along that run's sightline** (M\*, archaeological SFR,
mass-weighted age & Z, delayed+bq fit of the aperture's own archaeological SFH). The global
caesar/history values below are only the fallback — they match only the largest aperture.

**Essential:**

- **`stellar.m_star`** — SIMBA stellar mass at the anchor vs CIGALE's Bayesian estimate.
- **`sfh.sfr`** / **`sfh.sfr100Myrs`** — SIMBA SFR from the **smoothed + resampled** SFH
  (same kernel treatment as Part 7d, `simbanator.analysis.sfh_utils.recent_sfr`), averaged
  over the last 25 / 100 Myr before the anchor — the raw per-snapshot SFR is instantaneous
  and too stochastic to compare with an SED-derived estimate. For fully quenched fits
  CIGALE's posterior SFR underflows to ~0 — those points pin as open symbols at the plot
  floor.
- **`stellar.age_m_star`** — mass-weighted stellar age: caesar `ages.mass_weighted` (Gyr→Myr)
  vs CIGALE bc03's mass-weighted SSP age. Same definition on both sides.
- **`attenuation.Av_ISM`** — per run: the TRUE rest-frame Johnson-V attenuation
  $A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ of that run's (aperture, sightline) catalog
  pair vs CF00's `Av_ISM` (for these old-star-dominated systems the effective V attenuation
  ≈ `Av_ISM`; young stars add `Av_BC` on top). Annular runs difference the adjacent aperture
  catalogs ($F(<r_{\rm out})-F(<r_{\rm in})$, on & off). dust_off runs use truth $A_V = 0$ —
  a null test that CIGALE recovers no attenuation from dust-free photometry. The direct
  powderday measurement is always carried as **`A_V_powderday`** in the saved table, and 7g
  warns when a `results.fits` predates `attenuation.Av_ISM` in `variables` (no bayes.* → no
  panel; re-fit to get it).

**SFH shape** (`sfh.tau_main`, `sfh.age_main`, `sfh.age_bq`, `sfh.r_sfr`) — the truth is
**not** the SFT/QT clocks: it is the best fit of CIGALE's *own* `sfhdelayedbq` form to the
smoothed SIMBA SFH (Part 7d → `tables/sfh_delayedbq_fits.fits`), so both sides describe the
same model and the comparison isolates what the *SED* constrains, not the model mismatch.
Requires Part 7d to have run; its fit quality is carried as `sfh_fit_r2`, and the SFT/QT
clocks remain as context columns `age_sft_myr` / `age_qt_myr` (`t_sft`/`t_qt` are cosmic
times in **yr**).

**Dusty vs non-dusty.** The Part 7a dusty flag (global $A_V > 0.1$, same as Part 4b) is
joined into the truth table when `attenuation_vs_ism.fits` exists: dusty galaxies get a
**red ring** in every one-to-one panel (on top of the redshift colors) and the median-offset /
NMAD stats are printed per subsample — recovery biases that differ between the two are the
signature of attenuation-driven systematics in the fits. The `dusty` column is carried into
each saved `simba_vs_cigale.fits` (pass `highlight_col=None` to `compare_results` to disable).

Reading the output: `<prop>_cigale` is the Bayesian (posterior-mean) estimate with
`<prop>_cigale_err`; `<prop>_best` is the best-χ² model. **NaN bayes with finite best** means
the posterior weights underflowed because even the best model fits poorly — widen the grids in
`module_params` for those objects rather than trusting the best-fit value.

In [ ]:
# ── Part 7g: original SIMBA properties vs the CIGALE-recovered ones, per run ──
# Self-contained after Part 0: reads SELECTION_FITS, the anchor-history HDF5s in
# SFHDIR (smoothed truth SFR + mass-weighted age), the Part 7d delayed+bq fit
# table (SFH-parameter truth) + each <run_dir>/out/results.fits.
from simbanator.sed import cigale as cg
from simbanator.sed.flux_extraction import attenuation_mag

SEL, SNAPS, IDS = load_selection()
truth = Table()
truth["id"] = [f"snap{int(s):03d}_gal{int(g)}"
               for s, g in zip(SEL["snap"], SEL["gal_id"])]
truth["z_snap"]    = np.asarray(SEL["z_snap"], float)   # colors the one-to-one panels
truth["agn_class"] = SEL["agn_class"]                   # carried into the saved table
# truth columns named exactly like the CIGALE variables they are compared to:
truth["stellar.m_star"] = 10.0 ** np.asarray(SEL["log_mstar"], float)         # Msun
# truth SFR from the smoothed+resampled SFH (Part 7d treatment) — the raw per-snapshot
# SFR is instantaneous and too stochastic to compare with an SED-derived estimate.
# 100 Myr average -> CIGALE sfh.sfr100Myrs; 25 Myr -> ~instantaneous sfh.sfr, de-burst-ed.
from simbanator.analysis.sfh_utils import recent_sfr
_sfhdb = {}
for _hf in sorted(glob.glob(os.path.join(SFHDIR, "history_anchor_*.hdf5"))):
    with h5py.File(_hf, "r") as f:
        _snap0 = int(f["metadata/snapshots"][0])               # row 0 = anchor epoch
        _gid   = np.asarray(f["metadata/galaxy_ids"][:], int)
        _tgyr  = COSMO.age(f["redshift/Redshift"][:]).value    # Gyr
        _sfr   = np.asarray(f["properties/sfr"][:], float)     # (n_snap, n_gal)
    _o = np.argsort(_tgyr)
    for _j, _g in enumerate(_gid):
        _sfhdb[(_snap0, int(_g))] = (_tgyr[_o], _sfr[_o, _j])

def _truth_sfr(avg_myr):
    out = np.full(len(SEL), np.nan)
    for _k, (_s, _g) in enumerate(zip(SEL["snap"], SEL["gal_id"])):
        _tr = _sfhdb.get((int(_s), int(_g)))
        if _tr is not None:
            out[_k] = recent_sfr(_tr[0], _tr[1], avg_myr=avg_myr)
    return out

truth["sfh.sfr"]        = _truth_sfr(25.0)
truth["sfh.sfr100Myrs"] = _truth_sfr(100.0)
_nsfr = int(np.isfinite(np.asarray(truth["sfh.sfr"], float)).sum())
print(f"truth SFR from smoothed SFHs: {_nsfr}/{len(SEL)} matched to anchor histories")
# mass-weighted stellar age at the anchor (caesar 'ages.mass_weighted', Gyr):
# the same definition as CIGALE's bc03 'stellar.age_m_star' (Myr)
_agedb = anchor_row0(("ages.mass_weighted",))
truth["stellar.age_m_star"] = row0_arr(_agedb, SNAPS, IDS,
                                       "ages.mass_weighted") * 1e3    # Gyr -> Myr

# SFH-parameter truth = the best fit of CIGALE's OWN sfhdelayedbq form to the
# smoothed SIMBA SFH (Part 7d -> sfh_delayedbq_fits.fits). The SFT/QT clocks
# measure a different thing and are only carried as context columns.
_t_obs_yr = COSMO.age(truth["z_snap"]).value * 1e9      # t_sft / t_qt are in yr
truth["age_sft_myr"] = (_t_obs_yr - np.asarray(SEL["t_sft"], float)) / 1e6   # carried
truth["age_qt_myr"]  = (_t_obs_yr - np.asarray(SEL["t_qt"], float)) / 1e6    # carried
_fitf = os.path.join(TABLEDIR, "sfh_delayedbq_fits.fits")
if os.path.exists(_fitf):
    _ftab = Table.read(_fitf)
    _fdb = {(int(r["snap"]), int(r["gal_id"])): r for r in _ftab}
    for _tcol, _fcol in [("sfh.tau_main", "tau_main_myr"),
                         ("sfh.age_main", "age_main_myr"),
                         ("sfh.age_bq", "age_bq_myr"), ("sfh.r_sfr", "r_sfr")]:
        truth[_tcol] = np.array(
            [float(_fdb[(int(s), int(g))][_fcol]) if (int(s), int(g)) in _fdb
             else np.nan for s, g in zip(SEL["snap"], SEL["gal_id"])])
    truth["sfh_fit_r2"] = np.array(
        [float(_fdb[(int(s), int(g))]["r2"]) if (int(s), int(g)) in _fdb
         else np.nan for s, g in zip(SEL["snap"], SEL["gal_id"])])           # carried
    _nfit = int(np.isfinite(np.asarray(truth["sfh.age_bq"], float)).sum())
    print(f"SFH-parameter truth from {os.path.basename(_fitf)}: {_nfit}/{len(SEL)} fitted")
else:
    print(f"WARNING: {_fitf} missing — run Part 7d first; "
          "sfh.tau_main/age_main/age_bq/r_sfr will not be compared")

# dusty split (Part 7a, global A_V > 0.1 at the fiducial aperture/sightline — the same
# flag as Part 4b): compare_results rings the dusty galaxies in every one-to-one panel and
# prints the offset/NMAD stats per subsample. -1 = unmatched (never ringed).
_davg, truth["dusty"] = dusty_flags(SNAPS, IDS)
print(f"dusty split (Part 7a A_V > {AV_DUSTY:g}): {int((truth['dusty'] == 1).sum())} dusty"
      f" / {int((truth['dusty'] == 0).sum())} non-dusty"
      f" / {int((truth['dusty'] == -1).sum())} unmatched")

# aperture-matched truth cache (Part 7f): per (sightline, aperture) SIMBA
# properties measured exactly as CIGALE sees them; the global values above stay
# as the fallback when the cache is missing.
_apt_f = os.path.join(TABLEDIR, "aperture_truth.fits")
_APT = {}
if os.path.exists(_apt_f):
    for _r in Table.read(_apt_f):
        _APT[(str(_r["aperture"]), str(_r["incl"]),
              int(_r["snap"]), int(_r["gal_id"]))] = _r
    print(f"aperture-matched truth cache: {len(_APT)} rows from {os.path.basename(_apt_f)}")
else:
    print(f"WARNING: {_apt_f} missing — run Part 7f; using GLOBAL truth for all runs")

COMPARISONS = {}
for run_dir in sorted(glob.glob(os.path.join(RUN_BASE, "*"))):
    if not os.path.exists(os.path.join(run_dir, "out", "results.fits")):
        continue
    tag = os.path.basename(run_dir)
    # per-run A_V truth: rest-frame Johnson-V attenuation from the matched
    # dust_on/dust_off catalogs of this run's (aperture, sightline); compared
    # to CF00's Av_ISM — for these old-star-dominated systems the effective
    # V attenuation ~ Av_ISM (young stars add Av_BC on top). Annular runs use
    # the DIFFERENCED fluxes F(<r_out) - F(<r_in) of the adjacent aperture
    # catalogs (same construction as Part 7a; non-positive dF -> NaN).
    # dust_off runs get truth A_V = 0 (null test).
    _truth_run = truth.copy()
    _m = re.match(r"^(dust_on|dust_off)_(.+?)_(i\d+p\d+)(?:_snap\d+)?(?:_Zs\w+)?(?:_sfhinj)?$", tag)
    if _m is not None:
        _key, _lab, _il = _m.groups()

        def _vmap(_kk, _ll):
            _fc = os.path.join(CATDIR, f"catalog_{_kk}_{_ll}_{_il}.fits")
            if not os.path.exists(_fc):
                return None
            _tc = Table.read(_fc)
            if "Johnson.V.V" not in _tc.colnames:
                return None
            return {(int(s), int(g)): float(v) for s, g, v in
                    zip(_tc["snap"], _tc["gal_id"], _tc["Johnson.V.V"])}

        if _lab.startswith("ann"):          # annular V = F(<r_out) - F(<r_in)
            _outer = _lab.replace("ann", "ap")
            _inner = APERTURE_LABELS[APERTURE_LABELS.index(_outer) - 1]
            _don = _dof = None
            _vo, _vi = _vmap("dust_on", _outer), _vmap("dust_on", _inner)
            _wo, _wi = _vmap("dust_off", _outer), _vmap("dust_off", _inner)
            if None not in (_vo, _vi, _wo, _wi):
                _don = {k: _vo[k] - _vi[k] for k in _vo if k in _vi}
                _dof = {k: _wo[k] - _wi[k] for k in _wo if k in _wi}
        else:
            _don, _dof = _vmap("dust_on", _lab), _vmap("dust_off", _lab)
        if _don and _dof:
            _av = attenuation_mag(by_snap_gal(_don, SNAPS, IDS),
                                  by_snap_gal(_dof, SNAPS, IDS))
            # the DIRECT powderday on/off measurement is always carried into the
            # saved table (even for dust_off runs, where the compared truth is
            # the zero-A_V null test)
            _truth_run["A_V_powderday"] = np.round(_av, 3)
            _truth_run["attenuation.Av_ISM"] = (
                _av if _key == "dust_on" else np.zeros(len(SEL)))
    if _m is not None and _APT:
        # overlay the aperture-matched truth for this run's (aperture, sightline)
        _lab, _il = _m.group(2), _m.group(3)
        _cols = {"stellar.m_star": "mstar", "sfh.sfr": "sfr25",
                 "sfh.sfr100Myrs": "sfr100", "stellar.age_m_star": "age_m_star_myr",
                 "stellar.metallicity": "met_star",
                 "sfh.tau_main": "tau_main_myr", "sfh.age_main": "age_main_myr",
                 "sfh.age_bq": "age_bq_myr", "sfh.r_sfr": "r_sfr"}
        _nhit = 0
        for _tcol, _acol in _cols.items():
            _v = np.full(len(SEL), np.nan)
            for _k, (_s, _g) in enumerate(zip(SEL["snap"], SEL["gal_id"])):
                _r = _APT.get((_lab, _il, int(_s), int(_g)))
                if _r is not None:
                    _v[_k] = float(_r[_acol])
            if np.isfinite(_v).any():
                _truth_run[_tcol] = _v
                _nhit = max(_nhit, int(np.isfinite(_v).sum()))
        print(f"[{tag}] {'annulus' if _lab.startswith('ann') else 'aperture'}"
              f"-matched truth ({_lab}/{_il}): "
              f"{_nhit}/{len(SEL)} galaxies from the cache")
    elif not _APT:
        print(f"[{tag}] GLOBAL truth (no aperture cache)")
    # runs fitted before attenuation.Av_ISM entered `variables` have no
    # bayes.* for it -> the A_V panel silently disappears; make that loud
    _res_cols = Table.read(os.path.join(run_dir, "out", "results.fits")).colnames
    if ("attenuation.Av_ISM" in _truth_run.colnames
            and "bayes.attenuation.Av_ISM" not in _res_cols):
        print(f"[{tag}] WARNING: results.fits has no bayes.attenuation.Av_ISM — "
              "this run predates the current `variables`; re-run Part 7e + the "
              "array to get the A_V panel")
    print(f"\n{'=' * 25} {tag}: SIMBA vs CIGALE {'=' * 25}")
    COMPARISONS[tag] = cg.compare_results(run_dir, _truth_run)
    # -> prints per-galaxy table + offset/NMAD stats;
    #    saves <run_dir>/out/simba_vs_cigale.fits + .png

print(f"\n{len(COMPARISONS)} run(s) compared "
      f"-> simba_vs_cigale.fits/.png in each <run_dir>/out/")

# Part 7h — dust_on vs dust_off: fit stability & the Mdust prediction gap

The same galaxies, the same SFH/metallicity grids, two RTs: with the SIMBA dust and with
dust zeroed. Pairing each `dust_on_*` run with its `dust_off_*` twin (same aperture/annulus,
sightline, snapshot, Z group; matched by object id) answers two questions:

- **is the fit stable?** If CIGALE's stellar masses, ages and SFH parameters move when the
  attenuation changes, the recovery is degenerate with dust, not driven by the data;
- **what does CIGALE predict for `dust.mass` without FIR information?** dust_off runs are
  fitted on the optical→MIRI bands only, with a near-zero attenuation grid and a single
  dust-emission combo — their `dust.mass`/`dust.luminosity` come purely from energy balance,
  so the on−off offset shows how much of the Mdust estimate is data vs prior.

One row per (object, run pair) → `tables/dust_on_off_stability.fits`; one-to-one panels
(dust_on on x, dust_off on y, colored by anchor) → `plots/.../dust_on_off_stability.png`;
median offsets/NMAD printed per property (Gyr for ages/τ, dex for masses/SFR/Mdust).
The same pairing runs a second time against the **`agn_on`** results (fitted with the same
AGN-free grids) → `tables/dust_on_agn_stability.fits` + `dust_on_agn_stability.png`: the
on−agn offsets quantify how an **unmodeled AGN** biases $M_*$, ages, the SFH parameters and
$M_{\rm dust}$ (skipped with a note until the `agn_on` CIGALE runs exist). The flux-level
counterpart is Part 7a-agn.


In [ ]:
# ── Part 7h: dust_on vs dust_off (and vs agn_on) CIGALE estimates — stability + biases ──
# Self-contained after Part 0: reads the finished cigale_runs/ pairs.
from simbanator.sed.cigale import nmad

_PAIR_PROPS = ["stellar.m_star", "stellar.age_m_star", "sfh.sfr100Myrs",
               "sfh.tau_main", "sfh.age_bq", "sfh.r_sfr",
               "dust.mass", "dust.luminosity"]
_LOGP = {"stellar.m_star", "sfh.sfr100Myrs", "dust.mass", "dust.luminosity"}
_GYRP = {"stellar.age_m_star", "sfh.tau_main", "sfh.age_bq"}

def _pair_stability(other, out_stem, required=False):
    """Pair every dust_on_* run dir with its `<other>_*` twin (same aperture/annulus,
    sightline, snapshot, Z group; matched by object id) -> FITS + one-to-one panels.
    Returns the pair table, or None when no pairs exist yet (required=False)."""
    _rows = []
    for _don_dir in sorted(glob.glob(os.path.join(RUN_BASE, "dust_on_*"))):
        _oth_dir = os.path.join(RUN_BASE,
                                other + "_" + os.path.basename(_don_dir)[len("dust_on_"):])
        _fon = os.path.join(_don_dir, "out", "results.fits")
        _fot = os.path.join(_oth_dir, "out", "results.fits")
        if not (os.path.exists(_fon) and os.path.exists(_fot)):
            continue
        _ton, _tot = Table.read(_fon), Table.read(_fot)
        _ioth = {str(i): k for k, i in enumerate(_tot["id"])}
        _ptag = os.path.basename(_don_dir)[len("dust_on_"):]
        for _k, _oid in enumerate(_ton["id"]):
            _j = _ioth.get(str(_oid))
            if _j is None:
                continue
            _row = {"id": str(_oid), "pair": _ptag,
                    "snap": int(str(_oid).split("_")[0][4:])}
            for _p in _PAIR_PROPS:
                _c = f"bayes.{_p}"
                _row[f"{_p}_on"] = (float(_ton[_c][_k])
                                    if _c in _ton.colnames else np.nan)
                _row[f"{_p}_oth"] = (float(_tot[_c][_j])
                                     if _c in _tot.colnames else np.nan)
            _rows.append(_row)
    if not _rows:
        _msg = f"no finished dust_on/{other} run pairs under {RUN_BASE}"
        if required:
            raise FileNotFoundError(_msg)
        print(f"[{other}] {_msg} — skipped\n")
        return None
    _pairs = Table(rows=_rows)
    _out = os.path.join(TABLEDIR, f"{out_stem}.fits")
    _pairs.write(_out, overwrite=True)
    print(f"[{other}] {len(_pairs)} (object, run-pair) rows from "
          f"{len(set(_pairs['pair']))} run pairs -> {_out}\n")

    _snaps_u = sorted(set(int(s) for s in _pairs["snap"]))
    _cols = plt.cm.viridis(np.linspace(0, 0.9, len(_snaps_u)))
    _n, _nc = len(_PAIR_PROPS), 4
    _nr = int(np.ceil(_n / _nc))
    _fig, _axs = plt.subplots(_nr, _nc, figsize=(4.6 * _nc, 4.4 * _nr), squeeze=False)
    print(f"{'property':>22s} | {'median ' + other + '-on':>18s} {'NMAD':>8s} {'n':>5s}")
    for _k, _p in enumerate(_PAIR_PROPS):
        _ax = _axs[_k // _nc][_k % _nc]
        _x = np.asarray(_pairs[f"{_p}_on"], float)
        _y = np.asarray(_pairs[f"{_p}_oth"], float)
        _scale = 1e-3 if _p in _GYRP else 1.0
        _x, _y = _x * _scale, _y * _scale
        _ok = np.isfinite(_x) & np.isfinite(_y)
        if _p in _LOGP:
            _ok &= (_x > 0) & (_y > 0)
            _d = np.log10(_y[_ok]) - np.log10(_x[_ok]); _u = " dex"
        else:
            _d = _y[_ok] - _x[_ok]; _u = " Gyr" if _p in _GYRP else ""
        _nmad = nmad(_d)
        print(f"{_p:>22s} | {np.median(_d) if _d.size else np.nan:+18.3f}{_u:5s}"
              f"{_nmad:8.3f} {int(_ok.sum()):5d}")
        for _sv, _cc in zip(_snaps_u, _cols):
            _m = _ok & (np.asarray(_pairs["snap"], int) == _sv)
            _ax.scatter(_x[_m], _y[_m], s=14, color=_cc, alpha=0.6,
                        label=f"snap {_sv}" if _k == 0 else None)
        if _ok.any():
            _lo, _hi = (np.nanmin(np.r_[_x[_ok], _y[_ok]]),
                        np.nanmax(np.r_[_x[_ok], _y[_ok]]))
            _ax.plot([_lo, _hi], [_lo, _hi], "--", color="0.6", lw=1, zorder=0)
        if _p in _LOGP:
            _ax.set_xscale("log"); _ax.set_yscale("log")
        _unit = " [Gyr]" if _p in _GYRP else ""
        _ax.set_xlabel(f"dust_on {_p}{_unit}", fontsize=9)
        _ax.set_ylabel(f"{other} {_p}{_unit}", fontsize=9)
        _ax.tick_params(labelsize=8)
        _ax.set_title(_p, fontsize=10)
    for _k in range(_n, _nr * _nc):
        _axs[_k // _nc][_k % _nc].set_axis_off()
    _h, _l = _axs[0][0].get_legend_handles_labels()
    _fig.suptitle(f"CIGALE dust_on vs {other} — same galaxies, same grids", y=0.995)
    _fig.tight_layout(rect=(0, 0, 1, 0.94))
    if _h:
        _fig.legend(_h, _l, loc="upper center", ncol=len(_h), frameon=False,
                    fontsize=9, bbox_to_anchor=(0.5, 0.975))
    plt.savefig(os.path.join(PLOTDIR, f"{out_stem}.png"), dpi=140, bbox_inches="tight")
    plt.show()
    return _pairs

PAIRS = _pair_stability("dust_off", "dust_on_off_stability", required=True)
# agn_on runs are fitted with the SAME AGN-free grids -> the on-agn offsets are the
# unmodeled-AGN bias on M*, ages, SFH and Mdust. Skipped until those runs exist.
PAIRS_AGN = _pair_stability("agn_on", "dust_on_agn_stability")


# Part 7i — decoupled fits I: optical-only CIGALE runs on the dust_on photometry

Third fit configuration for the stability ladder (Part 7k): the same dust_on photometry as
the coupled runs, but with the optical side fitted **alone** — the energy-balance constraint
switched off. Every prepared `dust_on_*` run dir under `cigale_runs/` is cloned into an
`optonly_dust_on_*` twin by `simbanator.sed.cigale.optical_only_run`, which edits the ini
instead of re-preparing — same data file, same per-snapshot age grids and metallicity
sub-runs **by construction**:

- **`dl2014` removed** from the module chain: the CF00-absorbed luminosity is not re-emitted,
  so the IR pulls nothing. `dust.luminosity` (set by the attenuation module) is still
  estimated — that is the **L_abs** side of Part 7g's energy-balance test (`dust.mass` is
  created by dl2014 and is dropped from the variables).
- **MIRI + MIPS + SCUBA-2 + ALMA leave the fitted bands** but stay in the analysis `bands`,
  so the run still predicts `bayes.<band>` there. With no dust emission in the model those
  predictions are the pure attenuated **stellar continuum** — exactly what Part 7j subtracts
  from the observed IR before fitting DL2014 (essential for MIRI/MIPS 24 in a quenched
  sample, where the stellar Rayleigh–Jeans tail rivals the weak dust emission).

The model grid per run shrinks by the dl2014 factor (18×), so the array is cheap. Scope with
`OPTONLY_INCLUDE` (e.g. `r"ap100kpc_i0p0"` for a total-aperture face-on first pass — if
energy balance holds there, the aperture/annulus fan-out may be unnecessary). Needs the
Part 7e run dirs **prepared** (not fitted); re-run this cell whenever 7e re-prepares.
Workflow: this cell → `sbatch cigale_runs/submit_cigale_optonly_array.job` → Part 7j.


In [ ]:
# ── Part 7i: clone every prepared dust_on run dir into an optical-only twin ──
from simbanator.sed import cigale as cg

PCIGALE_CMD = cg.find_pcigale()      # dedicated conda env; see the Part 7e markdown
OPTONLY_INCLUDE = None    # regex on the coupled tag, e.g. r"ap100kpc_i0p0" for a scoped pass
CORES_PER_TASK  = 8
PLOT_SEDS       = True
SKIP_IF_DONE    = False   # as in 7e: True = finished runs exit at once on resubmit

_srcs = [d for d in sorted(glob.glob(os.path.join(RUN_BASE, "dust_on_*")))
         if os.path.isdir(d) and os.path.exists(os.path.join(d, "pcigale.ini"))]
if OPTONLY_INCLUDE:
    _srcs = [d for d in _srcs if re.search(OPTONLY_INCLUDE, os.path.basename(d))]
assert _srcs, f"no prepared dust_on_* run dirs under {RUN_BASE} — run Part 7e first"

OPTONLY_DIRS = []
for _src in _srcs:
    _dst = os.path.join(RUN_BASE, "optonly_" + os.path.basename(_src))
    cg.optical_only_run(_src, _dst, verbose=False)
    OPTONLY_DIRS.append(_dst)
print(f"{len(OPTONLY_DIRS)} optical-only run dirs cloned "
      "(no dl2014; MIR/FIR bands unfitted but still predicted)")

JOB_FILE = cg.write_slurm_array(
    OPTONLY_DIRS, os.path.join(RUN_BASE, "submit_cigale_optonly_array.job"),
    pcigale_cmd=PCIGALE_CMD,
    partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
    cores=CORES_PER_TASK, time="0-06:00", job_name="cigale_optonly",
    plots=PLOT_SEDS, skip_if_done=SKIP_IF_DONE)
# -> sbatch it on a login node; when the array drains, run Part 7j


# Part 7j — decoupled fits II: standalone Draine & Li (2014) fits of the IR residual

The dust side of the decoupled fit — run **after** the optonly array has drained. For every
`optonly_dust_on_*` run dir, `simbanator.sed.dl2014_fit` (executed with the **cigale env's
python** — it needs the pcigale template/filter databases; one subprocess handles all dirs):

1. reads the run's own `data_file` (observed IR fluxes) and its `out/results.fits`;
2. subtracts the attenuated stellar continuum predicted by the optical-only fit
   (`bayes.<band>`; SCUBA-2/ALMA have no prediction — BC03 stops at rest 160 µm — and are
   used unsubtracted, the Rayleigh–Jeans tail being negligible there);
3. fits the residual with DL2014 templates combined exactly as CIGALE's dl2014 module does
   ((1−γ)·δ(U_min) + γ·power law U_min→10⁷, α = 2), with the normalization **L_dust** solved
   analytically per (q_PAH, U_min, γ) grid point — no energy balance anywhere;
4. writes `out/dl2014_results.fits`: CIGALE-style `best.*`/`bayes.*` columns
   (`dust.luminosity` in W, `dust.mass` in kg — the CIGALE conventions, directly comparable
   to the coupled columns) plus per-band `fobs./fstar./fdust./fmodel.` diagnostics.

Error budget matches CIGALE: MC error ⊕ `additionalerror`·F_obs (read from the run's ini)
⊕ the stellar-prediction error, in quadrature. Grid: q_PAH/U_min include the coupled Part
7e nodes (points comparable); γ floats where the coupled grid pins 0.0085. A negative
best amplitude is clamped to 0 and flagged (`flag_upperlimit = 1`) — expected for the most
gas-poor quenched objects, whose L_dust is then bounded by `bayes.dust.luminosity_err` (1σ).


In [ ]:
# ── Part 7j: standalone DL2014 fits of the IR residual (cluster; cigale-env python) ──
from simbanator.sed import cigale as cg
from simbanator.sed import dl2014_fit as _dlmod

PCIGALE_CMD = cg.find_pcigale()      # dedicated conda env; see the Part 7e markdown
CIGALE_PY = os.path.join(os.path.dirname(PCIGALE_CMD), "python")

_cmd = [CIGALE_PY, os.path.abspath(_dlmod.__file__),
        "--run-base", RUN_BASE, "--pattern", "optonly_dust_on_*"]
print("$", " ".join(_cmd), "\n")
_p = subprocess.run(_cmd)
assert _p.returncode == 0, "dl2014_fit failed — see the output above"

DL_FILES = sorted(glob.glob(os.path.join(RUN_BASE, "optonly_dust_on_*",
                                         "out", "dl2014_results.fits")))
print(f"\n{len(DL_FILES)} dl2014_results.fits under {RUN_BASE}")


# Part 7k — three-way fit stability & the energy-balance test

The full decoupling ladder on the same objects, same SFH/metallicity grids:

| config | attenuation | dust emission | adds |
|---|---|---|---|
| **off** (`dust_off_*`, 7e) | ~none (dust-free RT) | prior-only | baseline: fitting-machinery systematics |
| **opt** (`optonly_dust_on_*`, 7i) | CF00, optical bands only | none | attenuation, no energy balance |
| **full** (`dust_on_*`, 7e) | CF00 | DL2014 via energy balance | the energy-balance constraint |
| **DL** (7j) | — | DL2014, free normalization | the IR data alone |

**opt − off** isolates the attenuation-law mismatch (CF00 vs the real powderday geometry);
**full − opt** isolates what the energy-balance coupling does to the optical solution. The
energy-balance test itself is **L_abs(opt)** (`dust.luminosity` of the optical-only run —
the CF00-absorbed power, never re-emitted) vs **L_dust(DL)** (the IR luminosity actually
there): their log ratio per aperture/annulus × sightline measures how badly energy balance
fails in the mocks — annuli, where dust heated inside can radiate outside (and vice versa),
are where it should fail hardest, and where the full fit should drift furthest from opt.

Output: one row per (object, run trio) → `tables/decoupled_stability.fits`; stability panels
(x = coupled full fit, y = optical-only, colored by aperture/annulus, dust_off in grey) →
`plots/…/decoupled_stability.png`; energy-balance figure (L_abs vs L_dust one-to-one, coupled
vs free L_dust, log-ratio histograms; DL upper limits as open triangles at their 1σ bound) →
`plots/…/energy_balance_decoupled.png`. Self-contained after Part 0.


In [ ]:
# ── Part 7k: three-way stability (full / opt / off) + the energy-balance test ──
# Self-contained after Part 0: reads the finished cigale_runs/ trios + DL fits.
from simbanator.sed.cigale import nmad
LSUN_W   = 3.828e26                     # IAU nominal solar luminosity [W]

_PROPS = ["stellar.m_star", "stellar.age_m_star", "sfh.sfr100Myrs",
          "sfh.tau_main", "sfh.age_bq", "sfh.r_sfr", "attenuation.Av_ISM"]
_LOGP  = {"stellar.m_star", "sfh.sfr100Myrs"}
_GYRP  = {"stellar.age_m_star", "sfh.tau_main", "sfh.age_bq"}
_AP_RE = re.compile(r"_((?:ap|ann)\d+kpc)_")

def _get(tab, j, col):
    return (float(tab[col][j]) if (tab is not None and j is not None
                                   and col in tab.colnames) else np.nan)

_rows = []
for _opt_dir in sorted(glob.glob(os.path.join(RUN_BASE, "optonly_dust_on_*"))):
    _tag = os.path.basename(_opt_dir)[len("optonly_"):]          # dust_on_...
    _files = {"opt":  os.path.join(_opt_dir, "out", "results.fits"),
              "full": os.path.join(RUN_BASE, _tag, "out", "results.fits"),
              "off":  os.path.join(RUN_BASE, "dust_off_" + _tag[len("dust_on_"):],
                                   "out", "results.fits"),
              "dl":   os.path.join(_opt_dir, "out", "dl2014_results.fits")}
    if not (os.path.exists(_files["opt"]) and os.path.exists(_files["full"])):
        continue                     # optonly or its coupled twin not fitted yet
    _t = {k: (Table.read(p) if os.path.exists(p) else None)
          for k, p in _files.items()}
    _idx = {k: ({str(v): j for j, v in enumerate(_t[k]["id"])}
                if _t[k] is not None else {}) for k in ("full", "off", "dl")}
    _m = _AP_RE.search("_" + _tag + "_")
    _ap = _m.group(1) if _m else "?"
    for _i, _oid in enumerate(np.asarray(_t["opt"]["id"], str)):
        _jf, _jo, _jd = (_idx[k].get(_oid) for k in ("full", "off", "dl"))
        _row = {"id": _oid, "tag": _tag, "aperture": _ap,
                "snap": int(_oid.split("_")[0][4:]),
                "is_annulus": _ap.startswith("ann")}
        for _p in _PROPS:
            _row[f"{_p}_opt"]  = _get(_t["opt"],  _i,  f"bayes.{_p}")
            _row[f"{_p}_full"] = _get(_t["full"], _jf, f"bayes.{_p}")
            _row[f"{_p}_off"]  = _get(_t["off"],  _jo, f"bayes.{_p}")
        _row["Labs_opt"]      = _get(_t["opt"],  _i,  "bayes.dust.luminosity")
        _row["Ldust_full"]    = _get(_t["full"], _jf, "bayes.dust.luminosity")
        _row["Mdust_full"]    = _get(_t["full"], _jf, "bayes.dust.mass")
        _row["Ldust_dl"]      = _get(_t["dl"], _jd, "bayes.dust.luminosity")
        _row["Ldust_dl_err"]  = _get(_t["dl"], _jd, "bayes.dust.luminosity_err")
        _row["Mdust_dl"]      = _get(_t["dl"], _jd, "bayes.dust.mass")
        _row["chi2red_dl"]    = _get(_t["dl"], _jd, "chi2_red")
        _ul = _get(_t["dl"], _jd, "flag_upperlimit")
        _row["dl_upperlimit"] = int(_ul) if np.isfinite(_ul) else -1
        _rows.append(_row)

assert _rows, (f"no optonly_dust_on_* runs with results under {RUN_BASE} — "
               "run Part 7i (+ its SLURM array) and Part 7j first")
TRIO = Table(rows=_rows)
_outf = os.path.join(TABLEDIR, "decoupled_stability.fits")
TRIO.write(_outf, overwrite=True)
_ndl = int(np.isfinite(np.asarray(TRIO["Ldust_dl"], float)).sum())
print(f"{len(TRIO)} (object, trio) rows from {len(set(TRIO['tag']))} tags "
      f"({_ndl} with a DL fit) -> {_outf}\n")

# ── stability: how far does the optical solution move when decoupled? ──
def _diff(p, a):
    x = np.asarray(TRIO[f"{p}_full"], float)
    y = np.asarray(TRIO[f"{p}_{a}"], float)
    if p in _GYRP:
        x, y = x * 1e-3, y * 1e-3
    ok = np.isfinite(x) & np.isfinite(y)
    if p in _LOGP:
        ok &= (x > 0) & (y > 0)
        return np.log10(y[ok]) - np.log10(x[ok]), " dex"
    return y[ok] - x[ok], (" Gyr" if p in _GYRP else "")

print(f"{'property':>22s} | {'med(opt-full)':>15s} {'NMAD':>7s} | "
      f"{'med(off-full)':>15s} {'NMAD':>7s}")
for _p in _PROPS:
    _cols = []
    for _a in ("opt", "off"):
        _d, _u = _diff(_p, _a)
        if _d.size:
            _nmad = nmad(_d)
            _cols.append(f"{np.median(_d):+11.3f}{_u:4s} {_nmad:7.3f}")
        else:
            _cols.append(f"{'-':>15s} {'-':>7s}")
    print(f"{_p:>22s} | {_cols[0]} | {_cols[1]}")

_aps = sorted(set(np.asarray(TRIO["aperture"], str)))
_apc = dict(zip(_aps, plt.cm.viridis(np.linspace(0, 0.9, len(_aps)))))
_nc = 4
_nr = int(np.ceil(len(_PROPS) / _nc))
_fig, _axs = plt.subplots(_nr, _nc, figsize=(4.6*_nc, 4.4*_nr), squeeze=False)
for _k, _p in enumerate(_PROPS):
    _ax = _axs[_k // _nc][_k % _nc]
    _sc = 1e-3 if _p in _GYRP else 1.0
    _x  = np.asarray(TRIO[f"{_p}_full"], float) * _sc
    _yo = np.asarray(TRIO[f"{_p}_opt"], float) * _sc
    _yf = np.asarray(TRIO[f"{_p}_off"], float) * _sc
    _ok = np.isfinite(_x) & np.isfinite(_yo)
    if _p in _LOGP:
        _ok &= (_x > 0) & (_yo > 0)
    _ax.scatter(_x, _yf, s=10, color="0.8", alpha=0.5, zorder=1,
                label="dust_off" if _k == 0 else None)
    for _a in _aps:
        _msk = _ok & (np.asarray(TRIO["aperture"], str) == _a)
        _ax.scatter(_x[_msk], _yo[_msk], s=14, color=_apc[_a], alpha=0.7,
                    zorder=2, label=_a if _k == 0 else None)
    if _ok.any():
        _lo = np.nanmin(np.r_[_x[_ok], _yo[_ok]])
        _hi = np.nanmax(np.r_[_x[_ok], _yo[_ok]])
        _ax.plot([_lo, _hi], [_lo, _hi], "--", color="0.6", lw=1, zorder=0)
    if _p in _LOGP:
        _ax.set_xscale("log"); _ax.set_yscale("log")
    _unit = " [Gyr]" if _p in _GYRP else ""
    _ax.set_xlabel(f"full (coupled) {_p}{_unit}", fontsize=9)
    _ax.set_ylabel(f"optical-only {_p}{_unit}", fontsize=9)
    _ax.set_title(_p, fontsize=10)
    _ax.tick_params(labelsize=8)
for _k in range(len(_PROPS), _nr * _nc):
    _axs[_k // _nc][_k % _nc].set_axis_off()
_h, _l = _axs[0][0].get_legend_handles_labels()
_fig.suptitle("decoupled (optical-only) vs coupled CIGALE — same dust_on photometry",
              y=0.995)
_fig.tight_layout(rect=(0, 0, 1, 0.93))
if _h:
    _fig.legend(_h, _l, loc="upper center", ncol=min(len(_h), 7), frameon=False,
                fontsize=9, bbox_to_anchor=(0.5, 0.97))
plt.savefig(os.path.join(PLOTDIR, "decoupled_stability.png"),
            dpi=140, bbox_inches="tight")
plt.show()

# ── the energy-balance test: L_abs (optical) vs L_dust (IR) ──
_labs = np.asarray(TRIO["Labs_opt"], float) / LSUN_W
_ldl  = np.asarray(TRIO["Ldust_dl"], float) / LSUN_W
_lde  = np.asarray(TRIO["Ldust_dl_err"], float) / LSUN_W
_lfu  = np.asarray(TRIO["Ldust_full"], float) / LSUN_W
_ann  = np.asarray(TRIO["is_annulus"], bool)
_ul   = np.asarray(TRIO["dl_upperlimit"], int) == 1
_det  = (np.asarray(TRIO["dl_upperlimit"], int) == 0) & np.isfinite(_ldl) & (_ldl > 0)

_fig, (_axA, _axB, _axC) = plt.subplots(1, 3, figsize=(15.5, 4.8))
for _a in _aps:
    _msk = _det & (np.asarray(TRIO["aperture"], str) == _a) & (_labs > 0)
    _axA.errorbar(_labs[_msk], _ldl[_msk], yerr=_lde[_msk], fmt="o", ms=4,
                  color=_apc[_a], alpha=0.7, lw=0.8, ls="none", label=_a)
_mu = _ul & (_labs > 0) & np.isfinite(_lde) & (_lde > 0)
_axA.scatter(_labs[_mu], _lde[_mu], marker="v", s=26, facecolors="none",
             edgecolors="0.4", label=r"DL upper limit ($1\sigma$)")
_fin = (_labs > 0) & (_det | _mu)
if _fin.any():
    _lo = np.nanmin(np.r_[_labs[_fin], _ldl[_det], _lde[_mu]])
    _hi = np.nanmax(np.r_[_labs[_fin], _ldl[_det], _lde[_mu]])
    _axA.plot([_lo, _hi], [_lo, _hi], "--", color="0.6", lw=1, zorder=0)
_axA.set_xscale("log"); _axA.set_yscale("log")
_axA.set_xlabel(r"$L_{\rm abs}$ optical-only (CF00, never re-emitted) [$L_\odot$]")
_axA.set_ylabel(r"$L_{\rm dust}$ standalone DL2014 [$L_\odot$]")
_axA.set_title("energy balance: absorbed vs re-emitted")
_axA.legend(fontsize=8, frameon=False)

_m2 = _det & np.isfinite(_lfu) & (_lfu > 0)
for _a in _aps:
    _msk = _m2 & (np.asarray(TRIO["aperture"], str) == _a)
    _axB.scatter(_lfu[_msk], _ldl[_msk], s=16, color=_apc[_a], alpha=0.7)
if _m2.any():
    _lo = np.nanmin(np.r_[_lfu[_m2], _ldl[_m2]])
    _hi = np.nanmax(np.r_[_lfu[_m2], _ldl[_m2]])
    _axB.plot([_lo, _hi], [_lo, _hi], "--", color="0.6", lw=1, zorder=0)
_axB.set_xscale("log"); _axB.set_yscale("log")
_axB.set_xlabel(r"$L_{\rm dust}$ coupled fit (energy balance) [$L_\odot$]")
_axB.set_ylabel(r"$L_{\rm dust}$ standalone DL2014 [$L_\odot$]")
_axB.set_title("dust luminosity: coupled vs free")

print()
for _lab, _msk in [("apertures", _det & ~_ann), ("annuli   ", _det & _ann)]:
    _msk = _msk & (_labs > 0)
    _r = np.log10(_ldl[_msk]) - np.log10(_labs[_msk])
    if _r.size:
        _axC.hist(_r, bins=25, histtype="step", lw=1.8,
                  label=f"{_lab.strip()} (n={_r.size})")
        print(f"log10(L_dust,DL / L_abs,opt) {_lab}: median {np.median(_r):+.2f}, "
              f"NMAD {nmad(_r):.2f}")
_axC.axvline(0, color="0.6", ls="--", lw=1)
_axC.set_xlabel(r"$\log_{10}\,L_{\rm dust,DL}/L_{\rm abs,opt}$")
_axC.set_ylabel("N")
_axC.set_title("energy-balance violation (0 = balanced)")
_axC.legend(fontsize=8, frameon=False)
_fig.tight_layout()
plt.savefig(os.path.join(PLOTDIR, "energy_balance_decoupled.png"),
            dpi=140, bbox_inches="tight")
plt.show()


# Run order (cheat sheet)

1. **cluster** — `BUILD_MULTI_Z=True` → run Parts 0–1 (histories); then `BUILD_BH=True` → Part 1
   BH cell. Flip both back to `False` afterwards.
2. Parts 2–3 (selection, SFT/QT, AGN split, statistics → `powderday_quenched_selection.fits`) —
   needs only the HDF5s from step 1. Then **Part 3b** (mass–size QC): `flag_too_large` /
   `flag_unresolved` into `SELECTION_FITS` (carried into every Part 7 catalog).
3. **cluster** — Part 4 (Stage 0 particle files), apply the **powderday aperture patch** (Part 5
   markdown), Part 5 cell, then `bash submit_all_snaps.sh` in **all three** run trees under
   `output/cis25/sed_quenched_regions/<run_tag>/powderday_sed_out/` (`dusty_simdust`, `nodust_1e-12`, `dusty_simdust_agn`).
   Any time after Stage 0: **Part 4b** (star/gas/dust counts per projected annulus × sightline →
   `tables/annulus_particle_counts.fits`) and **Part 4c** (mass-weighted Z_star/Z_gas per
   aperture & annulus → `tables/aperture_metallicities.fits`, the Part 7e Z priors).
4. When `.rtout.sed` files exist: Part 6 QC (must show 5 apertures × 4 inclinations + MC
   uncertainties), then Part 7 → the rest-frame per-aperture catalogs. Optional **Part 7a**:
   differential dust attenuation ($A_V=-2.5\log_{10}F_{\rm on}/F_{\rm off}$) vs ISM and
   quench/AGN diagnostics — no CIGALE needed (`tables/attenuation_vs_ism.fits`; Parts 4b/7g
   reuse its dusty flag), and **Part 7a-agn**: per-band AGN contribution $f_{\rm AGN}$ and the $A_V$ bias from the matched `agn_on`/`dust_on` catalogs (`tables/agn_flux_contribution.fits`). Then **Part 7b** → the observed-frame CIGALE inputs and **Part 7c**
   → the annular ones ($F(<r_{\rm out})-F(<r_{\rm in})$; `ann1kpc`≡`ap1kpc` not duplicated).
   Optional **Part 7d**: SFH priors from the true histories (descriptive record). Then
   **Part 7d2** (required): the aperture-matched smoothed SFH archive
   (`cigale/sfh_smoothed_aperture.h5`) that Part 7e injects into CIGALE via `sfhfromfile`.
5. **cluster** — **Part 7e**: prepare all run dirs + ONE SLURM job array; then
   `sbatch cigale_runs/submit_cigale_array.job` on a login node. Nothing heavy runs in the
   kernel. By default existing `out/` dirs are timestamped-and-kept by CIGALE and everything
   re-fits; `SKIP_IF_DONE=True` (re-run the cell first) makes finished runs exit at once.
   Needs Part 7d2 (injected SFHs) and Part 4c (Z priors). When the array drains, run
   **Part 7e2** — the AGN null test — *before* trusting any $A_V$ from these runs.
6. **cluster** — **Part 7f**: one pass over the Stage-0 cutouts → `tables/aperture_truth.fits`
   (per galaxy × sightline × aperture/annulus truth). Then **Part 7g** — SIMBA truth vs CIGALE
   estimates per run → `simba_vs_cigale.fits/.png` in each `<run_dir>/out/`.
7. **cluster** — **Part 7h** (dust_on vs dust_off stability + the dust_on vs agn_on unmodeled-AGN bias; needs only the Part 7e results),
   then the decoupled ladder: **Part 7i** clones every prepared `dust_on_*` run dir into an
   `optonly_*` twin → `sbatch cigale_runs/submit_cigale_optonly_array.job`; when it drains,
   **Part 7j** fits the IR residual with free-normalization DL2014 templates
   (`<run>/out/dl2014_results.fits`); **Part 7k** builds `tables/decoupled_stability.fits` +
   the stability and energy-balance figures.

**Caveats.**
- Stage 0 writes **100 pkpc region cutouts** (CGM + satellites, periodic-wrap safe) under the
  same filenames as the old plist files (`EXTRACT_OVERWRITE=True` replaces them); the RT grid
  is ±100 kpc (`zoom_box_len`), and the 5 hyperion-log-spaced apertures (1, 3.16, 10, 31.6,
  100 kpc) sample central → outskirts. Only the outermost aperture is slightly depth-truncated
  at its edge (sphere inscribed in the cube). `N_AP/AP_MIN_KPC/AP_MAX_KPC` + `THETA_DEG/PHI_DEG`
  here must match `SED_APERTURE_*` / `THETA/PHI` in `simbanator/sed/parameters_master*.py` at
  RT time. The 1 kpc aperture spans only ~3–4 softening lengths (m25n512) — indicative only.
  Part 7e fits the 3 run modes × 5 **cumulative** apertures × 4 sightlines = 60 CIGALE
  inputs only. The 48 annular inputs from Part 7c are deliberately excluded: an annulus does
  not contain the dust heated by the light it emits, so CIGALE's energy balance — the very
  pathway that sets $A_V$ — is ill-posed there. Narrow `ARMS` or the Part 7e glob (e.g. only
  `*_i0p0`) if runtime matters.
- The **`agn_on` run** is `dust_on` + AGN point sources (`parameters_master-agn.py`:
  `BH_SED=True`, Hopkins+2007 intrinsic quasar template, `BH_var=False` so
  $L_{\rm bol}=0.1\,\dot M_{\rm BH}c^2$ follows the SIMBA accretion rates; attenuated by
  the same live-dust grid). It needs `PartType5` in the Stage-0 cutouts — re-run Part 4
  first. `F_{\rm agn\_on}-F_{\rm on}` per aperture/sightline isolates the reprocessed AGN
  contribution. Switching to the Nenkova torus needs the ~2.4 GB CLUMPY file
  (`~/powderday/agn_models/clumpy_models_201410_tvavg.hdf5`, clumpy.org) — not installed.
  Since 2026-08-07 the Part 7e grids include **`skirtor2016` with `fracAGN` free (0.0 in
  the grid)** for *all three* arms, so the estimator is identical everywhere and `agn_on` is
  fitted AGN-aware. Part 7e2 checks the corollary: the AGN-free arms must recover
  `fracAGN` ≈ 0. The pre-2026-08-07 runs (no `_sfhinj` suffix) had no AGN module and measure
  the *unmodeled*-AGN bias instead — both generations are kept on disk.
- `<filter>_err` is the Hyperion **Monte-Carlo photon noise** propagated through the filter
  convolution — it is an RT-convergence error, not a mock observational depth. All-NaN error
  columns mean the run stored no uncertainties (patch not applied / `set_uncertainties` missing).
- Part 7 fluxes are rest-frame convolved; Part 7b re-extracts observed-frame for CIGALE
  (same `extract_flux_set` helper, `redshift=True`). Part 7a's $A_V$ is therefore a
  **rest-frame** attenuation, directly comparable across anchors.
- The dust_off run uses 1 dust-RT photon (`parameters_master-nodust.py`) — some galaxies can
  crash/truncate; the Part 7 cross-check + `missing_sources_*.txt` make any loss explicit.
- CIGALE error budget: the input files carry raw MC errors; the fit adds `additionalerror = 0.1`
  (10 %) in quadrature via `prepare_run` — change it there, not in Part 7b.
- **Negative errors** = "upper limit" to CIGALE. Part 7b catalogs written before the
  `convolveFilterWithSED` sign fix are all-negative → all-NaN fits; Part 7e's preflight
  (`cigale.sanitize_input_errors`) repairs them in place. Delete any stale all-NaN
  `cigale_runs/<tag>/out/` before re-running — `skip_if_done=True` would keep it.
